# RAG — Notebook, parte A: construir el pipeline (Clase 8)

*Lo que se recupera es el techo de lo que la respuesta puede ser.*

Este notebook acompaña al deck de la Semana 8. El deck **enuncia**; acá **exhibimos** la implementación de lo enunciado, corriendo de arriba a abajo.

La clase tiene un solo hilo. El generador no puede reponer un dato que no le llegó, así que casi toda la ingeniería de RAG —cortar el documento, indexarlo, buscar por dos canales, re-ordenar lo que volvió— es una sola cosa: **ingeniería de traer el pasaje correcto**. La generación decide cuánto de ese techo se alcanza, y con un modelo chico alcanza menos. Encima de eso corre la disciplina: un RAG se mide por **dos superficies de falla separadas** —¿recuperó lo correcto?, ¿la respuesta se mantuvo fiel a eso?—, porque una sola métrica al final te dice que salió mal, no **dónde**.

**El recorrido**, cada pieza apareciendo porque la anterior se queda corta:

0. Preparación.
1. El baseline de la Clase 4, corriendo tal cual.
2. El corpus: medio millar de actos de derecho europeo en español.
3. El embedder — y por qué cambiarlo obliga a re-indexar todo.
4. Partir el documento: en tokens, con solape, por estructura, y el techo que no avisa.
5. Guardar los vectores: de la lista en memoria al store — y la consulta que no encuentra nada.
6. Contextualizar antes de embeber.
7. El canal léxico y la fusión por puesto.
8. Se indexa el hijo, se entrega el padre.
9. El prompt de RAG: delimitar, abstenerse, citar.
10. La vara, la tabla entera — y HyDE, la primera técnica que la vara juzga.
11. Afinar: re-puntuar con un cross-encoder, y la pila completa.

**Y una parte B**, en `Clase08_nbB_rag.ipynb`: el mismo pipeline escrito en LlamaIndex (§12) y medido con RAGAS (§13). Va aparte porque entre las dos mitades se cargan cuatro modelos sobre la misma placa —el embedder, el re-puntuador, el generador, y después otra copia del embedder y del re-puntuador que levanta el framework—, y en una T4 eso no entra. Se corre después de ésta, o sola: no necesita nada de acá.

**Cómo está armado.** Todo libre y local: el corpus sale de MultiEURLEX en español, el embedder es `BAAI/bge-m3` vía `sentence-transformers`, el índice es Chroma, el canal léxico es `bm25s`, el re-puntuador es un cross-encoder multilingüe, y el generador es `qwen3:8b` corriendo en Ollama. Sin APIs pagas y sin claves.

**Cuánto tarda.** Con `SMOKE_TEST = True` (el default) son unos 10 minutos de cómputo más las descargas. Con `False`, unos 35. Las secciones 2 a 8 y la 11 **no necesitan Ollama**: si el bootstrap falla, el grueso del notebook corre igual.

> **Activá la GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU.*

## 0 · Preparación

Primero miramos qué hardware tocó. La respuesta cambia los tiempos, no lo que se puede correr: casi todo el notebook funciona en CPU, sólo que más lento.

In [1]:
# Que tenemos. Si no lista ninguna GPU, activala en Entorno de ejecucion -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "sin GPU"

Tesla T4, 15360 MiB


**Y quién más está usando la placa.** Es la primera aparición de un tema que vuelve tres veces en este notebook: en una T4 no sobra memoria, y todo lo que no se libera se le está sacando a otro. Acá el «otro» todavía no podemos ser nosotros —no cargamos nada—, así que si aparece alguien tomando memoria es un **proceso huérfano de un runtime anterior**. Colab reinicia el kernel, pero no siempre mata lo que ese kernel dejó corriendo.

Vale la pena mirarlo ahora. Si no, el síntoma llega ocho celdas más abajo, disfrazado: un `OutOfMemoryError` al cargar el embedder, que informa la memoria de **otro** proceso y parece un problema del embedder.

In [2]:
import os, subprocess

def _nvidia(consulta):
    # nounits deja numeros pelados, que es lo unico que queremos comparar.
    try:
        r = subprocess.run(["nvidia-smi", "--query-" + consulta,
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if r.returncode == 0 else ""
    except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
        return ""

_placa = _nvidia("gpu=memory.total,memory.free")
if not _placa:
    print("Sin GPU visible: todo corre en CPU, mas lento. Nada mas que mirar aca.")
else:
    _total, _libre = [int(x) for x in _placa.split(", ")]
    _yo, _mio, _ajeno = os.getpid(), 0, 0
    _filas = []
    for _linea in _nvidia("compute-apps=pid,process_name,used_gpu_memory").splitlines():
        _campos = [x.strip() for x in _linea.split(", ")]
        if len(_campos) != 3 or not _campos[0].isdigit():
            continue
        _pid, _nombre, _mem = int(_campos[0]), _campos[1], int(_campos[2])
        _filas.append((_pid, _nombre, _mem))
        if _pid == _yo:
            _mio += _mem
        else:
            _ajeno += _mem

    print("placa: %d MiB en total, %d MiB libres" % (_total, _libre))
    if _filas:
        print()
        for _pid, _nombre, _mem in _filas:
            print("  %-8d %-38s %6d MiB %s"
                  % (_pid, _nombre[-38:], _mem, "<- este kernel" if _pid == _yo else ""))

    # El umbral es ajeno, no libre: si reCORRES el notebook de arriba sin reiniciar, la
    # memoria tomada es la NUESTRA y el diagnostico es otro.
    if _ajeno > 2000:
        print()
        print("=" * 76)
        print("LA PLACA YA ESTA OCUPADA, y no por este kernel: %d MiB ajenos." % _ajeno)
        print()
        print("Casi siempre es un proceso huerfano de un runtime anterior. Colab reinicia")
        print("el kernel, pero ese proceso sigue vivo reservando su VRAM, y el kernel")
        print("nuevo no la ve. Si lo dejas, la seccion 3 muere con OutOfMemoryError.")
        print()
        print("  Entorno de ejecucion -> DESCONECTAR Y ELIMINAR el entorno de ejecucion.")
        print("  Reiniciar NO alcanza: el huerfano sobrevive al reinicio.")
        print("=" * 76)
    elif _mio > 2000:
        print()
        print("Este kernel ya tiene %d MiB tomados: venis de correr el notebook." % _mio)
        print("Para una corrida limpia, reinicia el entorno antes de seguir.")

placa: 15360 MiB en total, 14913 MiB libres


**Todas las instalaciones van juntas y acá arriba.** No es cosmética: un `pip install` que actualiza `numpy` o `pyarrow` respecto de lo que Colab trae precargado puede pedir **reiniciar el entorno**. Si la instalación estuviera repartida por el notebook, ese reinicio llegaría con el índice ya construido y los modelos ya cargados, y se perdería todo.

Las versiones van pineadas con número exacto. **LlamaIndex y RAGAS no están acá**: este notebook no los usa —viven en la parte B, que instala lo suyo aparte— y son los paquetes más pesados de bajar del stack.

In [3]:
# Un solo install, con numeros exactos. Si alguna linea falla, el resto igual se instala.
!pip install -q \
  "chromadb==1.5.9" \
  "sentence-transformers==5.7.0" \
  "faiss-cpu==1.15.1" \
  "bm25s==0.3.11" "PyStemmer==3.1.0" \
  "ollama" "huggingface_hub>=0.23" "pyarrow>=21"

# Por que estas versiones y no las ultimas:
#  - sentence-transformers 6.x exige transformers 5.x, que a su vez fuerza un upgrade
#    grande sobre lo que Colab trae precargado. 5.7.0 convive sin pelear.
#  - pyarrow>=21 lo necesita pandas para leer el parquet del corpus en la seccion 2.
#  - El resto va con numero exacto por politica, y la politica tiene una razon concreta
#    que van a ver en la parte B: ragas 0.3.9 importa langchain_community.chat_models.
#    vertexai, un modulo que langchain-community 0.4.2 elimino -ChatVertexAI se mudo a su
#    propio paquete-. ragas no actualizo el import ni pineo la dependencia, asi que con
#    "la ultima" de langchain-community el "import ragas" falla de entrada. Una version
#    de mas rompe un notebook entero, y por eso ninguna queda librada al azar.
#
# Lo que NO instalamos aca: los cinco paquetes de llama-index, ragas, langchain-ollama y
# langchain-community. La parte A no importa ninguno. Son ademas los que disparan el
# pedido de reinicio de entorno de Colab, asi que sacarlos de aca ahorra la descarga
# grande y, casi siempre, el reinicio.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 94.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 115.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 118.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 99.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━

> **Si Colab te pide reiniciar el entorno, reiniciá y seguí desde acá.** Puede pasar, aunque con esta lista corta es poco probable: el `pip install` de arriba puede actualizar `numpy`, `pyarrow` o `huggingface_hub` respecto de lo que venía precargado. No hace falta volver a correr la celda de instalación después del reinicio.

In [4]:
# Las versiones que realmente quedaron resueltas. Vale la pena mirarlas: casi todos los
# errores raros de este stack son en realidad una version distinta de la que dice arriba.
import importlib.metadata as meta
for p in ("chromadb", "sentence-transformers", "faiss-cpu", "bm25s",
          "transformers", "huggingface_hub", "pyarrow", "numpy"):
    try:
        print("%-24s %s" % (p, meta.version(p)))
    except Exception:
        print("%-24s (no instalado)" % p)

chromadb                 1.5.9
sentence-transformers    5.7.0
faiss-cpu                1.15.1
bm25s                    0.3.11
transformers             5.16.1
huggingface_hub          1.29.0
pyarrow                  23.0.1
numpy                    2.1.3


### 0.1 · Ollama

El generador del curso corre local. La secuencia es la misma de siempre: instalar, levantar el servidor en segundo plano, y bajar los modelos.

Lo único distinto de otras clases es que **dejamos una bandera**, `OLLAMA_OK`. Si el bootstrap falla —el `install.sh` no responde, la descarga de 5 GB se corta— el notebook **no se cae**: las secciones que no necesitan un modelo de lenguaje corren igual, y las que sí lo necesitan lo dicen y siguen de largo. Son dos tercios del contenido.

In [5]:
# zstd hace falta para descomprimir el instalador de Ollama en Colab.
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> C

In [6]:
import os, subprocess, time, requests

# Sin paralelismo, y DOS modelos residentes como maximo. El dos no es generosidad: hay
# dos momentos del notebook donde se alterna entre un generador y un embedder de Ollama en
# la misma celda -la seccion 1, y RAGAS en la 13 de la parte B-. Con el limite en uno,
# Ollama descarga uno para cargar el otro en CADA llamada, y una evaluacion que deberia
# tardar dos minutos se va a timeout. Los dos que conviven son chicos: 2,5 GB el juez y
# 274 MB el embedder. Al generador de 8b lo descargamos a mano con liberar_gpu() antes.
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"

# Y estas dos, que valen una seccion entera de esta clase. JAX -que entra de prestado con
# bm25s, en la seccion 7- PREASIGNA EL 75% DE LA VRAM en cuanto inicializa su backend.
# En una T4 son ~11 GB que desaparecen en un import, que PyTorch no ve -no son suyos, asi
# que empty_cache() no los toca- y que Ollama tampoco ve: cuando va a cargar el generador
# encuentra la placa llena y lo reparte con el procesador. No hay ningun error. Hay una
# celda que tarda veinte veces mas. Tienen que estar ANTES de que nadie importe jax.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_PLATFORMS"] = "cpu"

def ollama_vivo():
    try:
        return requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

# Guard de reentrada: si ya responde, no lanzamos un segundo proceso. El segundo no puede
# bindear el puerto y muere, pero antes ensucia la salida de celdas posteriores.
if ollama_vivo():
    print("Ollama ya estaba corriendo.")
else:
    # Barrido previo. Si un servidor anterior murio, el proceso que tenia el modelo cargado
    # -el "runner"- puede seguir vivo RESERVANDO SU VRAM. El servidor nuevo no la ve, cree
    # que la placa esta llena, y carga el modelo en CPU: la generacion pasa de ~25 tokens
    # por segundo a ~3 y no avisa. Matarlo antes cuesta cinco segundos.
    subprocess.run(["pkill", "-9", "-f", "ollama"], check=False)
    time.sleep(3)

    # stdout y stderr al vacio: si no, los logs del server heredan el stdout del kernel
    # y se filtran en la salida de otras celdas, en momentos arbitrarios.
    # start_new_session=True NO es cosmetico. Sin eso el servidor hereda el grupo de
    # procesos del kernel, y cuando interrumpis una celda -el boton de stop, un Ctrl-C-
    # el SIGINT le llega A EL TAMBIEN y se muere. Despues las secciones que no usan
    # el generador siguen corriendo como si nada, y el error aparece recien en la 9.
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if ollama_vivo():
            print("Ollama está listo.")
            break
        time.sleep(1)
    else:
        print("El servidor tardó en levantar; volvé a correr esta celda.")

OLLAMA_OK = ollama_vivo()
print("OLLAMA_OK =", OLLAMA_OK)
print("VRAM libre en la placa:",
      subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

Ollama está listo.
OLLAMA_OK = True
VRAM libre en la placa: 14913 MiB


In [7]:
MODEL      = "qwen3:8b"          # el generador del curso (~5 GB)
MODEL_JUEZ = MODEL               # el juez de la seccion 13. Si, el mismo: ver el comentario
MODEL_C4   = "nomic-embed-text"  # el embedder de la Clase 4 (~274 MB), solo para la seccion 1

# Los tags van siempre explicitos, nunca "qwen3" a secas: sin el tag, Ollama resuelve a
# latest y el notebook deja de ser reproducible.
#
# Y el juez es EL MISMO MODELO que genera. No es lo que uno querria -es sesgo de
# auto-preferencia servido en bandeja- y esta explicado en la seccion 13 de la parte B.
# La razon es hardware: probamos con qwen3:4b de juez, y en una T4 no llega. Produce JSON
# malformado, RAGAS reintenta, los reintentos se van a timeout, y ademas hay que alternar
# TRES modelos -generador, juez y embedder- con lugar para dos. El resultado eran cuatro
# metricas vacias. Un juez sesgado que mide algo ensena mas que cuatro celdas en NaN.
if OLLAMA_OK:
    for m in (MODEL_C4, MODEL):
        print("pull", m, "...", flush=True)
        r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
        if r.returncode != 0:
            print("   FALLO:", (r.stderr or r.stdout).strip()[:200])
            OLLAMA_OK = False
    if OLLAMA_OK:
        !ollama list
else:
    print("Sin Ollama. Las secciones 2 a 8 y la 11 corren igual; 1, 9, 10, 12 y 13 no.")

pull nomic-embed-text ...
pull qwen3:8b ...
NAME                       ID              SIZE      MODIFIED               
qwen3:8b                   500a1f067a9f    5.2 GB    Less than a second ago    
nomic-embed-text:latest    0a109f422b47    274 MB    35 seconds ago            


### 0.2 · Banderas y ayudantes

`SMOKE_TEST` es la convención del curso: en `True` el notebook corre barato y entero, en `False` corre con el corpus completo. Los números que aparecen en las láminas salen de la corrida con `False`.

In [8]:
import re, time, random
import numpy as np

SMOKE_TEST = False          # True = corrida barata (~12 min). False = la de las laminas (~45 min).

# SMOKE_TEST NO achica el corpus. Achicar el corpus dejaria sin su documento a la mitad
# de las preguntas del eval set, y la tabla estaria midiendo sobre una vara mutilada.
# Lo que achica es lo que de verdad cuesta: cuantas preguntas se miden y cuantas llamadas
# al generador se hacen. Indexar 7.000 pedazos con el embedder en fp16 son ~3 min por pasada.
# Los 514 del modo barato son el PREFIJO de los 1.544: el muestreo ordena con semilla
# fija y corta por presupuesto, asi que agrandar el corpus AGREGA actos, no los
# reordena. Por eso el eval set vale en los dos modos: sus 30 documentos estan en los
# primeros 514. Mas corpus = mas vecinos confundibles = la tabla tiene donde moverse.
N_ACTOS       = 514 if SMOKE_TEST else 1544
N_EVAL        = 12 if SMOKE_TEST else 30   # preguntas por fila de la tabla
N_AB          = 200 if SMOKE_TEST else 400 # actos del A/B de la seccion 4:
                                           # los golds del eval set MAS distractores.
                                           # Con pocos distractores las tres variantes
                                           # aciertan todo y el A/B no discrimina.
TAM_HIJO      = 350        # tokens por pedazo, medidos con el tokenizer del re-puntuador
SOLAPE        = 50         # ~15% de TAM_HIJO
K_ENTREGO     = 5          # cuantos pasajes le pasamos al generador
N_CANDIDATOS  = 50         # cuantos recupera la primera etapa antes de re-puntuar

np.random.seed(1337)
random.seed(1337)

# La tabla de resultados vive en UNA sola lista que crece a lo largo del notebook. Nadie la
# reescribe ni la reordena: cada tecnica le agrega su fila cuando le toca, y el orden en que
# aparecen ES el orden en que las construimos.
TABLA_MADRE = []           # [(nombre, {columna: valor}), ...] -- las columnas, en la seccion 6
RESULTADOS  = {}           # numeros sueltos que no son fila de tabla: tiempos, tokens, cosenos
FALLAS_GENERADOR = []      # llamadas al generador que fallaron; el Cierre las reporta

print("modo:", "SMOKE_TEST (barato)" if SMOKE_TEST else "COMPLETO", "·", N_ACTOS, "actos")

modo: COMPLETO · 1544 actos


**Un ayudante que reusamos en todo el notebook.** `ask()` es el mismo de la Clase 6 y la Clase 7: una llamada a Ollama con temperatura cero y el filtro del monólogo interno de Qwen3.

In [9]:
import ollama

def asegurar_ollama():
    """Se asegura de que el servidor este vivo, y lo revive si no.

    Entre la seccion 1 y la 9 pasan dos pasadas de embedding y unos cuarenta minutos sin
    que nadie le hable a Ollama. Si se murio en el medio -por una interrupcion, por
    memoria- el error no aparece cuando pasa: aparece mucho despues y en otra seccion."""
    if not OLLAMA_OK or ollama_vivo():
        return True
    print("Ollama no responde. Lo vuelvo a levantar...")
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if ollama_vivo():
            print("Ollama esta listo otra vez.")
            return True
        time.sleep(1)
    print("No pude levantarlo. Las secciones que generan no van a correr.")
    return False

def ask(prompt, system=None, temperature=0.0, model=None, think=False,
        fmt=None, num_ctx=8192, num_predict=512):
    if not OLLAMA_OK:
        return "[sin Ollama: esta celda no corrió]"
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    # num_ctx explicito y siempre: qwen3:8b anuncia una ventana de 40K, y reservar el KV
    # cache de 40K en una T4 se come varios GB que despues le faltan al embedder.
    opciones = {"temperature": temperature, "num_ctx": num_ctx, "num_predict": num_predict}

    # Un reintento, y NUNCA una excepcion. Este notebook se corre entero y sin nadie
    # adelante: si la llamada numero trescientos revienta, no se pierde esa celda, se
    # pierden todas las de abajo. Preferimos una marca visible en la salida y seguir.
    for intento in (1, 2):
        try:
            r = ollama.chat(model=model or MODEL, messages=msgs, think=think, format=fmt,
                            options=opciones)
            content = r.message.content or ""
            # Qwen3 es hibrido: a veces, aun con think=False, filtra su cadena de
            # pensamiento dentro de .content, cerrada con </think>. Nos quedamos con lo
            # que viene DESPUES de ese cierre.
            if "</think>" in content:
                content = content.split("</think>")[-1]
            return content.strip()
        except Exception as e:
            FALLAS_GENERADOR.append("%s: %s" % (type(e).__name__, str(e)[:80]))
            if intento == 1:
                asegurar_ollama()      # la causa mas comun es que el servidor se cayo
            else:
                print("   ⚠ el generador no respondió (%s). Sigo." % type(e).__name__)
    return "[el generador no respondió]"

if OLLAMA_OK:
    print(ask("Respondé en una sola palabra: ¿capital de Francia?"))

París


**Cuatro ayudantes de impresión.** `tabla()` es el mismo de la Clase 7 y es el que va a mostrar la tabla de resultados de toda la segunda mitad. `mostrar()` imprime un bloque con título, para los antes-y-después.

Y dos que resuelven el mismo problema de maneras opuestas: en una notebook, un pasaje de mil caracteres impreso de un saque es una línea que hay que scrollear de lado, y nadie la lee. `parrafo()` lo envuelve en renglones de 74 columnas —lo usamos siempre que mostramos **un** pasaje entero—; `recortar()` lo trunca a un renglón —para las **listas** de resultados, donde cada ítem tiene que ocupar una línea y sólo una, aunque la frase quede cortada—.

In [10]:
import textwrap

def tabla(titulo, columnas, filas):
    """Imprime una tabla chiquita. filas = [(nombre, {columna: valor}), ...]."""
    w0 = max([len(str(f[0])) for f in filas] + [10]) + 2
    # El ancho de cada columna sale del encabezado Y DE LOS VALORES. Mirando solo el
    # encabezado, un valor mas largo desborda su celda y se pega con el de al lado.
    anchos = [max([len(str(f[1].get(c, "—"))) for f in filas] + [len(str(c))]) + 3
              for c in columnas]
    print("\n" + titulo)
    print("─" * (w0 + sum(anchos)))
    print("".ljust(w0) + "".join(str(c).rjust(w) for c, w in zip(columnas, anchos)))
    for nombre, vals in filas:
        print(str(nombre).ljust(w0) +
              "".join(str(vals.get(c, "—")).rjust(w) for c, w in zip(columnas, anchos)))
    print()

def parrafo(texto, ancho=74, sangria="   ", lineas=None):
    """Un texto largo, envuelto en renglones cortos para poder leerlo.

    En una notebook, una linea de mil caracteres es una linea que nadie lee: hay que
    scrollear de lado. Esta funcion es la que usamos cada vez que mostramos un pasaje
    entero -un pedazo, una respuesta del generador, un documento-.

    Respeta los saltos de linea que el texto ya traiga: envuelve cada renglon por
    separado, asi un bloque de "campo: valor" no se convierte en una sopa.

    Recibe:
      texto    el texto a mostrar, con o sin saltos de linea
      ancho    columnas por renglon; 74 es el ancho de los separadores del notebook
      sangria  lo que va al principio de cada renglon
      lineas   tope de renglones. None no recorta; con un numero, lo que sobra se
               marca con [...] al final del ultimo

    Devuelve un unico string, listo para print().
    """
    renglones = []
    for original in re.sub(r"[ \t]+", " ", texto.strip()).split("\n"):
        renglones += textwrap.wrap(original, width=ancho) or [""]
    if lineas and len(renglones) > lineas:
        renglones = renglones[:lineas]
        renglones[-1] += " …"
    return "\n".join(sangria + r for r in renglones)

def mostrar(titulo, cuerpo):
    """Un bloque con titulo, para los antes-y-despues. El cuerpo va envuelto."""
    print("─" * 74)
    print(titulo)
    print("─" * 74)
    print(parrafo(cuerpo, sangria=""))
    print()

def recortar(texto, n=160):
    """Una linea, sin saltos, para meter un pasaje adentro de una tabla o una lista.

    Es la hermana de parrafo() para el otro caso: cuando lo que se imprime es una
    LISTA -los cinco resultados con su puntaje alineado- y cada item tiene que
    ocupar un renglon y solo uno, aunque eso signifique cortar la frase.
    """
    limpio = re.sub(r"\s+", " ", texto).strip()
    return limpio[:n] + ("…" if len(limpio) > n else "")

**Y tres ayudantes para la placa.** Vuelve el tema de la primera celda: en una T4 no sobra memoria, y en este notebook conviven **dos runtimes** que se la disputan —PyTorch, que carga el embedder y el re-puntuador, y Ollama, que carga el generador—. Ninguno de los dos ve lo que reserva el otro, así que ninguno puede avisarte.

`vram()` dice quién tiene la memoria ahora mismo, `liberar_gpu()` se la devuelve a la placa antes de cargar un modelo nuevo, y `donde_corre()` confirma que el generador entró en la GPU y no en el procesador. Van a aparecer invocadas en cada sección que cargue o descargue un modelo; están acá juntas porque las tres contestan la misma pregunta.

> **Leé los comentarios de esta celda aunque no leas el código.** Describen tres formas concretas en que este notebook se puede volver veinte veces más lento **sin tirar un solo error**.

In [11]:
import torch

# Que placa toco, y si hay alguna. Todo el notebook corre igual en CPU, mas lento.
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

def vram(etiqueta=""):
    """Tres numeros. El diagnostico esta en la distancia entre el segundo y el tercero.

      libre         lo que queda en la placa, para todos los procesos
      PyTorch       lo que el asignador cacheante de PyTorch tiene reservado
      este proceso  lo que el driver le atribuye a NUESTRO proceso de Python

    Con solo los dos primeros no se pueden separar dos situaciones que piden lo contrario:
    que haya crecido OTRO proceso -Ollama cargando el modelo, y hay que descargarlo- o que
    haya crecido el nuestro por fuera de PyTorch -una libreria que reserva por su cuenta,
    y PyTorch no la ve ni la libera con empty_cache()-.

    Si el tercero es muy superior al segundo, es lo segundo. Y no se manifiesta como un
    error: se manifiesta como un modelo de lenguaje diez veces mas lento."""
    if not torch.cuda.is_available():
        return
    libre = subprocess.run(["nvidia-smi", "--query-gpu=memory.free",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout.strip()
    apps = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_gpu_memory",
                           "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout
    mio = "?"
    for linea in apps.splitlines():
        campos = [x.strip() for x in linea.split(",")]
        if len(campos) == 2 and campos[0] == str(os.getpid()):
            mio = campos[1]
    print("   [vram] %-26s libre %6s · PyTorch %5.1f GB · este proceso %6s MiB"
          % (etiqueta, libre, torch.cuda.memory_reserved() / 1e9, mio))

def liberar_gpu(modelo=None):
    """Devuelve la VRAM que PyTorch retiene y, si se le pide, descarga el modelo de Ollama.

    Son DOS RUNTIMES distintos peleando por la misma placa. PyTorch cachea lo que libera y
    no se lo devuelve al sistema: despues de embeber decenas de miles de pedazos puede estar
    reteniendo varios GB que no usa. Ollama, que es otro proceso, ve esa memoria ocupada y
    descarga capas del modelo a CPU. La generacion cae de ~25 tokens por segundo a ~3 y NO
    AVISA: no hay error, solo una celda que parece colgada.

    Se llama antes de cada seccion que carga un modelo nuevo."""
    import gc
    antes = torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0.0
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        ahora = torch.cuda.memory_reserved() / 1e9
        libre = subprocess.run(["nvidia-smi", "--query-gpu=memory.free",
                                "--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip()
        print("VRAM · PyTorch retenía %.1f GB, ahora %.1f GB · libre en la placa: %s"
              % (antes, ahora, libre))
    if modelo and OLLAMA_OK:
        # Descargarlo obliga a Ollama a recargarlo con la memoria ya liberada. La primera
        # llamada despues de esto tarda ~90 s; las siguientes vuelven a ir rapido.
        subprocess.run(["ollama", "stop", modelo], check=False, capture_output=True)

def donde_corre():
    """Imprime en que procesador quedo el modelo. La columna que importa es PROCESSOR.

    Si dice algo como "98%/2% CPU/GPU", el modelo no entro en la placa y esta generando
    por CPU: mismo resultado, diez veces mas lento, sin ningun error que lo delate.

    Y distingue DOS estados que se parecen y no son lo mismo: "no hay ningun modelo
    cargado" y "no hay servidor". Con el servidor vivo y sin modelos, `ollama ps` igual
    imprime su encabezado; para que stdout salga vacio el comando tiene que fallar. Si
    los confundis -y es facil- terminas diagnosticando un problema de memoria cuando lo
    que tenes es un proceso muerto."""
    r = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        print("OLLAMA NO RESPONDE: el servidor no esta corriendo.")
        print("Volve a correr la celda del bootstrap de la seccion 0.1.")
        if r.stderr.strip():
            print("  (%s)" % r.stderr.strip().splitlines()[0][:110])
    elif len(r.stdout.strip().splitlines()) == 1:
        print(r.stdout.rstrip())
        print("  (servidor vivo, sin ningun modelo residente en este momento)")
    else:
        print(r.stdout.rstrip())

print("ayudantes de placa listos · dispositivo:", DISPOSITIVO)

ayudantes de placa listos · dispositivo: cuda


> **Checkpoint.** Están el entorno, Ollama (o la constancia de que no), las banderas y los ayudantes —el generador, la impresión y la placa—. De acá en adelante el notebook construye una cosa por vez.

▶ **Láminas 1–4** (por qué existe RAG; las tres vías; la escalera de cuatro escalones).

## 1 · El baseline de la Clase 4, tal cual

Antes de romper nada, lo volvemos a armar. Esto es literalmente la sección 6 del notebook de la Clase 4: una base de cinco hechos sobre la materia, embebidos con `nomic-embed-text`, recuperación por coseno, y `qwen3` generando sobre lo recuperado.

Se corre sin tocarle una línea, y por una razón: **eso ya es RAG**. Todo lo que hacemos hoy no es reemplazarlo, es descubrir dónde se rompe cuando el corpus deja de tener cinco frases.

In [12]:
def coseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embeber_ollama(textos):
    # ollama.embed acepta una lista y devuelve un embedding por texto (sin API key, local).
    resp = ollama.embed(model=MODEL_C4, input=textos)
    return np.array(resp.embeddings, dtype=np.float32)

# RAG se luce cuando la respuesta NO esta en los pesos del modelo. Por eso la base son
# hechos de ESTA materia: cosas que qwen3 no tiene forma de conocer.
base = [
    "El primer trabajo práctico (TP1) de la materia se entrega el 27 de septiembre.",
    "La evaluación es 100% por trabajos prácticos grupales, en grupos de hasta 4 integrantes.",
    "El examen final es único y de tipo aprobado/desaprobado, sin nota numérica.",
    "La infraestructura del curso es libre: Ollama y HuggingFace, sin APIs pagas.",
    "El proyecto final se entrega el 29 de noviembre.",
]

base_emb = None
if OLLAMA_OK:
    try:
        base_emb = embeber_ollama(base)
        print("Base embebida con", MODEL_C4 + ":", base_emb.shape)
    except Exception as e:
        # Esta es la primera vez que el notebook le pide algo a Ollama de verdad, asi que
        # es donde aparecen los problemas de la placa. El mas comun, y el mas confuso: el
        # kernel viene de una corrida anterior y todavia tiene la VRAM tomada. Volver a
        # correr las celdas de arriba no la libera -son el mismo proceso de Python-: hay
        # que reiniciar el kernel.
        print("No pude embeber con Ollama:", type(e).__name__)
        print(" ", str(e).strip().splitlines()[0][:150])
        print()
        print("Apago OLLAMA_OK y sigo. Las secciones 2 a 8 y la 11 corren igual;")
        print("1, 9, 10, 12 y 13 se saltean solas.")
        print("Si dice 'out of memory', mira la celda del diagnostico de la seccion 0:")
        print("si el kernel figura ahi tomando memoria, reinicialo y volve a empezar.")
        OLLAMA_OK = False
else:
    print("Sin Ollama: la sección 1 no corre. Seguí en la 2.")

Base embebida con nomic-embed-text: (5, 768)


In [13]:
def responder(pregunta, usar_rag, k=2):
    if usar_rag:
        # RECUPERAR: top-k de la base por coseno (mismo embedder para la query y la base).
        q = embeber_ollama([pregunta])[0]
        scores = [coseno(q, d) for d in base_emb]
        top = [int(i) for i in np.argsort(scores)[::-1][:k]]
        print("  [retrieval] recuperado de la base:")
        for i in top:
            print(f"      cos={scores[i]:.3f}  {base[i]}")
        contexto = "\n".join(f"- {base[i]}" for i in top)
        prompt = (f"Usá SÓLO este contexto para responder.\n\n"
                  f"Contexto:\n{contexto}\n\n"
                  f"Pregunta: {pregunta}\nRespondé en una sola oración.")
    else:
        # SIN RAG: el modelo responde solo, con lo que tenga en los pesos.
        prompt = f"{pregunta}\nRespondé en una sola oración."
    return ask(prompt)

if OLLAMA_OK:
    pregunta = "¿Cuándo se entrega el primer trabajo práctico de la materia?"
    print("SIN RAG (no puede saberlo → inventa una fecha o esquiva):")
    print("  ", responder(pregunta, usar_rag=False), "\n")
    print("CON RAG (recupera el dato de la base y responde con él):")
    print("  ", responder(pregunta, usar_rag=True))

SIN RAG (no puede saberlo → inventa una fecha o esquiva):
   El primer trabajo práctico de la materia se entrega al final de la primera semana de clases. 

CON RAG (recupera el dato de la base y responde con él):
  [retrieval] recuperado de la base:
      cos=0.848  El primer trabajo práctico (TP1) de la materia se entrega el 27 de septiembre.
      cos=0.611  La evaluación es 100% por trabajos prácticos grupales, en grupos de hasta 4 integrantes.
   El primer trabajo práctico de la materia se entrega el 27 de septiembre.


**Lo que hay que ver.** Cuatro piezas: un embedder, un coseno, un top-k, y un generador que escribe sobre lo recuperado. No hay nada más. Ése es el esqueleto entero de RAG, y ya lo teniamos desde la clase de embeddings.

**Y lo que la Clase de Embeddings dijo que faltaba.**
 
- Estas cinco frases entran enteras en el embedder. Un documento de veinte páginas, no. Hay que **partirlo**, y dónde se corta cambia lo que se puede encontrar.
- Cinco vectores en una lista de Python se recorren enteros. Un millón, no.
- Y hay consultas que el coseno **no puede** resolver, por más grande que sea el corpus y por mejor que sea el embedder.

Las tres aparecen en las secciones 4, 5 y —la última— al cierre de la 5.

▶ **Láminas 5–7** (el loop mínimo rearmado; lo que les dijimos que faltaba; y el cambio de embedder).

## 2 · El corpus

Quinientos catorce actos del derecho de la Unión Europea, en español, de **MultiEURLEX**. Documentos reales, largos, con estructura interna y con un identificador oficial por documento — el **CELEX**, que va a ser nuestra etiqueta de «cuál era el documento correcto».

**Cuatro palabras que conviene no mezclar**, porque vuelven en todo el notebook:

| palabra | qué es | ejemplo |
|---|---|---|
| **acto** (o documento) | una norma europea entera. Es la unidad que bajamos, indexamos y evaluamos | el reglamento sobre radiadores de aluminio chinos |
| **CELEX** | el número de catálogo del acto, asignado por EUR-Lex. **No está escrito adentro del documento** | `32012R1039` |
| **artículo** | una sección adentro de un acto. Un acto tiene tres artículos, o cincuenta | `Artículo 2` |
| **cita** | el nombre con el que una norma se menciona **adentro del texto** de otra, como una nota al pie | `Reglamento (CE) no 1005/2009` |

Los dos códigos son los que se confunden, y conviene fijarlos con una analogía de biblioteca: el **CELEX** es el número pegado en el lomo del libro, que puso el bibliotecario; la **cita** es una referencia impresa adentro del libro, que apunta a otro libro.

De ahí salen los dos usos, que son opuestos. El CELEX **no sirve como consulta** —nadie lo escribió en ningún texto, no hay dónde encontrarlo—: sirve para corregir el examen, para decir «la respuesta a esta pregunta estaba en este documento». Las citas, en cambio, **sí** están literales en el cuerpo de los actos, y son las que vamos a usar como consultas más adelante.

**Antes de mirar éste, qué se le pide a cualquier corpus.** La calidad de un RAG está acotada por arriba por su corpus: ninguna de las técnicas de hoy repone algo que los documentos no tienen. Siete propiedades, ordenadas por cuánto duelen cuando faltan.

1. **Que la respuesta esté ahí.** Obvio, y sistemáticamente salteado. Antes de construir un pipeline hay que contestar si las preguntas que la gente va a hacer tienen respuesta en estos documentos. Más proyectos de RAG fracasan por esto que por la tecnología.
2. **Que el texto esté limpio.** Casi todo corpus real llega en PDF: columnas, tablas, encabezados repetidos, páginas escaneadas. Convertirlo a texto usable es la mitad del trabajo y la mitad de los errores de un proyecto real. Una tabla mal extraída produce pasajes que parecen texto y son ruido, y nadie se entera hasta que el modelo responde cualquier cosa.
3. **Que los documentos tengan estructura.** Secciones, capítulos, artículos. La estructura es lo que permite cortar por un lugar que significa algo en vez de cortar cada 300 palabras a ciegas. Un corpus sin estructura se puede usar; se corta peor.
4. **Que cada documento tenga un identificador estable.** Habilita las dos cosas por las que se elige RAG y no otra vía: **citar** —de dónde salió esto— y **filtrar** —sólo lo vigente, sólo lo de este cliente, sólo lo que este usuario puede ver—. Sin identificadores no hay trazabilidad, y sin trazabilidad el sistema no es auditable.
5. **Que haya metadatos.** Fecha, área, estado, permisos. Casi toda pregunta real trae un filtro implícito: «¿qué dice la política?» quiere decir «la vigente», no las cuatro versiones anteriores.
6. **Que se pueda actualizar.** Un corpus vivo incorpora documentos, versiona los que cambian y da de baja los que dejan de valer. Decidir cómo se re-indexa es parte del diseño, no una tarea posterior.
7. **Que exista un conjunto de evaluación.** Un puñado de preguntas reales con su documento correcto etiquetado a mano. Es lo que casi nadie tiene y lo que separa una demo de un sistema de ingeniería: sin eso, «mejoramos el RAG» es una opinión. Por eso media sección 2 son treinta preguntas escritas a mano.

**Y una que no está en la lista porque no es una propiedad deseable, sino un hecho:** el tamaño importa mucho menos de lo que parece, y el **parecido entre documentos** importa mucho más. Un corpus chico y bien elegido rinde más que uno enorme y sucio; y si todos los documentos hablan de lo mismo en el mismo registro, distinguir el correcto es difícil por más grande que sea el modelo.

**Por qué este corpus y no otro.** No se eligió por el tema. Se eligió porque cumple, sin trabajo extra, las cuatro condiciones que la clase necesita para poder mostrar sus problemas: documentos **largos** (si entraran enteros en el embedder, la sección 4 no tendría sentido), con **estructura explícita** (`Artículo N` en línea propia, que habilita las secciones 4 y 8), con un **identificador oficial por documento** (el CELEX: ground truth gratis, sin anotadores) y con **identificadores literales adentro del texto** (las citas a otras normas, que son las consultas que van a hacer fallar al coseno en la sección 5.3).

**Y está curado, lo cual se avisa.** Se descartaron los actos de menos de 8.000 caracteres —no habría nada que partir— y los de más de 60.000 —desbalancean el índice y el tiempo de indexado—. Es transparente y es razonable, pero un corpus real no viene así: viene como viene, y descubrir qué forma tiene es parte del trabajo.

| propiedad | en un corpus real | acá | por qué |
|---|---|---|---|
| documentos reales y largos | sí | **igual** | actos de 8.000 a 60.000 caracteres |
| texto sucio, extraído de PDF | sí, y cuesta | **distinto**: llega en texto | OCR, tablas y columnas son otro problema, y feo |
| estructura interna | a veces | **igual**: artículos numerados | habilita cortar por estructura y entregar el padre |
| identificador estable | debería | **igual**: el CELEX | oficial, único, gratis |
| metadatos ricos | sí | **distinto**: casi ninguno | se simulan en la sección 5.2 para mostrar el filtrado |
| corpus que cambia | sí | **distinto**: congelado | la re-indexación incremental no entra en la clase |
| documentos parecidos entre sí | sí, y es el problema | **igual, y a propósito** | mismo registro jurídico: vecinos confundibles de sobra |
| conjunto de evaluación | casi nunca lo tienen | **igual**, escrito a mano | es el corazón de la segunda mitad |
| multi-idioma, permisos, versionado | frecuente | **fuera de alcance** | se nombran, no se implementan |

**La síntesis, para tenerla a mano cuando armen el Proyecto 2.** Toda la tabla se resume en una frase: lo que sacamos del camino es **el trabajo sucio de datos**; lo que dejamos intacto es **todo lo que hace difícil la recuperación**.

In [14]:
import pandas as pd
from huggingface_hub import hf_hub_download

# Camino primario: el subset ya filtrado y publicado. Camino de respaldo: reconstruirlo
# desde el espejo parquet con los mismos parametros. El segundo tarda ~1 min mas.
URL_CORPUS = ""   # <- pegar acá la URL del clase08_corpus.jsonl.gz publicado

REPO_HF, COMMIT_HF = "coastalcph/multi_eurlex", "6278c7994cd86297e823becad7fdf897faaea174"
# El presupuesto abarca el pool entero de actos que pasan los filtros. Quien decide
# cuantos se usan es N_ACTOS, no este numero.
PISO, TECHO, MIN_ART, PRESUPUESTO_CHARS, SEMILLA = 8_000, 60_000, 3, 40_000_000, 1337

# "Articulo N" en linea propia es el unico nivel de jerarquia que este corpus garantiza,
# y es el que vamos a usar como padre en la seccion 8.
RE_ARTICULO = re.compile(r"^[ \t]*Art[ií]culo\s+\d+", re.MULTILINE | re.IGNORECASE)

def reconstruir_desde_el_espejo():
    # load_dataset() NO sirve: el repo es script-based y datasets>=4.0 ya no ejecuta scripts.
    # Vamos directo al parquet. validation+test son 46 MB; train solo son 185 MB mas.
    marcos = []
    for split in ("validation", "test"):
        for rev in (COMMIT_HF, "refs/convert/parquet"):
            try:
                marcos.append(pd.read_parquet(hf_hub_download(
                    REPO_HF, "es/%s/0000.parquet" % split, repo_type="dataset", revision=rev)))
                break
            except Exception:
                continue
    if not marcos:
        raise RuntimeError("No se pudo bajar el corpus ni de la URL ni del espejo parquet.")
    d = pd.concat(marcos, ignore_index=True)
    d["largo"] = d["text"].str.len()
    d["articulos"] = d["text"].map(lambda t: len(RE_ARTICULO.findall(t)))
    d = d[(d.largo >= PISO) & (d.largo <= TECHO) & (d.articulos >= MIN_ART)]
    d = d.sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)
    return d[d["largo"].cumsum() <= PRESUPUESTO_CHARS].reset_index(drop=True)

if URL_CORPUS:
    corpus = pd.read_json(URL_CORPUS, lines=True, compression="gzip")
    print("corpus leído del subset publicado")
else:
    print("no hay URL publicada: reconstruyo desde el espejo parquet (~1 min)")
    corpus = reconstruir_desde_el_espejo()

if "largo" not in corpus.columns:
    corpus["largo"] = corpus["text"].str.len()
    corpus["articulos"] = corpus["text"].map(lambda t: len(RE_ARTICULO.findall(t)))

corpus = corpus.head(N_ACTOS).reset_index(drop=True)
TEXTO_DE = dict(zip(corpus.celex_id.astype(str), corpus.text))

print("actos            :", len(corpus))
print("caracteres       : %.1f M" % (corpus.largo.sum() / 1e6))
# "piezas Articulo N" y no "articulos" a secas: la seccion 8 cuenta PADRES, que son estas
# piezas MAS un encabezado por acto. Con el mismo rotulo parecia que habia mas padres que
# articulos, que es imposible.
print("piezas «Artículo»:", int(corpus.articulos.sum()))

no hay URL publicada: reconstruyo desde el espejo parquet (~1 min)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


es/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 19.0MB            

es/validation/0000.parquet: downloading bytes:           |  0.00B            

es/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 27.4MB            

es/test/0000.parquet: downloading bytes:           |  0.00B            

actos            : 1544
caracteres       : 30.4 M
piezas «Artículo»: 14436


**Cómo viene un acto.** Vale la pena mirarlo una vez, porque todo lo que sigue —dónde cortamos, qué prefijo le ponemos a cada pedazo, qué cuenta como padre— depende de esta forma concreta.

In [15]:
ejemplo = corpus.iloc[0]
print("CELEX:", ejemplo.celex_id, "|", "{:,}".format(ejemplo.largo).replace(",", "."),
      "caracteres |", ejemplo.articulos, "artículos")
print("─" * 74)
print(parrafo(ejemplo.text[:1800]))
print("\n   [...]")

CELEX: 32015R0009 | 23.804 caracteres | 3 artículos
──────────────────────────────────────────────────────────────────────────
   REGLAMENTO (UE) 2015/9 DE LA COMISIÓN
   de 6 de enero de 2015
   que modifica el Reglamento (UE) no 142/2011, por el que se establecen las
   disposiciones de aplicación del Reglamento (CE) no 1069/2009 del
   Parlamento Europeo y del Consejo por el que se establecen las normas
   sanitarias aplicables a los subproductos animales y los productos
   derivados no destinados al consumo humano, y la Directiva 97/78/CE del
   Consejo en cuanto a determinadas muestras y unidades exentas de los
   controles veterinarios en la frontera en virtud de la misma
   (Texto pertinente a efectos del EEE)
   LA COMISIÓN EUROPEA,
   Visto el Tratado de Funcionamiento de la Unión Europea,
   Vista el Acta de Adhesión de Croacia y, en particular, su artículo 50,
   Visto el Reglamento (CE) no 1069/2009 del Parlamento Europeo y del
   Consejo, de 21 de octubre de 2009, por el q

### 2.1 · La estructura, que es la única jerarquía que este corpus garantiza

Un acto europeo tiene un encabezado y después una serie de `Artículo N`, cada uno en su propia línea. Eso nos da **dos niveles**: el documento y el artículo. No más: TÍTULO y CAPÍTULO aparecen sólo en los actos más largos.

Dos niveles es exactamente lo que hace falta. La sección 8 va a indexar pedazos chicos y entregar el artículo entero que los contiene; con tres niveles habría que elegir cuál es el padre, y eso es una decisión de diseño que no queremos tomar hoy.

> **Decisión didáctica, y se avisa:** esto es lo único que parseamos. Convertir un PDF a texto —OCR, tablas, columnas— es un problema distinto y bastante más feo, y queda fuera de la clase. El corpus llega ya en texto.

In [16]:
def partir_en_articulos(texto):
    """Devuelve [(etiqueta, cuerpo), ...] cortando por 'Articulo N' en linea propia."""
    cortes = [m.start() for m in RE_ARTICULO.finditer(texto)]
    if not cortes:
        return [("preámbulo", texto)]
    partes = []
    # Todo lo anterior al primer articulo es el encabezado: se conserva, porque de ahi sale
    # el titulo con el que vamos a contextualizar los pedazos en la seccion 6.
    if cortes[0] > 0:
        partes.append(("encabezado", texto[:cortes[0]]))
    for i, ini in enumerate(cortes):
        fin = cortes[i + 1] if i + 1 < len(cortes) else len(texto)
        trozo = texto[ini:fin]
        etiqueta = re.sub(r"\s+", " ", trozo.split("\n")[0]).strip()
        partes.append((etiqueta, trozo))
    return partes

def titulo_de(texto, lineas=2):
    """Las primeras lineas no vacias del acto: es su titulo, repartido en varias lineas."""
    utiles = [re.sub(r"[ \t]+", " ", l).strip() for l in texto.split("\n")]
    return " — ".join([l for l in utiles if l][:lineas])[:150]

art = partir_en_articulos(ejemplo.text)
print("título:", titulo_de(ejemplo.text))
print("piezas:", len(art))
for etiqueta, cuerpo in art[:4]:
    print("  %-16s %6d caracteres \n%s" % (etiqueta[:16], len(cuerpo), parrafo(cuerpo, 70)))
    print("─" * 74)

título: REGLAMENTO (UE) 2015/9 DE LA COMISIÓN — de 6 de enero de 2015
piezas: 4
  encabezado        19580 caracteres 
   REGLAMENTO (UE) 2015/9 DE LA COMISIÓN
   de 6 de enero de 2015
   que modifica el Reglamento (UE) no 142/2011, por el que se establecen
   las disposiciones de aplicación del Reglamento (CE) no 1069/2009 del
   Parlamento Europeo y del Consejo por el que se establecen las normas
   sanitarias aplicables a los subproductos animales y los productos
   derivados no destinados al consumo humano, y la Directiva 97/78/CE del
   Consejo en cuanto a determinadas muestras y unidades exentas de los
   controles veterinarios en la frontera en virtud de la misma
   (Texto pertinente a efectos del EEE)
   LA COMISIÓN EUROPEA,
   Visto el Tratado de Funcionamiento de la Unión Europea,
   Vista el Acta de Adhesión de Croacia y, en particular, su artículo 50,
   Visto el Reglamento (CE) no 1069/2009 del Parlamento Europeo y del
   Consejo, de 21 de octubre de 2009, por el que se est

### 2.2 · Los ejemplos fijos

La clase vuelve una y otra vez sobre los mismos ejemplos, en vez de inventar uno nuevo en cada sección: así no hay que entender un documento distinto cada quince minutos. Son dos, y se eligen acá.

**Una consulta.** Una cita normativa literal —del tipo `Reglamento (CE) no 1005/2009`—, de las que aparecen escritas adentro del texto de los actos. Se eligió con un criterio: **rara como cadena, común como tema**. Rara, porque la mencionan pocos actos y entonces hay una respuesta correcta identificable. Común, porque el acto que la cita comparte tema con cientos de actos del corpus, así que hay de sobra con qué confundirse. Y una tercera condición, menos obvia: la cita está **enterrada** en medio del preámbulo, no suelta en un párrafo corto. Esa combinación —fácil de nombrar, difícil de distinguir por el tema, y perdida dentro de un texto largo— es la que la vuelve interesante.

**Tres documentos.** Tres actos concretos —derecho antidumping sobre radiadores de aluminio, control de la actividad pesquera, aeronavegabilidad de aeronaves— para cuando haga falta mostrar algo sobre un documento de verdad. Se eligieron por ser bien distintos entre sí y de tamaños distintos.

La celda de abajo, además de fijarlos, **verifica que estén en el corpus que te tocó**. Si la consulta elegida no aparece —porque estás en modo barato y ese acto no entró— elige sola otra con el mismo criterio, y la clase sigue igual.

In [17]:
# La consulta del caso corrido. Sale de la escalera de candidatas del notebook de
# preparacion, que las ordena por "vecindad tematica": cuantos actos del corpus comparten
# etiqueta EUROVOC con el que la menciona. Esta encabeza la lista con 476 vecinos.
#
# Los tres criterios por los que se eligio, y conviene entenderlos porque son los que
# hacen que la seccion 5.3 demuestre algo:
#   - rara como cadena : la menciona UN solo acto, asi que hay una respuesta correcta.
#   - comun como tema  : cientos de actos vecinos con los que el coseno se puede confundir.
#   - ENTERRADA        : aparece en el considerando (1), en medio del preambulo. Eso
#     importa mas de lo que parece. Si la cita cae en un pedazo corto -el cierre de un
#     articulo, por ejemplo- ocupa buena parte de ese pedazo, el vector del pedazo termina
#     pareciendose a la consulta, y la busqueda densa la encuentra: la falla que queremos
#     mostrar no se ve. Enterrada en 350 tokens de considerandos, la cita es una fraccion
#     minima del texto y el vector habla de otra cosa.
CONSULTA_ID          = "Reglamento (CEE) no 3730/87"
CONSULTA_ID_RESPALDO = "Reglamento (CE) no 288/2009"

# Los dos o tres actos que la clase usa como ejemplo fijo, elegidos a mano de las fichas.
CASO_CORRIDO = [
    "32012R1039",   # derecho antidumping sobre radiadores de aluminio de China (51.915 chars)
    "32010D0352",   # participacion financiera de la Union en control y vigilancia pesquera
    "32014R1321",   # mantenimiento de la aeronavegabilidad de aeronaves
]

# Las dos familias de numeracion europea son asimetricas: las directivas van ano/numero
# (Directiva 2014/24/UE) y los reglamentos numero/ano (Reglamento (UE) n.o 345/2013).
# Un solo patron para las dos inventa citas que no existen en el derecho europeo.
RE_CITA = re.compile(
    r"(?:Directiva|Decisi[oó]n|Recomendaci[oó]n)\s+\d{1,4}/\d{1,4}/(?:CEE|CE|UE|CECA|Euratom)"
    r"|(?:Reglamento|Directiva|Decisi[oó]n)\s+\((?:CEE|CE|UE|CECA|Euratom)\)\s*n\.?[oº°]?\s*\d{1,5}/\d{2,4}",
    re.IGNORECASE)

# La abreviatura de "numero" aparece en el corpus como "no", "n.o", "nº" y "n°" segun el acto.
# Buscar una forma dentro de un texto escrito con otra no devuelve nada -y no da error-.
# canon() lleva las dos puntas a la misma forma antes de comparar.
RE_NUM_ABREV = re.compile(r"n\.?\s*[oº°]?\s*(?=\d)", re.IGNORECASE)

def canon(s):
    return RE_NUM_ABREV.sub("n", re.sub(r"\s+", " ", s).strip().lower())

TEXTO_CANON = {c: canon(t) for c, t in TEXTO_DE.items()}

def duenos_de(cita):
    """Que actos mencionan esta cita en su cuerpo, comparando en forma canonica."""
    c = canon(cita)
    return [k for k, v in TEXTO_CANON.items() if c in v]

In [18]:
GOLD_ID = duenos_de(CONSULTA_ID)
if not GOLD_ID:
    print("⚠  '%s' no está en este subset. Pruebo el respaldo." % CONSULTA_ID)
    CONSULTA_ID = CONSULTA_ID_RESPALDO
    GOLD_ID = duenos_de(CONSULTA_ID)

if not GOLD_ID:
    print("⚠  el respaldo tampoco. Elijo una del corpus con el mismo criterio.")
    conteo = {}
    for t in TEXTO_DE.values():
        for bruta in set(RE_CITA.findall(t)):
            k = canon(bruta)
            conteo[k] = conteo.get(k, 0) + 1
    # Rara como cadena: entre 1 y 4 actos la mencionan. Si aparece en muchos, encontrarla no
    # tiene merito. Y descartamos las que son el TITULO del propio acto que las contiene.
    for cita in sorted(c for c, n in conteo.items() if 1 <= n <= 4):
        duenos = duenos_de(cita)
        nums = [x for x in re.findall(r"\d+", cita) if len(x) >= 3]
        if duenos and not all(any(x.zfill(4) in d for x in nums) for d in duenos):
            CONSULTA_ID, GOLD_ID = cita, duenos
            break

print("consulta del identificador :", CONSULTA_ID)
print("actos que la mencionan     :", GOLD_ID if GOLD_ID else "NINGUNO (revisar el subset)")
if GOLD_ID:
    t = TEXTO_DE[GOLD_ID[0]]
    pos = canon(t).find(canon(CONSULTA_ID))
    print("en contexto                :")
    print(parrafo("…%s…" % re.sub(r"\s+", " ", t[max(0, pos - 120):pos + 160]).strip()))
print()
print("caso corrido:")
for c in CASO_CORRIDO:
    print("  %-12s %s" % (c, recortar(titulo_de(TEXTO_DE[c]), 80) if c in TEXTO_DE
                          else "NO ESTA en este subset (¿estás en modo SMOKE_TEST?)"))

consulta del identificador : Reglamento (CEE) no 3730/87
actos que la mencionan     : ['32012R0121']
en contexto                :
   …Comité de las Regiones (2), De conformidad con el procedimiento
   legislativo ordinario (3), Considerando lo siguiente: (1) El Reglamento
   (CEE) no 3730/87 del Consejo, de 10 de diciembre de 1987, por el que se
   establecen las normas generales aplicables al suministro a determinadas…

caso corrido:
  32012R1039   REGLAMENTO DE EJECUCIÓN (UE) No 1039/2012 DEL CONSEJO — de 29 de octubre de 2012
  32010D0352   DECISIÓN DE LA COMISIÓN — de 22 de junio de 2010
  32014R1321   REGLAMENTO (UE) No 1321/2014 DE LA COMISIÓN — de 26 de noviembre de 2014


### 2.3 · El eval set: la vara para medir

Treinta preguntas escritas a mano, cada una con **el documento que la responde**. Eso es todo lo que hace falta para poder decir «esto mejoró» con un número en vez de con una impresión: se hace la pregunta, y se mira si el buscador trajo ese documento.

Tres decisiones, cortas:

- **Las escribió una persona, no un modelo.** A un modelo le pedís «una pregunta sobre este texto» y te devuelve el texto con signos de interrogación.
- **Están parafraseadas, no copiadas.** Si la pregunta repite las palabras del documento, encontrarlo es trivial y la medición se infla sola.
- **Se etiqueta el documento, no el pedazo.** Más adelante vamos a cambiar varias veces la forma de cortar los documentos; si la vara apuntara a pedazos, se invalidaría en cada cambio.

Va también una pregunta suelta, plausible para este corpus, que el corpus **no** responde. Sirve para ver si el modelo sabe decir «no sé» en vez de inventar.

La segunda celda no agrega nada: comprueba que las treinta preguntas tengan su documento en este corpus —por si te tocó un subset distinto— y marca los números como no citables si estás en modo barato.

In [19]:
# (pregunta, celex_id del acto que la responde). Treinta filas escritas a mano contra las
# fichas del notebook de preparacion, parafraseando en vez de copiar: si la pregunta repite
# el vocabulario del documento, el canal lexico acierta por copia y la tabla de la segunda
# mitad queda inflada. El termometro de la seccion 10 mide exactamente eso.
EVAL_SET = [
    ("¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos "
     "de vigilancia de la actividad pesquera durante 2010?", "32010D0352"),
    ("¿Qué reglas comunes rigen para que un avión matriculado en un país de la Unión siga "
     "estando en condiciones de volar?", "32014R1321"),
    ("¿Cuándo puede cerrarse de un día para otro una zona de pesca en el Mar del Norte, y a "
     "qué especies alcanza esa medida?", "32010R0724"),
    ("¿Qué instrumento permite que una patente tenga efecto uniforme en varios países sin "
     "tener que registrarla en cada uno?", "32012R1257"),
    ("¿Qué condiciones hay que cumplir para afirmar en una etiqueta que un alimento ayuda a "
     "bajar el colesterol?", "32014R0686"),
    ("¿Qué ventaja aduanera tienen las mercancías que se importan en las islas atlánticas "
     "portuguesas, y durante cuánto tiempo rige?", "32010R0973"),
    ("¿Qué norma de 2012 amplió la lista de sustancias autorizadas en productos biocidas, y "
     "desde cuándo debían aplicarse sus disposiciones?", "32012L0002"),
    ("¿Qué se resolvió sobre la deuda que una empresa búlgara mantenía con el Estado sin que "
     "nadie la ejecutara durante años?", "32012D0706"),
    ("¿Qué se decidió sobre la venta en Europa de un algodón modificado genéticamente, y qué "
     "código único se le asignó?", "32011D0354"),
    ("¿Qué gravamen pagan los radiadores de aluminio que llegan desde China, y por qué se lo "
     "impuso?", "32012R1039"),
    ("¿Cómo se le cobra a los trenes de carga por el ruido que producen, y quién fija ese "
     "importe?", "32015R0429"),
    ("¿Con qué métodos de laboratorio se comprueba si un aceite de oliva cumple lo que "
     "declara su etiqueta?", "32013R1348"),
    ("¿En qué programa conjunto de investigación sobre mediciones participa la Unión junto a "
     "una treintena de países?", "32014D0555"),
    ("¿Qué protección obtuvo un dulce típico checo, y qué pasó con las marcas alemanas que "
     "usaban un nombre parecido?", "32011R0744"),
    ("¿Qué programa de financiación sostuvo entre 2014 y 2020 las acciones para construir un "
     "espacio común de justicia?", "32013R1382"),
    ("¿Qué porcentaje máximo de bandadas de pollos puede dar positivo a dos tipos de "
     "salmonela?", "32012R0200"),
    ("¿Cuánta cantidad de gases que dañan la capa de ozono se permitió importar en 2011, y "
     "cómo se repartió entre las empresas?", "32011D0185"),
    ("¿Qué norma de 2013 sumó una sustancia más a la lista de biocidas autorizados, y desde "
     "cuándo rigen sus disposiciones?", "32013L0005"),
    ("¿Qué maíz transgénico obtuvo autorización de venta en 2010, y con qué identificador "
     "quedó registrado?", "32010D0429"),
    ("¿Hasta qué fecha se extendió el mandato de la enviada especial de la Unión para Sudán "
     "y Sudán del Sur?", "32011D0499"),
    ("¿Dónde se establece cuánto le corresponde por año a cada país de los fondos "
     "estructurales y de la iniciativa de empleo joven?", "32014D0190"),
    ("¿Qué formularios deben usar los bancos para hacer pública la composición de su capital "
     "regulatorio?", "32013R1423"),
    ("¿A qué tres países se les permitió trasladar al comprador la obligación de pagar el "
     "IVA en la venta de celulares y microprocesadores?", "32010D0710"),
    ("¿Hasta cuándo se prorrogó el mandato del representante de la Unión en Afganistán?",
     "32014D0383"),
    ("¿Qué controles se exigieron a los alimentos que llegaban desde Japón después del "
     "accidente nuclear de 2011?", "32011R0297"),
    ("¿Cómo funciona la red europea que conecta ofertas de empleo entre países, y quién la "
     "coordina?", "32012D0733"),
    ("¿Qué se modificó en 2015 sobre los límites de residuos de plaguicidas en alimentos, y "
     "qué pasa con lo producido antes?", "32015R0400"),
    ("¿Qué restricciones al movimiento de cerdos se impusieron en Lituania por un brote de "
     "peste porcina?", "32011D0508"),
    ("¿Cuánto se podía pescar en el Mar Báltico durante 2012, y qué se cambió sobre el "
     "esfuerzo pesquero del bacalao?", "32011R1256"),
    ("¿Cómo cooperan los países para que los sistemas de identidad digital de uno sean "
     "reconocidos por los demás?", "32015D0296"),
]

# Una pregunta plausible para este corpus y que este corpus NO responde. Tiene que cumplir
# DOS condiciones, y la segunda es la que hace que la seccion 9.1 muestre algo:
#
#   - PLAUSIBLE. Si preguntaramos por la capital de Francia, el modelo contesta de sus pesos
#     y no probamos nada.
#   - TENTADORA. Tiene que pedir un dato del MISMO TIPO que el que abunda en los pasajes que
#     se van a recuperar. Un modelo moderno y alineado se abstiene solo cuando lo recuperado
#     no tiene nada que ver; confabula cuando tiene delante un dato con la forma correcta y
#     sabe que le pidieron una respuesta.
#
# Movemos el ANO y no el sujeto. Dos intentos anteriores fallaron y conviene dejarlos
# anotados: un plazo procesal ante el Tribunal de Justicia trae pasajes tan lejanos que el
# modelo se abstiene sin esfuerzo; y preguntar por pavos en vez de pollos no era trampa,
# porque 32012R1190 -salmonela en pavos- esta en el corpus. El corpus llega hasta ~2015,
# asi que una cuota de pesca de 2020 no existe, pero el top-k va a venir lleno de tablas de
# cuotas del Baltico con la forma exacta de la respuesta. La pregunta es casi la misma que
# la fila 29 del eval set, que pregunta por 2012: lo unico que cambia es el ano.
PREGUNTA_FUERA = ("¿Cuánto se podía pescar de bacalao en el Mar Báltico durante 2020, y qué "
                  "se cambió sobre el esfuerzo pesquero?")

# El intento anterior, por si hiciera falta: sigue estando fuera del corpus, pero lo
# recuperado queda tan lejos del tema que el modelo se abstiene sin dudar.
PREGUNTA_FUERA_ALT = ("¿De cuántos días dispone un Estado miembro para recurrir ante el "
                      "Tribunal de Justicia una decisión de la Comisión sobre ayudas estatales?")

# Validacion blanda de la trampa, con el mismo criterio que el resto de la seccion 2. El
# CELEX lleva el ano en las posiciones 1 a 4 (32012R1039 -> 2012), asi que alcanza con
# mirar si hay actos del ano que preguntamos.
_anios = sorted({c[1:5] for c in TEXTO_DE if c[1:5].isdigit()})
if "2020" in _anios:
    print("⚠  el corpus tiene actos de 2020: la pregunta fuera de corpus podría tener "
          "respuesta. Revisá si conviene usar PREGUNTA_FUERA_ALT.")

In [20]:
# Validacion blanda: informa y sigue. Nunca un assert, porque un assert que falla aca le
# borra al lector las diez secciones siguientes, que es el 90% del notebook.
faltantes = [c for _, c in EVAL_SET if c not in TEXTO_DE]
EVAL_SET = [(p, c) for p, c in EVAL_SET if c in TEXTO_DE]

EVAL_ES_PROVISORIO = len(EVAL_SET) < 20 or SMOKE_TEST
BANNER_EVAL = ("  ⚠ medido sobre %d de %d preguntas (SMOKE_TEST). Para los números del deck: "
               "SMOKE_TEST = False." % (min(N_EVAL, len(EVAL_SET)), len(EVAL_SET))
               ) if SMOKE_TEST else (
               "  ⚠ eval set incompleto (%d filas): los números no son citables."
               % len(EVAL_SET) if EVAL_ES_PROVISORIO else "")

if faltantes:
    print("⚠  %d de las 30 filas apuntan a actos que no están en este subset:" % len(faltantes),
          ", ".join(faltantes[:5]))
    print("   Revisá que el corpus sea el mismo con el que se escribieron (hash del .meta.json).")
    print()

print("eval set: %d filas · se miden las primeras %d" % (len(EVAL_SET), min(N_EVAL, len(EVAL_SET))))
if BANNER_EVAL:
    print(BANNER_EVAL)
print()
for pregunta, celex in EVAL_SET[:5]:
    print("  %-12s  %s" % (celex, parrafo(pregunta, 88)))

eval set: 30 filas · se miden las primeras 30

  32010D0352       ¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos de
   vigilancia de la actividad pesquera durante 2010?
  32014R1321       ¿Qué reglas comunes rigen para que un avión matriculado en un país de la Unión siga
   estando en condiciones de volar?
  32010R0724       ¿Cuándo puede cerrarse de un día para otro una zona de pesca en el Mar del Norte, y a
   qué especies alcanza esa medida?
  32012R1257       ¿Qué instrumento permite que una patente tenga efecto uniforme en varios países sin
   tener que registrarla en cada uno?
  32014R0686       ¿Qué condiciones hay que cumplir para afirmar en una etiqueta que un alimento ayuda a
   bajar el colesterol?


### 2.4 · Un segundo bloque de consultas, de otra naturaleza

Las treinta preguntas de arriba son **temáticas**: describen de qué trata el documento. Hay otro tipo de consulta que en un corpus normativo es al menos igual de común, y que se comporta al revés: **buscar un acto por su identificador**.

Ésas no se escriben a mano, porque no hay nada que parafrasear — un identificador no tiene sinónimos, así que la pregunta *es* la cita. Se derivan del corpus quedándonos con las citas que aparecen en **un solo acto**: si aparecen en varios, no hay un documento correcto que etiquetar.

Las separamos en un bloque propio a propósito. Son el caso donde la búsqueda por significado falla de la forma más limpia, y tener las dos columnas al lado hace visible algo que una sola métrica promediaría hasta borrarlo: **una técnica puede mejorar un tipo de consulta y no tocar el otro**.

In [21]:
def eval_identificadores(n):
    citas = {}
    for celex, texto in TEXTO_DE.items():
        for bruta in set(RE_CITA.findall(texto)):
            c = canon(bruta)
            if c not in citas:
                citas[c] = {"forma": re.sub(r"\s+", " ", bruta).strip(), "duenos": set()}
            citas[c]["duenos"].add(celex)

    salida = []
    for c in sorted(citas):                       # sorted: la lista es siempre la misma
        duenos = citas[c]["duenos"]
        if len(duenos) != 1:
            continue                              # sin un unico dueno no hay gold que etiquetar
        celex = next(iter(duenos))
        # Descartamos las citas que son el TITULO del propio acto que las contiene: ahi el
        # identificador no esta "adentro de otro documento", que es el caso que nos interesa.
        nums = [x for x in re.findall(r"\d+", c) if len(x) >= 3]
        if any(x.zfill(4) in celex for x in nums):
            continue
        salida.append((citas[c]["forma"], celex))
        if len(salida) >= n:
            break
    return salida

EVAL_ID = eval_identificadores(8 if SMOKE_TEST else 15)

print("consultas de identificador:", len(EVAL_ID))
for consulta, celex in EVAL_ID[:5]:
    print("  %-12s  %s" % (celex, consulta))

consultas de identificador: 15
  32014R0557    Decisión (UE) no 1982/2006
  32014D0554    Decisión (UE) no 2013/743
  32012D0602    Decisión 1219/2011/UE
  32014D0415    Decisión 1313/2013/UE
  32014R0721    Decisión 1982/2006/CE


**Checkpoint.** Tenemos mil actos con su identificador, un parser de dos niveles, una consulta concreta que verificamos que existe, y una vara para medir. Nada de esto usó un modelo todavía.

▶ **Láminas 19–20** (qué guarda el índice; el filtro por metadatos como control de acceso).

## 3 · El embedder — y por qué cambiarlo cuesta

El corpus está en español. El embedder de la Clase 4, `nomic-embed-text`, es inglés-céntrico. Así que cambiamos a **`BAAI/bge-m3`**: multilingüe, 1024 dimensiones.

Y ahí se cobra la invariante de la Clase 4: **un embedder, un espacio**. Los vectores del modelo viejo y los del nuevo no son comparables — ni siquiera tienen la misma cantidad de dimensiones. Cambiar de embedder obliga a **re-indexar el corpus entero**. No es una molestia de implementación: es la razón por la que esa decisión se toma temprano y se cambia poco.

**Y un detalle** `nomic` exige prefijos de tarea —`search_document:` para el corpus, `search_query:` para la consulta. `bge-m3` **no lleva prefijos**: su model card lo dice explícito. La regla que sí es universal no es «poné prefijos», es: **corpus y consulta se embeben con el mismo modelo y el mismo régimen**. Cuál es ese régimen lo dice la model card, no la costumbre.

In [22]:
from sentence_transformers import SentenceTransformer

EMBEDDER_ID    = "BAAI/bge-m3"

embedder = SentenceTransformer(EMBEDDER_ID, device=DISPOSITIVO)

# Dos ajustes que no son opcionales en una T4:
#  - fp16 baja los pesos de 2,3 GB a ~1,2 GB y casi duplica la velocidad.
#  - max_seq_length viene en 8192 por default. No lo bajamos por memoria (sentence-transformers
#    paddea al item mas largo del lote, no a la ventana), sino para que el costo de un lote
#    no dependa de un solo documento largo que se cuele. 1024 deja pasar holgado el pedazo
#    mas grande que vamos a probar en la seccion 4, que son ~900 tokens.
if DISPOSITIVO == "cuda":
    embedder.half()
embedder.max_seq_length = 1024

def embeber(textos, lote=16):
    # normalize_embeddings=True deja los vectores de norma 1, asi el coseno es un producto
    # punto y la busqueda se vuelve una multiplicacion de matrices.
    # astype(float32): con el modelo en fp16 los vectores salen en fp16, y despues FAISS
    # los pide en float32 igual. Convertimos una vez aca en vez de en cada uso.
    return embedder.encode(list(textos), batch_size=lote, normalize_embeddings=True,
                           show_progress_bar=len(textos) > 400).astype(np.float32)

v = embeber(["prueba"])
print("dispositivo:", DISPOSITIVO)
print("dimensiones:", v.shape[1])
vram("con el embedder cargado")
print("ventana declarada por el modelo: 8192 · la que usamos:", embedder.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

dispositivo: cuda
dimensiones: 1024
   [vram] con el embedder cargado    libre   5518 · PyTorch   2.8 GB · este proceso   2808 MiB
ventana declarada por el modelo: 8192 · la que usamos: 1024


### 3.1 · Tres modelos, tres ventanas, tres tokenizadores

En este notebook trabajan **tres modelos distintos**, y conviene separarlos desde ahora porque es fácil confundir lo que hace cada uno:

- **El embedder** (`bge-m3`, el que acabamos de cargar) convierte texto en vectores. Es el que **busca**: embebe los pedazos del corpus una vez, embebe la consulta, y compara. Barato, y se precalcula.
- **El re-puntuador** (un cross-encoder, entra en la sección 11) lee la consulta y un pasaje **juntos** y devuelve un puntaje de qué tan bien se responden. Es el que **reordena** lo que trajo el buscador. Más preciso y mucho más caro: no se puede precalcular, así que sólo corre sobre unos pocos candidatos.
- **El generador** (`qwen3:8b`) es el que **escribe la respuesta**. No busca ni ordena nada: recibe el **texto** de los pasajes recuperados, pegado en su prompt —nunca ve un vector—, y redacta con eso. Es el modelo de lenguaje de siempre; acá su único trabajo es contestar usando lo que el buscador le puso adelante.

Y acá está la trampa que conviene ver antes de cortar nada: **cada uno tiene su propia ventana y su propio tokenizador**. Un texto de 400 tokens para el embedder no son 400 tokens para el generador, así que las cuentas no se trasladan de uno a otro.

Y la ventana que aprieta **no es la obvia**.

In [23]:
from transformers import AutoTokenizer
import transformers

# Vamos a contar los tokens de textos MAS LARGOS que la ventana del modelo -es justamente
# el punto de esta seccion-. El tokenizer avisa con un warning en rojo cada vez, y ese
# warning parece un error sin serlo. Lo apagamos aca y lo explicamos en vez de sufrirlo.
transformers.logging.set_verbosity_error()
REPUNTUADOR_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
TOK = {"embedder": embedder.tokenizer,
       "repuntuador": AutoTokenizer.from_pretrained(REPUNTUADOR_ID)}

# El tokenizer del generador no lo expone Ollama, asi que lo traemos de HuggingFace. Si no
# baja, degradamos a una estimacion por caracteres y lo decimos: mas vale un numero marcado
# como aproximado que un numero falso presentado como exacto.
TOK_GENERADOR_REAL = True
try:
    TOK["generador"] = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
except Exception:
    TOK_GENERADOR_REAL = False
    print("⚠  no bajó el tokenizer de Qwen3: el generador se estima a 3,5 caracteres por token.")

def contar_tokens(texto, quien="repuntuador"):
    """Tokens de 'texto' SEGUN EL MODELO 'quien'. Nunca mezclar: no son la misma unidad."""
    if quien not in TOK:
        return int(len(texto) / 3.5)
    return len(TOK[quien].encode(texto, add_special_tokens=False))

VENTANAS = {"generador": 8192, "embedder": 8192, "repuntuador": 512}
tabla("Tres modelos leen estos textos, y cada uno tiene su propia ventana",
      ["ventana", "quién es"],
      [("generador",   {"ventana": VENTANAS["generador"],   "quién es": "qwen3:8b (la fijamos)"}),
       ("embedder",    {"ventana": VENTANAS["embedder"],    "quién es": "bge-m3"}),
       ("re-puntuador", {"ventana": VENTANAS["repuntuador"], "quién es": "cross-encoder (sec. 11)"})])
print("  El re-puntuador acepta 16 veces menos que el embedder.")
print("  Por eso el tamaño del pedazo lo manda ÉL, y no el modelo que parece más relevante.")
print()

# Detalle que conviene mirar y sorprende: dos de los tres comparten tokenizador.
mismo = contar_tokens("prueba de conteo", "embedder") == contar_tokens("prueba de conteo", "repuntuador")
print("  ¿embedder y re-puntuador cuentan igual?", "sí" if mismo else "no",
      "— los dos son XLM-RoBERTa, así que comparten vocabulario.")
print("  El generador no:", contar_tokens("prueba de conteo", "generador"),
      "tokens contra", contar_tokens("prueba de conteo", "embedder"),
      "para el mismo texto.")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            


Tres modelos leen estos textos, y cada uno tiene su propia ventana
──────────────────────────────────────────────────
                 ventana                  quién es
generador           8192     qwen3:8b (la fijamos)
embedder            8192                    bge-m3
re-puntuador         512   cross-encoder (sec. 11)

  El re-puntuador acepta 16 veces menos que el embedder.
  Por eso el tamaño del pedazo lo manda ÉL, y no el modelo que parece más relevante.

  ¿embedder y re-puntuador cuentan igual? sí — los dos son XLM-RoBERTa, así que comparten vocabulario.
  El generador no: 5 tokens contra 4 para el mismo texto.


**Lo que revela.** El embedder acepta 8.192 tokens, así que uno diría que podemos cortar pedazos grandes y listo. Pero en la sección 11 va a entrar un modelo que **re-puntúa** lo recuperado, y ése corta en **512**. Dieciséis veces menos.

Si cortáramos pensando en el embedder, cada pedazo llegaría al re-puntuador mutilado — y sin un solo mensaje de error, porque truncar es el comportamiento normal de estas librerías, no una excepción. Así que el tamaño de los pedazos lo fija el re-puntuador: **350 tokens**, con margen para el prefijo que les vamos a agregar en la sección 6.

▶ **Láminas 7, 12** (el cambio de embedder y la invariante; el techo duro que no avisa).

## 4 · Partir el documento

Acá se rompe lo primero. Las cinco frases de la sección 1 entraban enteras en el embedder. Un acto europeo, no.

In [24]:
largo_ej = {q: contar_tokens(ejemplo.text, q) for q in ("embedder", "repuntuador", "generador")}
print("Un acto real:", ejemplo.celex_id)
print("  caracteres                :", "{:,}".format(int(ejemplo.largo)).replace(",", "."))
print("  tokens del embedder       :", largo_ej["embedder"],
      "  → cabe en 8.192" if largo_ej["embedder"] <= 8192 else "  → NO cabe en 8.192")
print("  tokens del re-puntuador   :", largo_ej["repuntuador"],
      "  → %.0f veces la ventana de 512" % (largo_ej["repuntuador"] / 512))
print("  tokens del generador      :", largo_ej["generador"],
      "" if TOK_GENERADOR_REAL else "  (estimado)")
RESULTADOS["tokens_acto_ejemplo"] = largo_ej

Un acto real: 32015R0009
  caracteres                : 23.804
  tokens del embedder       : 5357   → cabe en 8.192
  tokens del re-puntuador   : 5357   → 10 veces la ventana de 512
  tokens del generador      : 6752 


**Y hay un problema anterior al tamaño.** Aunque entrara, no conviene: un vector por documento es un vector que promedia veinte temas. Preguntás por uno y recuperás el documento entero, del que el 95 % no tiene nada que ver.

Así que se corta. Y el primer parámetro de un RAG no es el modelo: es **dónde cortás**.

In [25]:
def partir_por_tokens(texto, tam=TAM_HIJO, solape=SOLAPE, quien="repuntuador"):
    """Corta en pedazos de 'tam' tokens con 'solape' de superposicion.

    Se mide en TOKENS y no en caracteres porque el limite de los modelos esta en tokens.
    Cortar a 1.500 caracteres funciona hasta que aparece un texto con muchos numeros,
    tildes o palabras largas -como este- y de golpe cada pedazo pesa el doble."""
    tok = TOK.get(quien)
    if tok is None:
        paso = int((tam - solape) * 3.5)
        return [texto[i:i + int(tam * 3.5)] for i in range(0, len(texto), max(1, paso))]
    ids = tok.encode(texto, add_special_tokens=False)
    pedazos, paso = [], max(1, tam - solape)
    for i in range(0, len(ids), paso):
        trozo = ids[i:i + tam]
        if len(trozo) < 20 and pedazos:      # una cola de menos de 20 tokens no es un pedazo
            break
        pedazos.append(tok.decode(trozo))
    return pedazos

muestra_pedazos = partir_por_tokens(ejemplo.text)
print("el acto de arriba, en", len(muestra_pedazos), "pedazos de", TAM_HIJO, "tokens")
print()
for i, p in enumerate(muestra_pedazos[:5]):
    print(f"pedazo {i}  · {parrafo(p, 150)}", )

el acto de arriba, en 18 pedazos de 350 tokens

pedazo 0  ·    REGLAMENTO (UE) 2015/9 DE LA COMISIÓN de 6 de enero de 2015 que modifica el Reglamento (UE) no 142/2011, por el que se establecen las disposiciones de
   aplicación del Reglamento (CE) no 1069/2009 del Parlamento Europeo y del Consejo por el que se establecen las normas sanitarias aplicables a los
   subproductos animales y los productos derivados no destinados al consumo humano, y la Directiva 97/78/CE del Consejo en cuanto a determinadas muestras
   y unidades exentas de los controles veterinarios en la frontera en virtud de la misma (Texto pertinente a efectos del EEE) LA COMISIÓN EUROPEA, Visto
   el Tratado de Funcionamiento de la Unión Europea, Vista el Acta de Adhesión de Croacia y, en particular, su artículo 50, Visto el Reglamento (CE) no
   1069/2009 del Parlamento Europeo y del Consejo, de 21 de octubre de 2009, por el que se establecen las normas sanitarias aplicables a los subproductos
   animales y los product

### 4.1 · El corte que parte la respuesta al medio  ·  *fixed-size chunking* y solape

El corte es **ciego**: cuenta tokens, no entiende el texto. Tarde o temprano cae justo en el medio de la oración que contestaba la pregunta, y entonces ninguna de las dos mitades la contesta — y ninguna de las dos se parece mucho a la pregunta.

In [26]:
# Buscamos en el corpus una oracion larga y la partimos al medio a proposito, para medir
# cuanto pierde cada mitad respecto de la oracion entera.
oracion = None
for t in list(TEXTO_DE.values())[:40]:
    for o in re.split(r"(?<=\.)\s+", re.sub(r"\s+", " ", t)):
        if 220 <= len(o) <= 400 and o.count(",") >= 2:
            oracion = o.strip()
            break
    if oracion:
        break
if oracion is None:
    # Con este corpus no deberia pasar, pero una celda que depende de encontrar algo
    # tiene que decir que no lo encontro en vez de tirar un TypeError diez lineas abajo.
    oracion = re.sub(r"\s+", " ", list(TEXTO_DE.values())[0])[:320]
    print("⚠  no encontré una oración con la forma buscada; uso un fragmento cualquiera.")

mitad = len(oracion) // 2
izq, der = oracion[:mitad].strip(), oracion[mitad:].strip()

# La pregunta se arma con terminos distintivos de LAS DOS MITADES. Es el caso que importa:
# una pregunta cuya respuesta necesita las dos partes de la oracion. Con una pregunta
# generica -"que establece esto?"- todo da parecido y el experimento no muestra nada.
def distintivos(texto, n=3):
    palabras = [w.strip(".,;:()[]\u00ab\u00bb").lower() for w in texto.split()]
    return sorted({w for w in palabras if len(w) >= 9 and w.isalpha()}, reverse=True)[:n]

de_izq, de_der = distintivos(izq), distintivos(der)
pregunta_ej = "\u00bfQu\u00e9 se establece sobre %s?" % ", ".join(de_izq + de_der)

vecs = embeber([oracion, izq, der, pregunta_ej])
cos_entera, cos_izq, cos_der = (float(vecs[3] @ vecs[i]) for i in (0, 1, 2))

mostrar("LA ORACI\u00d3N ENTERA", recortar(oracion, 300))
mostrar("MITAD IZQUIERDA", recortar(izq, 200))
mostrar("MITAD DERECHA", recortar(der, 200))
print("la pregunta, armada con t\u00e9rminos de las dos mitades:")
print("  ", pregunta_ej)
print("   de la izquierda:", ", ".join(de_izq) or "(ninguno)")
print("   de la derecha  :", ", ".join(de_der) or "(ninguno)")
tabla("Coseno contra la pregunta", ["coseno"],
      [("oraci\u00f3n entera", {"coseno": "%.3f" % cos_entera}),
       ("mitad izquierda", {"coseno": "%.3f" % cos_izq}),
       ("mitad derecha", {"coseno": "%.3f" % cos_der})])
print("  Ninguna de las dos mitades llega a lo que valía la oración entera.")
print("  El corte no destruyó la información: destruyó la POSIBILIDAD DE ENCONTRARLA.")

──────────────────────────────────────────────────────────────────────────
LA ORACIÓN ENTERA
──────────────────────────────────────────────────────────────────────────
(2) En el Reglamento (UE) no 142/2011 de la Comisión (2) se establecen las
disposiciones de aplicación del Reglamento (CE) no 1069/2009, incluidas
normas para adoptar métodos alternativos de uso o eliminación de
subproductos animales o productos derivados, y las condiciones de
comercialización de ab…

──────────────────────────────────────────────────────────────────────────
MITAD IZQUIERDA
──────────────────────────────────────────────────────────────────────────
(2) En el Reglamento (UE) no 142/2011 de la Comisión (2) se establecen las
disposiciones de aplicación del Reglamento (CE) no 1069/2009, incluidas
normas para adoptar métodos a

──────────────────────────────────────────────────────────────────────────
MITAD DERECHA
──────────────────────────────────────────────────────────────────────────
lternativos de uso o 

**El parche barato es el solape.** Cada pedazo se lleva un pedacito del anterior, del orden del 10 al 20 %. Si el corte cayó mal, la oración completa sobrevive en el pedazo vecino. Cuesta un 15 % más de índice y arregla la mayoría de los casos.

**Y lo que el solape no arregla.** El pedazo perdió sus **referentes**. «La empresa deberá presentarlo antes del trimestre siguiente» — ¿qué empresa?, ¿presentar qué?, ¿qué trimestre? Eso estaba tres párrafos más arriba, y el solape de cincuenta tokens no llega hasta ahí. Ese agujero queda abierto: lo retomamos en las secciones 6 y 8.

In [27]:
sin_solape  = partir_por_tokens(ejemplo.text, solape=0)
con_solape  = partir_por_tokens(ejemplo.text, solape=SOLAPE)
print("sin solape : %2d pedazos" % len(sin_solape))
print("con solape : %2d pedazos  (+%.0f%% de índice)" % (
    len(con_solape), 100.0 * (len(con_solape) - len(sin_solape)) / len(sin_solape)))
print()
print("El final del primer pedazo y el principio del segundo, con solape:")
print(parrafo("…" + con_solape[0][-180:]))
print()
print(parrafo(con_solape[1][:180] + "…"))

sin solape : 16 pedazos
con solape : 18 pedazos  (+12% de índice)

El final del primer pedazo y el principio del segundo, con solape:
   … 11; su artículo 21, apartado 6, letra d); su artículo 23, apartado 3; su
   artículo 27, letra c); su artículo 31, apartado 2; su artículo 40, letra
   f); su artículo 41, apartado 3, y

   ); su artículo 20, apartado 11; su artículo 21, apartado 6, letra d); su
   artículo 23, apartado 3; su artículo 27, letra c); su artículo 31,
   apartado 2; su artículo 40, letra f); su…


### 4.2 · Cortar por estructura, dimensionar por tarea  ·  *structure-aware chunking*

El corte ciego es el que hay cuando el documento no te dice nada. Pero éste **sí** te dice: los artículos están marcados. Un artículo es una unidad de sentido escrita por alguien que decidió dónde termina una idea.

Así que cortamos por artículo, y sólo partimos por tokens los artículos que se pasan de 350.

**¿Y si el documento no tuviera costuras?** Entonces hay que adivinarlas, y la técnica se llama *semantic chunking*: se embeben las oraciones consecutivas y se corta donde la distancia entre una y la siguiente se dispara, que es donde presumiblemente cambia el tema. Acá no hace falta —este corpus trae las costuras escritas, y usar un modelo para adivinar lo que el legislador ya marcó es pagar de más—, pero es la alternativa a tener a mano cuando el material llega sin estructura.

In [28]:
def pedazos_de(texto, celex, por_estructura=True, tam=TAM_HIJO, solape=SOLAPE):
    """Los pedazos de un acto, con sus metadatos. Un pedazo es {id, texto, celex, seccion}."""
    salida = []
    unidades = partir_en_articulos(texto) if por_estructura else [("ciego", texto)]
    for etiqueta, cuerpo in unidades:
        # Solo partimos lo que se pasa de la ventana: si el articulo entra, se respeta entero.
        trozos = ([cuerpo] if por_estructura and contar_tokens(cuerpo) <= tam
                  else partir_por_tokens(cuerpo, tam, solape))
        for j, tr in enumerate(trozos):
            salida.append({"id": "%s:%d" % (celex, len(salida)), "texto": tr,
                           "celex": celex, "seccion": etiqueta, "orden": j})
    return salida

def cortar_corpus(marco, por_estructura=True, tam=TAM_HIJO, solape=SOLAPE):
    out = []
    for r in marco.itertuples(index=False):
        out.extend(pedazos_de(r.text, str(r.celex_id), por_estructura, tam, solape))
    return out

# El A/B corre sobre una PORCION del corpus, no sobre todo: lo que comparamos es la forma de
# cortar, y para eso no hace falta el corpus entero. Asi el experimento cuesta minutos y no
# media hora, que es la diferencia entre que alguien lo corra y que no.
# La porcion tiene que contener los documentos que el eval set pregunta. Si se toman los
# primeros N actos a secas, la mayoria de los golds queda afuera y las tres variantes
# aciertan lo mismo -lo poco alcanzable- por una razon que no tiene nada que ver con como
# se corta. Eso no es un empate: es un techo, y se lee igual que un empate.
_golds_ab = {g for _, g in EVAL_SET[:N_EVAL]}
_celex    = corpus.celex_id.astype(str)
porcion   = pd.concat([corpus[_celex.isin(_golds_ab)],
                       corpus[~_celex.isin(_golds_ab)].head(max(0, N_AB - len(_golds_ab)))])
_alcanzables = sum(1 for _, g in EVAL_SET[:N_EVAL]
                   if g in set(porcion.celex_id.astype(str)))
print("porcion del A/B: %d actos · golds alcanzables: %d de %d"
      % (len(porcion), _alcanzables, N_EVAL))
if _alcanzables < N_EVAL:
    print("  OJO: %d preguntas son imposibles para las tres variantes por igual."
          % (N_EVAL - _alcanzables))
print()
variantes = {
    "ciego 350 tokens":     cortar_corpus(porcion, por_estructura=False, tam=350),
    "ciego 900 tokens":     cortar_corpus(porcion, por_estructura=False, tam=900),
    "por artículo (≤350)":  cortar_corpus(porcion, por_estructura=True,  tam=350),
}
for k, v in variantes.items():
    largos = [contar_tokens(p["texto"]) for p in v[:300]]
    print("%-22s %5d pedazos · mediana %3d tokens · máximo %4d" % (
        k, len(v), int(np.median(largos)), int(np.max(largos))))

porcion del A/B: 400 actos · golds alcanzables: 30 de 30

ciego 350 tokens        5713 pedazos · mediana 350 tokens · máximo  351
ciego 900 tokens        2153 pedazos · mediana 900 tokens · máximo  901
por artículo (≤350)     7675 pedazos · mediana 256 tokens · máximo  351


### 4.3 · El techo duro: se trunca en silencio

Antes de medir, una advertencia que vale para todo el notebook. Si un pedazo excede la ventana del modelo que lo procesa, **el sobrante se descarta**. Sin excepción, sin warning y sin log. El pedazo de 900 tokens que acabamos de construir entra al embedder completo —su ventana es amplia— pero al re-puntuador de la sección 11 le llegaría cortado a la mitad.

Es una falla que no avisa, y por eso es peor que una que se rompe.

In [29]:
# Lo mostramos en vez de contarlo: encodeamos un pedazo largo con la ventana del re-puntuador
# y contamos cuantos tokens sobrevivieron. No hay excepcion, no hay warning: hay perdida.
largo = variantes["ciego 900 tokens"][0]["texto"]
tok   = TOK["repuntuador"]
entero    = tok.encode(largo, add_special_tokens=False)
truncado  = tok(largo, truncation=True, max_length=512)["input_ids"]

print("un pedazo de %d tokens, leído por un modelo con ventana de 512:" % len(entero))
print("  sobreviven :", len(truncado), "tokens")
print("  se pierden : %d (%.0f%% del pedazo)" % (
    len(entero) - len(truncado), 100.0 * (len(entero) - len(truncado)) / len(entero)))
print()
print("lo último que el modelo llega a leer:")
print(parrafo("…" + tok.decode(truncado[-60:])))
print("lo primero que ya no:")
print(parrafo(tok.decode(entero[len(truncado):len(truncado) + 60]) + "…"))
print()
print("Ni una excepción, ni un warning. Por eso el tamaño del hijo lo manda el re-puntuador.")

un pedazo de 900 tokens, leído por un modelo con ventana de 512:
  sobreviven : 512 tokens
  se pierden : 388 (43% del pedazo)

lo último que el modelo llega a leer:
   …las publicaciones científicas. (4) Mediante la Decisión no 912/2009/CE
   del Parlamento Europeo y del Consejo (4), la Comunidad decidió aportar una
   contribución financiera al Programa Europeo de Investigación en Metrología
   (PEIM) («el PEIM») equivalente a la de los</s>
lo primero que ya no:
   , pero sin exceder de 200 000 000 EUR, durante el período de vigencia del
   Séptimo programa marco de la Comunidad Europea para acciones de
   investigación, desarrollo tecnológico y demostración (2007 a 2013)
   establecido por la Decisión no 1982/2006/CE del Parlamento Europeo y del
   Consejo (5).…

Ni una excepción, ni un warning. Por eso el tamaño del hijo lo manda el re-puntuador.


### Paréntesis de medición · las cuatro funciones con las que se mide todo

Antes del primer experimento hay que construir el instrumento — y conviene saber desde ahora que **no se vuelve a tocar**: las cuatro funciones de la celda de abajo son las que miden cada técnica del notebook, de acá hasta el final.

La vara es la más barata que existe: **¿el documento correcto entró entre los que recuperamos?** y **¿en qué puesto quedó?**. Sin juez, sin modelo, sin nada — sólo el eval set y una cuenta.

Y el buscador es el mismo de la Clase 4: coseno contra todos, ordenar, quedarse con los primeros. En la sección 5 vamos a ver por qué eso no escala; por ahora alcanza.

In [30]:
# Las cuatro funciones con las que se mide TODO el notebook. Se definen una sola vez
# aca y no se vuelven a tocar: cada tecnica que agreguemos de la seccion 6 en adelante
# se evalua con estas mismas cuatro, sobre el mismo eval set y con el mismo k. Que la
# vara no cambie es lo unico que hace comparables entre si las filas de la tabla final.
#
#   indice_plano  construye el indice: la matriz de vectores
#   buscar_plano  busca en esa matriz
#   a_documentos  traduce pedazos a documentos, que es la unidad que se evalua
#   medir         corre el eval set entero y arma una fila de la tabla

def indice_plano(pedazos, lote=16):
    """Embebe una lista de pedazos y devuelve el indice mas simple que existe.

    Es la lista de la Clase 4, mas grande: no hay base de datos, hay una matriz de
    numpy en memoria. "Plano" quiere decir sin atajos: buscar es compararse contra
    TODOS los vectores, sin excepcion. Eso lo hace exacto, y es lo que no escala.
    La seccion 5 lo mide y muestra los tres escalones para salir de aca.

    Recibe:
      pedazos  lista de dicts {id, texto, celex, seccion, orden}
      lote     cuantos textos procesa el embedder por pasada

    Devuelve:
      (M, pedazos), con M un array (len(pedazos) x 1024) de vectores normalizados,
      donde la fila i corresponde a pedazos[i]. Devolvemos tambien la lista porque
      una matriz sin sus pedazos no sirve para nada: el par tiene que viajar junto.
    """
    M = embeber([p["texto"] for p in pedazos], lote=lote)
    return M, pedazos


def buscar_plano(M, pedazos, pregunta, k=K_ENTREGO):
    """Devuelve los k pedazos mas parecidos a la pregunta, por coseno.

    Recibe:
      M         la matriz que devolvio indice_plano()
      pedazos   la lista de pedazos, en el MISMO orden que las filas de M
      pregunta  el texto de la consulta, tal cual: bge-m3 no lleva prefijos
      k         cuantos pedazos devolver

    Devuelve:
      lista de k pedazos ordenados de mayor a menor coseno, cada uno con un campo
      "score" agregado. Son copias: la lista original no se toca.
    """
    # Vectores normalizados => el coseno es el producto punto, y buscar es multiplicar
    # una matriz por un vector. Es literalmente el buscar() de la Clase 4.
    q = embeber([pregunta])[0]
    scores = M @ q
    orden = np.argsort(scores)[::-1][:k]
    return [dict(pedazos[int(i)], score=float(scores[int(i)])) for i in orden]


def a_documentos(recuperados):
    """Traduce una lista de pedazos a la lista de documentos de los que salieron.

    El eval set etiqueta DOCUMENTOS (un celex), no pedazos, asi que esta funcion es el
    puente entre lo que devuelve el buscador y lo unico que sabemos evaluar.

    Recibe:
      recuperados  lista de pedazos, ya ordenada por relevancia

    Devuelve:
      lista de celex sin repetir, conservando el orden de llegada. Tres pedazos del
      mismo acto cuentan como un acto, y ese acto queda en el puesto del primero.
    """
    vistos, salida = set(), []
    for r in recuperados:
        if r["celex"] not in vistos:
            vistos.add(r["celex"])
            salida.append(r["celex"])
    return salida


def medir(recuperador, k=K_ENTREGO, evaluacion=None):
    """Corre el eval set entero contra un recuperador y arma una fila de la tabla.

    Recibe:
      recuperador  funcion (pregunta, k) -> lista de pedazos. TODA tecnica del
                   notebook se envuelve en una funcion con esta firma, y por eso
                   todas terminan midiendose con este mismo codigo.
      k            cuantos pasajes se le piden al recuperador por consulta
      evaluacion   lista de (pregunta, celex_gold). Por default las preguntas
                   tematicas; pasandole EVAL_ID mide el bloque de identificadores
                   sin cambiar una linea. Si fueran dos funciones, dos filas de la
                   tabla podrian diferir en COMO se midieron y no en que se midio.

    Devuelve un dict con las columnas de la tabla:
      "¿entró?"   aciertos/total: en cuantas preguntas el documento correcto quedo
                  entre los recuperados
      "tasa"      lo mismo en fraccion, para comparar filas entre si
      "puesto"    posicion promedio del documento correcto, promediada SOLO SOBRE LOS
                  ACIERTOS. Por eso se lee siempre junto a "¿entró?": una tecnica que
                  acierta menos puede mostrar un puesto mejor y no ser mejor.
      "contexto"  mediana de tokens que le llegarian al generador. No mide
                  recuperacion: mide QUE LE LLEGA. Esta porque hay una tecnica, la de
                  la seccion 8, cuyo aporte entero vive ahi y seria invisible en las
                  otras dos columnas.

    El eval set etiqueta DOCUMENTOS, asi que colapsamos los pedazos a documentos antes
    de buscar el gold: de otro modo cada tecnica que cambia el tamano del pedazo
    cambiaria la metrica sin haber mejorado nada.
    """
    usadas = EVAL_SET[:N_EVAL] if evaluacion is None else evaluacion
    aciertos, puestos, contextos = 0, [], []
    for pregunta, gold in usadas:
        pasajes = recuperador(pregunta, k)
        contextos.append(sum(contar_tokens(p["texto"], "generador") for p in pasajes))
        docs = a_documentos(pasajes)
        if gold in docs:
            aciertos += 1
            puestos.append(docs.index(gold) + 1)
    return {"¿entró?": "%d/%d" % (aciertos, len(usadas)),
            "tasa": aciertos / max(1, len(usadas)),
            "puesto": "%.1f" % (sum(puestos) / len(puestos)) if puestos else "—",
            "contexto": "%d tok" % int(np.median(contextos)) if contextos else "—"}

### 4.4 · Medir: la primera comparación del día

Con el instrumento listo, el experimento: **las mismas preguntas contra tres formas de cortar**, y una sola variable de diferencia entre las filas.

In [31]:
# El A/B: la misma vara, tres formas de cortar. Una variable por vez.
# La columna de tiempo dice "embeber+indexar" y no "indexar" a secas porque mide los
# dos pasos juntos. En la lamina de las dos fases, "indexar" es solo el ultimo paso
# -guardar los vectores-; lo que se lleva los segundos aca es el anterior, embeber.
filas_ab = []
# Los tres indices quedan guardados: la celda de abajo vuelve a consultarlos para
# mirar UNA pregunta en detalle, y re-embeberlos costaria los mismos cuatro minutos.
INDICES_AB = {}
for nombre, pedazos in variantes.items():
    t0 = time.time()
    M, ps = indice_plano(pedazos)
    INDICES_AB[nombre] = (M, ps)
    # k=1, y no los 5 de siempre. Para comparar FORMAS DE CORTAR hay que preguntar si el
    # documento correcto sale PRIMERO: con cinco lugares cualquier corte razonable lo mete
    # adentro y las tres filas empatan -no porque den lo mismo, sino porque la vara no
    # discrimina-. Es la misma trampa que la saturacion, del otro lado.
    r = medir(lambda q, k, M=M, ps=ps: buscar_plano(M, ps, q, k), k=1)
    filas_ab.append((nombre, {"pedazos": len(pedazos), "sale 1º": r["¿entró?"],
                              "embeber+indexar": "%.0f s" % (time.time() - t0)}))

tabla("Tres formas de cortar · ¿sale PRIMERO el documento correcto?",
      ["pedazos", "sale 1º", "embeber+indexar"], filas_ab)
print(BANNER_EVAL)
print("  El tamaño no se hereda de un tutorial: se mide. Y se mide con TU corpus.")
RESULTADOS["ab_chunking"] = filas_ab

Batches:   0%|          | 0/358 [00:00<?, ?it/s]

Batches:   0%|          | 0/135 [00:00<?, ?it/s]

Batches:   0%|          | 0/480 [00:00<?, ?it/s]


Tres formas de cortar · ¿sale PRIMERO el documento correcto?
───────────────────────────────────────────────────────────
                        pedazos   sale 1º   embeber+indexar
ciego 350 tokens           5713     24/30              87 s
ciego 900 tokens           2153     23/30              91 s
por artículo (≤350)        7675     23/30              87 s


  El tamaño no se hereda de un tutorial: se mide. Y se mide con TU corpus.


In [32]:
# La tabla de arriba dice SI acerto. Esta celda dice QUE trajo, que es lo que un
# numero no puede mostrar: la misma pregunta contra los tres indices, y el pedazo
# que cada corte puso primero.
# parrafo() y recortar() salen de los ayudantes de la seccion 0.2.

PREG_AB, GOLD_AB = EVAL_SET[0]

print("LA PREGUNTA")
print(parrafo(PREG_AB))
print()
print("EL DOCUMENTO QUE LA RESPONDE:", GOLD_AB)
print(parrafo(titulo_de(TEXTO_DE[GOLD_AB]), lineas=2))

for nombre, (M, ps) in INDICES_AB.items():
    top = buscar_plano(M, ps, PREG_AB, k=1)[0]
    veredicto = "ACERTÓ" if top["celex"] == GOLD_AB else "falló "
    print()
    print("─" * 78)
    print("%-21s %s · trajo %s · %s · coseno %.3f"
          % (nombre, veredicto, top["celex"], top["seccion"][:26], top["score"]))
    print("─" * 78)
    print(parrafo(top["texto"]))

print()
print("Mirá DÓNDE empieza y DÓNDE termina cada uno, no sólo si acertó: el corte ciego")
print("arranca y corta a mitad de oración, el corte por artículo empieza donde empieza")
print("una idea. Eso es lo que la tabla de arriba no puede ver, y lo que va a importar")
print("cuando ese texto sea lo que le pasemos al generador.")

LA PREGUNTA
   ¿Con qué criterio se reparte entre los países el dinero europeo que cubre
   los gastos de vigilancia de la actividad pesquera durante 2010?

EL DOCUMENTO QUE LA RESPONDE: 32010D0352
   DECISIÓN DE LA COMISIÓN — de 22 de junio de 2010

──────────────────────────────────────────────────────────────────────────────
ciego 350 tokens      ACERTÓ · trajo 32010D0352 · ciego · coseno 0.713
──────────────────────────────────────────────────────────────────────────────
   DECISIÓN DE LA COMISIÓN de 22 de junio de 2010 relativa a la participación
   financiera de la Unión en los programas de control, inspección y
   vigilancia de la actividad pesquera realizados por los Estados miembros en
   2010 [notificada con el número C(2010) 3940] (Los textos en lenguas
   alemana, búlgara, danesa, eslovena, española, estonia, finesa, francesa,
   griega, inglesa, italiana, lituana, neerlandesa, polaca, rumana y sueca
   son los únicos auténticos) (2010/352/UE) LA COMISIÓN EUROPEA, Visto el


**Cómo leerlo.** Tres cosas, y una de ellas suele sorprender.

Los pedazos grandes tienen **más adentro**, así que es más probable que contengan la respuesta — pero también más ruido, y el vector promedia más temas. Los chicos son más precisos y se quedan cortos más seguido. No hay un ganador universal: hay un punto que depende del corpus y de las preguntas.

El corte **por estructura** produce **más** pedazos que el ciego del mismo tamaño, y eso sorprende hasta que se lo piensa: un artículo de sesenta tokens ya no se junta con el siguiente para llenar el cupo de trescientos cincuenta, se queda solo. Se paga con un índice más grande, y se compra algo que el corte ciego pisa: **un artículo entero es una idea entera**, y ninguna pieza cruza el límite que el legislador escribió.

> **Para discutir:** si el corte por estructura no gana en esta tabla, ¿lo descartarías? Fijate que la tabla mide una sola cosa —si el documento correcto entró— y el corte por estructura mejora algo que esta vara no ve: **qué le llega al generador**. Un artículo entero se lee; medio artículo cortado al medio, no. Volvemos a esto en la sección 8.

▶ **Láminas 8–12** (el documento no entra; el tamaño en tokens; el solape; dónde cortar; el techo que no avisa).

## 5 · Guardar los vectores

El buscador de arriba compara la consulta **contra todos** los vectores, uno por uno. Con dos mil pedazos eso es instantáneo. Veamos qué pasa cuando no lo es.

In [33]:
# De aca en adelante trabajamos con el corpus entero y con el corte por estructura.
PEDAZOS = cortar_corpus(corpus, por_estructura=True)
print("pedazos del corpus completo:", len(PEDAZOS))

t0 = time.time()
MATRIZ, PEDAZOS = indice_plano(PEDAZOS)
RESULTADOS["t_embeber"] = time.time() - t0
print("embebidos en %.0f s · matriz %s · %.1f MB en memoria" % (
    RESULTADOS["t_embeber"], MATRIZ.shape, MATRIZ.nbytes / 1e6))
vram("despues de embeber el corpus")

pedazos del corpus completo: 30730


Batches:   0%|          | 0/1921 [00:00<?, ?it/s]

embebidos en 348 s · matriz (30730, 1024) · 125.9 MB en memoria
   [vram] despues de embeber el corpus libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB


**Lo que hay que ver.** Ese número de megabytes es el corpus entero convertido en vectores, y está **en la memoria de esta sesión**. Si cerrás el notebook, se fue: hay que volver a embeber todo.

Y hay algo peor que perderlo, que es no poder actualizarlo. Agregar un documento significa recalcular y reasignar la matriz completa. Borrar uno, lo mismo. Ahora midamos lo otro: cuánto tarda una consulta.

In [34]:
consultas_ej = [p for p, _ in EVAL_SET][:20]

def cronometrar(fn, repeticiones=20):
    q = embeber(consultas_ej[:repeticiones])
    t0 = time.time()
    for i in range(repeticiones):
        fn(q[i % len(q)])
    return (time.time() - t0) / repeticiones * 1000   # ms por consulta

ms_plano = cronometrar(lambda q: np.argsort(MATRIZ @ q)[::-1][:K_ENTREGO])
print("lista en memoria : %.1f ms por consulta sobre %d pedazos" % (ms_plano, len(PEDAZOS)))
print()
print("extrapolado, si el corpus fuera de un millón de pedazos:")
print("  ~%.1f s por consulta · %.1f GB sólo de vectores" % (
    ms_plano / 1000 * 1e6 / len(PEDAZOS), 1e6 * 1024 * 4 / 1e9))
print()
print("Y encima: se pierde al cerrar el notebook, no se puede actualizar de a uno,")
print("y no hay forma de filtrar por metadatos antes de buscar.")

lista en memoria : 3.2 ms por consulta sobre 30730 pedazos

extrapolado, si el corpus fuera de un millón de pedazos:
  ~0.1 s por consulta · 4.1 GB sólo de vectores

Y encima: se pierde al cerrar el notebook, no se puede actualizar de a uno,
y no hay forma de filtrar por metadatos antes de buscar.


### 5.1 · Los escalones

De la lista en memoria a un vector store hay tres escalones. Conviene no saltárselos, porque cada uno resuelve un problema **distinto** — y porque se los suele presentar como tres productos que compiten, cuando en realidad se apilan: el tercero contiene al segundo.

**Primero una palabra que venimos usando sin definir: «plano».** Un índice es plano cuando **no tiene ninguna estructura de atajos**: los vectores están uno al lado del otro y buscar significa compararse contra **todos**. Es lo que hace nuestro `indice_plano()` desde la sección 4, y lo que hacía el `buscar()` de la Clase 4. Tiene una virtud y un defecto, y los dos son consecuencia de lo mismo: es **exacto** —si mirás todos, el mejor no se te puede escapar— y **crece con el corpus** —el doble de pedazos es el doble de trabajo—.

**Escalón 1 · FAISS con un índice plano.** FAISS es una *librería* de búsqueda por similitud. Con un índice plano hace exactamente la misma cuenta que nosotros —producto punto contra todos, resultado exacto— pero en C++ vectorizado. No cambia **qué** se calcula ni **qué** devuelve: cambia **dónde** corre el cálculo.

**Escalón 2 · El índice aproximado.** HNSW y parientes dejan de ser planos: arman un grafo de vecinos y lo navegan, mirando sólo una fracción de los vectores. Acá sí cambia el resultado —puede devolver el quinto mejor en vez del cuarto— y a cambio la búsqueda deja de crecer con el corpus. Ojo con el vocabulario: **HNSW no es un producto, es un algoritmo**, y está adentro de casi todos los productos. A la familia entera se la llama **ANN** —*approximate nearest neighbor*, vecino más cercano aproximado—, por oposición a la búsqueda **exacta** o *brute force*, que es la de los escalones anteriores. Cuando una base de datos dice que tiene «índice vectorial», casi siempre quiere decir ANN.

**Escalón 3 · El vector store.** Chroma, y también pgvector o Qdrant. Agrega lo que ninguno de los dos anteriores da: **persiste** en disco, **se actualiza** de a un documento, y guarda **metadatos** al lado de cada vector para poder filtrar antes de buscar. Por dentro su índice es HNSW: el escalón 3 no reemplaza al 2, lo empaqueta.

| escalón | qué cambia | qué **no** cambia |
|---|---|---|
| FAISS plano | dónde corre el cálculo | el resultado, idéntico al del bucle en Python — y a esta escala, casi tampoco el tiempo |
| índice aproximado | deja de crecer con el tamaño del corpus | — y ahora el resultado es aproximado |
| vector store | persiste, se actualiza, filtra por metadatos | la búsqueda por dentro: sigue siendo HNSW |

**Y dos cosas contraintuitivas, para no llevarse la idea equivocada.** La primera la vamos a medir en la celda de abajo: **el escalón 1 tampoco es más rápido** que la lista de numpy, al menos a esta escala. `MATRIZ @ q` ya llama a BLAS —la misma familia de código optimizado que FAISS usa por dentro—, y lo que domina el tiempo no es la cuenta sino **leer los 126 MB de vectores**, que los dos tienen que recorrer igual. La ganancia de FAISS aparece con muchos más vectores, con muchas consultas a la vez, o cuando el índice deja de ser plano.

La segunda: **el único que acelera de verdad es el escalón 3 — pero no por ser un store.** Chroma va a medir bastante menos que los otros dos, y la razón es que por dentro es **aproximado**: es el único de los tres que no mira todos los vectores. La ganancia no viene de la carrocería, viene del motor que trae adentro, que es el escalón 2. Y aun así hablamos de milisegundos, contra los **segundos** que tarda después el generador — por eso un vector store no se elige por velocidad.

In [35]:
import faiss

# IndexFlatIP = producto punto exacto, fuerza bruta. Con vectores normalizados es el coseno.
# El paquete es faiss-cpu: ni "faiss" (build no oficial de 2019) ni "faiss-gpu" (congelado).
indice_faiss = faiss.IndexFlatIP(MATRIZ.shape[1])
indice_faiss.add(np.ascontiguousarray(MATRIZ, dtype=np.float32))

ms_faiss = cronometrar(lambda q: indice_faiss.search(
    np.ascontiguousarray(q.reshape(1, -1), dtype=np.float32), K_ENTREGO))

# Verificamos que devuelve lo mismo: el punto es que el calculo es identico.
q0 = embeber([consultas_ej[0]])[0]
top_np    = list(np.argsort(MATRIZ @ q0)[::-1][:K_ENTREGO])
top_faiss = list(indice_faiss.search(np.ascontiguousarray(q0.reshape(1, -1), dtype=np.float32),
                                     K_ENTREGO)[1][0])
print("¿mismo resultado que a mano?", [int(x) for x in top_np] == [int(x) for x in top_faiss])
print("FAISS plano: %.1f ms por consulta" % ms_faiss)

¿mismo resultado que a mano? True
FAISS plano: 7.7 ms por consulta


In [36]:
import chromadb

# PersistentClient escribe en disco: el indice sobrevive al cierre del notebook.
cliente = chromadb.PersistentClient(path="/content/chroma_clase08")
try:
    cliente.delete_collection("actos")
except Exception:
    pass
# embedding_function=None a proposito: nosotros le pasamos los vectores ya calculados, y
# el default de Chroma instancia un modelo ONNX propio que reserva memoria por su cuenta.
coleccion = cliente.create_collection("actos", metadata={"hnsw:space": "cosine"},
                                      embedding_function=None)

# Junto a cada vector guardamos EL TEXTO y LOS METADATOS. Eso es lo que convierte al indice
# en algo con lo que se puede trabajar: de aca salen la cita con fuente, el filtro previo,
# y el borrado o actualizacion por documento.
t0 = time.time()
LOTE = 2000
for i in range(0, len(PEDAZOS), LOTE):
    trozo = PEDAZOS[i:i + LOTE]
    coleccion.add(
        ids=[p["id"] for p in trozo],
        embeddings=[MATRIZ[i + j].tolist() for j in range(len(trozo))],
        documents=[p["texto"] for p in trozo],
        metadatas=[{"celex": p["celex"], "seccion": p["seccion"][:60]} for p in trozo],
    )
RESULTADOS["t_indexar_chroma"] = time.time() - t0

ms_chroma = cronometrar(lambda q: coleccion.query(query_embeddings=[q.tolist()],
                                                  n_results=K_ENTREGO))
vram("con Chroma levantado")
tabla("El mismo corpus, tres veces", ["ms por consulta", "exacto", "persiste", "filtra"],
      [("lista en memoria", {"ms por consulta": "%.1f" % ms_plano, "exacto": "sí",
                             "persiste": "no", "filtra": "no"}),
       ("FAISS plano",      {"ms por consulta": "%.1f" % ms_faiss, "exacto": "sí",
                             "persiste": "a mano", "filtra": "no"}),
       ("Chroma",           {"ms por consulta": "%.1f" % ms_chroma, "exacto": "aprox.",
                             "persiste": "sí", "filtra": "sí"})])
# La conclusion se calcula. En una corrida la lista gana, en otra gana Chroma: lo que
# no cambia es el orden de magnitud, y esa es la afirmacion que queremos hacer.
_spread = max(ms_plano, ms_faiss, ms_chroma) - min(ms_plano, ms_faiss, ms_chroma)
print("  A esta escala la búsqueda exacta sobra: entre el más rápido y el más lento hay")
print("  %.1f ms, y la llamada al generador que viene después tarda segundos." % _spread)
print("  El vector store no se elige por velocidad. Se elige por las otras tres columnas.")

   [vram] con Chroma levantado       libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB

El mismo corpus, tres veces
─────────────────────────────────────────────────────────────────
                     ms por consulta   exacto   persiste   filtra
lista en memoria                 3.2       sí         no       no
FAISS plano                      7.7       sí     a mano       no
Chroma                           3.4   aprox.         sí       sí

  A esta escala la búsqueda exacta sobra: entre el más rápido y el más lento hay
  4.6 ms, y la llamada al generador que viene después tarda segundos.
  El vector store no se elige por velocidad. Se elige por las otras tres columnas.


### 5.2 · Qué guarda el índice, y dos cosas que no son obvias

Un registro del índice tiene **tres** partes: el texto del pedazo, sus metadatos y el vector. La que solemos olvidar es la primera.

In [37]:
reg = coleccion.get(ids=[PEDAZOS[3]["id"]], include=["documents", "metadatas", "embeddings"])
print("id        :", reg["ids"][0])
print("metadatos :", reg["metadatas"][0])
print("vector    : [%s, …]  (%d dimensiones)" % (
    ", ".join("%.3f" % x for x in reg["embeddings"][0][:5]), len(reg["embeddings"][0])))
print("texto     :")
print(parrafo(reg["documents"][0], lineas=4))
print()
print("Dos consecuencias que no son obvias:")
print("  1. Indexar creó una SEGUNDA COPIA del corpus. El store guarda el texto, no sólo")
print("     el vector. Hay que versionarla, borrarla y cuidarla como al original.")
print("  2. El filtro por metadatos ES el control de acceso de un índice compartido.")

id        : 32015R0009:3
metadatos : {'celex': '32015R0009', 'seccion': 'encabezado'}
vector    : [-0.002, 0.019, -0.008, 0.041, -0.044, …]  (1024 dimensiones)
texto     :
   con los Estados miembros y las organizaciones de las partes interesadas,
   la opción de los Estados miembros de aumentar el volumen hasta un máximo
   de 50 kg por semana se suprimirá cuando la excepción transitoria se
   convierta en opción permanente. Procede asimismo suprimir el artículo 36, …

Dos consecuencias que no son obvias:
  1. Indexar creó una SEGUNDA COPIA del corpus. El store guarda el texto, no sólo
     el vector. Hay que versionarla, borrarla y cuidarla como al original.
  2. El filtro por metadatos ES el control de acceso de un índice compartido.


In [38]:
# El filtro corre ANTES de la busqueda. En un indice compartido entre usuarios, esto no es
# una comodidad: es lo unico que impide que un top-k le devuelva a cualquiera un pasaje
# que no deberia ver. Un top-k sin filtro es una filtracion esperando a pasar.
un_celex = str(corpus.iloc[0].celex_id)
q = embeber(["obligaciones de los Estados miembros"])[0]

sin_filtro  = coleccion.query(query_embeddings=[q.tolist()], n_results=3)
con_filtro  = coleccion.query(query_embeddings=[q.tolist()], n_results=3,
                              where={"celex": un_celex})

print("sin filtro  →", [m["celex"] for m in sin_filtro["metadatas"][0]])
print("con filtro  →", [m["celex"] for m in con_filtro["metadatas"][0]], " (sólo", un_celex + ")")

sin filtro  → ['32011L0016', '32015L2193', '32013D0448']
con filtro  → ['32015R0009', '32015R0009', '32015R0009']  (sólo 32015R0009)


**El diagnóstico.** Las dos consultas son la misma y el índice es el mismo: lo único que cambió es que la segunda declaró de qué documento quiere resultados. Eso corre **antes** de la búsqueda, no después.

En un sistema real con un índice compartido entre usuarios, esa cláusula no es una comodidad de programación: es lo único que separa a un usuario de los documentos de otro. Un top-k sin filtro es una filtración esperando a pasar, y no va a dar ningún error cuando pase.

> **Para discutir:** ¿dónde debería vivir ese filtro — en el código que arma la consulta, o en el índice? Si vive en el código, alcanza con que alguien se olvide de ponerlo en un endpoint nuevo.

### 5.3 · La consulta que el coseno no encuentra

Ahora la tercera falla, y es la contraintuitiva. Las dos anteriores eran de escala: el documento no entraba, la lista no escalaba. Ésta no se arregla con más corpus ni con un embedder mejor.

Buscamos una **cita normativa literal**. No un tema: una referencia exacta, de las que aparecen dentro del texto de los actos.

In [39]:
resultados_id = buscar_plano(MATRIZ, PEDAZOS, CONSULTA_ID, k=5)
print("consulta:", CONSULTA_ID)
print("gold    :", GOLD_ID, "\n")
print("TOP-5 POR COSENO")
print("─" * 74)
for i, r in enumerate(resultados_id, 1):
    marca = "  ← es el correcto" if r["celex"] in GOLD_ID else ""
    print("%d. cos=%.3f  %-12s %s%s" % (i, r["score"], r["celex"], parrafo(r["texto"], 92, lineas=2), marca))

# El coseno del pasaje correcto, para poder compararlo con los cinco de arriba.
if GOLD_ID:
    # canon() de los dos lados, como en la seccion 2: el corpus escribe "no 1005/2009"
    # y una comparacion literal contra otra forma devolveria una lista vacia sin avisar.
    idx_gold = [i for i, p in enumerate(PEDAZOS)
                if p["celex"] in GOLD_ID and canon(CONSULTA_ID) in canon(p["texto"])]
    if idx_gold:
        q = embeber([CONSULTA_ID])[0]
        mejor = max(idx_gold, key=lambda i: float(MATRIZ[i] @ q))
        cos_gold = float(MATRIZ[mejor] @ q)
        print("─" * 74)
        print("el pasaje que SÍ la menciona:  cos=%.3f  %-12s %s" % (
            cos_gold, PEDAZOS[mejor]["celex"], parrafo(PEDAZOS[mejor]["texto"], 80, lineas=2)))
        puesto = int((MATRIZ @ q > cos_gold).sum()) + 1
        print("quedó en el puesto %d de %d." % (puesto, len(PEDAZOS)))
        RESULTADOS["falla_id"] = {"cos_gold": cos_gold, "puesto": puesto,
                                  "cos_top5": [r["score"] for r in resultados_id]}

consulta: Reglamento (CEE) no 3730/87
gold    : ['32012R0121'] 

TOP-5 POR COSENO
──────────────────────────────────────────────────────────────────────────
1. cos=0.599  32012R0826      el artículo 37 del Reglamento (UE) no 1095/2010. HA ADOPTADO EL PRESENTE REGLAMENTO:
   CAPÍTULO I DISPOSICIONES GENERALES
2. cos=0.586  32013R0373      REGLAMENTO DE EJECUCIÓN (UE) No 373/2013 DE LA COMISIÓN de 23 de abril de 2013 por el que se
   aprueba la sustancia activa Candida oleophila, cepa O, con arreglo al Reglamento (CE) no …
3. cos=0.576  32014D0291      los Reglamentos (CEE) no 352/78, (CE) no 165/94, (CE) no 2799/98, (CE) no 814/2000, (CE) no
   1290/2005 y (CE) no 485/2008 (6). HA ADOPTADO LA PRESENTE DECISIÓN:
4. cos=0.575  32010R0453      artículo 133 del Reglamento (CE) no 1907/2006. HA ADOPTADO EL PRESENTE REGLAMENTO:
5. cos=0.574  32012D0199      3286/94 y, en particular, sus artículos 8 y 9; f) el Reglamento (CE) no 385/96 y, en
   particular, su artículo 5, apartados 12 y 13, su 

**El diagnóstico.** Los cinco que volvieron hablan del mismo tema, en el mismo registro, con el mismo vocabulario. Y el pasaje que efectivamente menciona la cita no está entre ellos. No es que salga cuarto: no aparece.

No es un error del embedder. Es **lo que un embedder hace**. Comprimir un pasaje a 1024 números obliga a quedarse con el significado y tirar la forma superficial: qué palabras exactas, en qué orden, con qué puntuación. Para un buscador semántico eso es una virtud — es lo que hace que «cómo se aprueba la materia» encuentre un texto que dice «evaluación». Para una referencia exacta es una pérdida total, porque **la forma superficial era toda la información**.

Un identificador no tiene sinónimos. `1005/2009` no significa nada: sólo es igual a sí mismo.

> **Guardá este beat.** Esta falla queda abierta a propósito. No se arregla con nada de lo que vimos hasta acá, y la vamos a cerrar en la sección 7 — con una idea que parece un retroceso.

▶ **Láminas 13–17** (la lista que no sobrevive; los escalones; qué hace HNSW; la consulta que el coseno no encuentra; por qué falla).

## 6 · Contextualizar antes de embeber

Volvamos a lo que quedó abierto en la sección 4. El corte le sacó al pedazo sus **referentes**: quedó hablando de «la empresa», «dicho plazo», «ese artículo», sin que nada en el pedazo diga cuál.

Esa falla tiene **dos mitades**, y se arreglan con dos técnicas distintas:

- **Del lado de la recuperación:** el pedazo se indexa mutilado. Su vector no dice de qué acto es ni de qué artículo, así que la búsqueda no lo encuentra cuando la pregunta menciona el tema del documento.
- **Del lado de la entrega:** aunque lo encuentres, al generador le llega un fragmento sin contexto, y el generador tampoco sabe de qué habla.

Esta sección arregla la primera: la técnica se conoce como **contextual retrieval**, por el nombre que le puso Anthropic al publicarla. La segunda mitad es la sección 8, y también tiene nombre propio: **parent-document retrieval**.

In [40]:
# Buscamos pedazos que arrancan con un referente colgado. No es un ejemplo inventado:
# es lo que produce cortar cada 350 tokens un texto escrito para leerse entero.
HUERFANOS = re.compile(r"^(dicho|dicha|ese|esa|este|esta|el mismo|la misma|tal|dichos|dichas)\b",
                       re.IGNORECASE)
sueltos = [p for p in PEDAZOS if HUERFANOS.match(p["texto"].strip()[:14])][:3]

if not sueltos:
    sueltos = [p for p in PEDAZOS if p["orden"] > 0][:3]
if not sueltos:
    sueltos = PEDAZOS[1:4]   # ultimo recurso: cualquiera que no sea el primero

for p in sueltos:
    print("─" * 74)
    print("%s · %s" % (p["celex"], p["seccion"][:50]))
    print(parrafo(p["texto"], lineas=4))
print("─" * 74)
print("Leído solo, ninguno de estos dice de qué norma habla. El vector tampoco.")

──────────────────────────────────────────────────────────────────────────
32015R0009 · encabezado
   dicho Reglamento, por los otros medios establecidos en el capítulo IV del
   anexo VI del Reglamento (UE) no 142/2011. De conformidad con lo dispuesto
   en el artículo 36, apartado 3, del Reglamento (UE) no 142/2011, esta
   posibilidad se limita al período transitorio que concluye el 31 de …
──────────────────────────────────────────────────────────────────────────
32011D0874 · encabezado
   dicho Reglamento. (19) Para facilitar la disponibilidad de certificados
   multilingües, es conveniente que el certificado sanitario que se exige
   para la introducción en la Unión sin ánimo comercial de un máximo de cinco
   perros, gatos o hurones se base en los modelos establecidos en la Decisión …
──────────────────────────────────────────────────────────────────────────
32013D0197 · encabezado
   este sentido, la Comisión considera que el contrato inicial es el contrato
   final, y, por lo 

### 6.1 · Arreglar lo que se indexa

El arreglo es **barato y determinístico**: antes de embeber, le pegamos adelante el título del acto y el artículo del que sale. No se toca el embedder ni el generador; sólo cambia **lo que se indexa**.

```
[Decisión de la Comisión de 24 de marzo de 2011 · Artículo 2] la cantidad de sustancias reguladas…
```

Es la demostración más limpia de la idea central del día: mover lo que se recupera, sin tocar ningún modelo.

> **Ojo con los números que circulan.** Anthropic reporta una baja de alrededor de un tercio en la tasa de fallas de recuperación con esta familia de técnicas — pero **eso es de la variante que genera el contexto de cada pedazo con un modelo de lenguaje, uno por pedazo**, no de la versión determinística que implementamos acá. Sobre el atajo más parecido al nuestro el propio post dice que vieron *«very limited gains»*. Por eso lo medimos con nuestra propia vara en vez de heredar un número ajeno.

In [41]:
# El titulo que sirve para CONTEXTUALIZAR no es el mismo que sirve para imprimir.
#
# En un acto europeo las dos primeras lineas son siempre el tipo con su numero y la fecha:
#   REGLAMENTO (UE) 2015/9 DE LA COMISION
#   de 6 de enero de 2015
# Eso es formulario. Son tokens que comparten cientos de actos del corpus, asi que empujan
# el vector hacia el medio de la nube en vez de hacia el tema del documento -y el prefijo
# termina acercando TODOS los pedazos de TODOS los actos al mismo lugar, que es lo contrario
# de lo que queremos-. La linea que dice de QUE trata el acto es la siguiente:
#   que modifica el Reglamento (UE) no 142/2011, por el que se establecen las
#   disposiciones de aplicacion ... en cuanto a determinadas muestras y unidades exentas
# Esa es la que queremos adelante del pedazo.
RE_FECHA = re.compile(r"^de\s+\d{1,2}\s+de\s+\w+", re.IGNORECASE)

def titulo_informativo(texto, tope=140):
    """El tipo de acto MAS su clausula descriptiva, salteando la linea de la fecha.

    Recibe:
      texto  el acto completo
      tope   caracteres maximos del resultado

    Devuelve un string corto para usar de prefijo. Si no encuentra clausula descriptiva
    -un acto raro, o texto mal extraido- devuelve solo la primera linea: un prefijo peor,
    pero nunca vacio. Degradar, no romper.
    """
    lineas = [l for l in (re.sub(r"[ \t]+", " ", l).strip() for l in texto.split("\n")) if l]
    if not lineas:
        return ""
    # La primera linea larga que no sea la fecha, mirando solo el arranque del acto.
    tema = next((l for l in lineas[1:6] if len(l) >= 40 and not RE_FECHA.match(l)), "")
    return (lineas[0] + (" — " + tema if tema else ""))[:tope]

TITULO_DE = {str(r.celex_id): titulo_informativo(r.text)
             for r in corpus.itertuples(index=False)}

def contextualizar(p):
    """El mismo pedazo, con su procedencia adelante. Nada mas: ni resumen ni modelo."""
    cabeza = TITULO_DE.get(p["celex"], p["celex"])
    seccion = p["seccion"] if p["seccion"] not in ("encabezado", "preámbulo") else ""
    prefijo = "[%s%s] " % (recortar(cabeza, 140), " · " + seccion if seccion else "")
    return dict(p, texto=prefijo + p["texto"], crudo=p["texto"])

# Los dos titulos, para ver que se gana. El de arriba es el que el notebook imprime en las
# listas; el de abajo es el que se pega adelante de cada pedazo antes de embeber.
print("TÍTULO CORTO      (el que se imprime) :", recortar(titulo_de(ejemplo.text), 100))
print("TÍTULO INFORMATIVO (el que se indexa) :")
print(parrafo(TITULO_DE[str(ejemplo.celex_id)], lineas=3))
print()

ej = sueltos[0]
ej_ctx = contextualizar(ej)
mostrar("CRUDO", recortar(ej["texto"], 240))
mostrar("CONTEXTUALIZADO", recortar(ej_ctx["texto"], 340))

# El mismo pedazo cae en OTRO lugar del espacio. Lo medimos contra una consulta que
# menciona el tema del documento, que es justo lo que el pedazo crudo no dice.
consulta_tema = recortar(TITULO_DE.get(ej["celex"], ""), 110)
vcrudo, vctx, vq = embeber([ej["texto"], ej_ctx["texto"], consulta_tema])
print("consulta        :", consulta_tema)
print("coseno crudo    : %.3f" % float(vq @ vcrudo))
print("coseno con ctx  : %.3f" % float(vq @ vctx))

TÍTULO CORTO      (el que se imprime) : REGLAMENTO (UE) 2015/9 DE LA COMISIÓN — de 6 de enero de 2015
TÍTULO INFORMATIVO (el que se indexa) :
   REGLAMENTO (UE) 2015/9 DE LA COMISIÓN — que modifica el Reglamento (UE) no
   142/2011, por el que se establecen las disposiciones de aplicació

──────────────────────────────────────────────────────────────────────────
CRUDO
──────────────────────────────────────────────────────────────────────────
dicho Reglamento, por los otros medios establecidos en el capítulo IV del
anexo VI del Reglamento (UE) no 142/2011. De conformidad con lo dispuesto
en el artículo 36, apartado 3, del Reglamento (UE) no 142/2011, esta
posibilidad se limita a…

──────────────────────────────────────────────────────────────────────────
CONTEXTUALIZADO
──────────────────────────────────────────────────────────────────────────
[REGLAMENTO (UE) 2015/9 DE LA COMISIÓN — que modifica el Reglamento (UE)
no 142/2011, por el que se establecen las disposiciones de aplicació]
di

In [42]:
# Re-indexamos TODO con el prefijo. Es la invariante de la Clase 4 otra vez: cambió lo que
# se embebe, así que hay que volver a embeberlo entero. No hay atajo.
t0 = time.time()
PEDAZOS_CTX = [contextualizar(p) for p in PEDAZOS]
MATRIZ_CTX  = embeber([p["texto"] for p in PEDAZOS_CTX])
print("re-indexado en %.0f s" % (time.time() - t0))
vram("despues de re-indexar")
print("el prefijo agregó %.0f tokens de mediana por pedazo" % np.median(
    [contar_tokens(PEDAZOS_CTX[i]["texto"]) - contar_tokens(PEDAZOS[i]["texto"])
     for i in range(0, len(PEDAZOS), max(1, len(PEDAZOS) // 60))]))

Batches:   0%|          | 0/1921 [00:00<?, ?it/s]

re-indexado en 401 s
   [vram] despues de re-indexar      libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB
el prefijo agregó 42 tokens de mediana por pedazo


### Paréntesis de medición · una sola función de recuperación, y una tabla que crece

Acá definimos **la** función de recuperación del notebook. Una sola, con banderas. Cada técnica que agreguemos de ahora en adelante **enciende una bandera y nada más**.

No es prolijidad: es lo que hace honesta la comparación. Si cada técnica tuviera su propia función, dos filas de la tabla podrían diferir en dos cosas sin que nadie lo note, y el «movimos una variable» sería una promesa en vez de un hecho.

Dos de las piezas —el canal léxico y el re-puntuador— todavía no existen: se construyen en las secciones 7 y 11. Python resuelve los nombres recién **cuando se llama** a la función, así que podemos escribirla entera ahora y enchufar las piezas después.

In [43]:
_AVISADO = set()

def _falta(pieza):
    """True si la pieza todavia no se construyo. Avisa una sola vez y sigue."""
    if pieza in globals():
        return False
    if pieza not in _AVISADO:
        _AVISADO.add(pieza)
        print("⚠  '%s' se construye más abajo. Por ahora sigo sin esa pieza." % pieza)
    return True

def recuperar(pregunta, n=K_ENTREGO, contextual=True, lexico=False,
              padre=False, rerank=False, texto_busqueda=None):
    """Devuelve n pasajes. Cada bandera agrega UNA pieza; ninguna cambia a las otras.

    texto_busqueda: si se pasa, el canal DENSO busca con ESE texto en vez de con la
    pregunta. Es lo unico que hace HyDE (seccion 10): no toca el modelo, cambia la llave
    de busqueda del unico canal que la necesita. El lexico sigue con la pregunta."""
    clave = texto_busqueda or pregunta
    base  = PEDAZOS_CTX if contextual else PEDAZOS
    M     = MATRIZ_CTX  if contextual else MATRIZ

    # El ancho de la primera etapa es SIEMPRE el mismo, tenga o no las otras piezas: si
    # variara, dos filas de la tabla diferirian en dos cosas y la comparacion mentiria.
    q = embeber([clave])[0]
    candidatos = [base[int(i)] for i in np.argsort(M @ q)[::-1][:N_CANDIDATOS]]

    if lexico and not _falta("buscar_bm25"):
        # OJO con que texto recibe cada canal. El denso busca con la clave -ahi es donde
        # actua HyDE-; el lexico busca SIEMPRE con la pregunta original. Un parrafo
        # inventado no contiene la cita literal, que es lo unico que este canal sabe
        # encontrar: pasarle la clave seria dejarlo buscando otra cosa, y entonces la
        # fila de HyDE mediria DOS cambios a la vez en vez de uno.
        candidatos = rrf([candidatos, buscar_bm25(pregunta, N_CANDIDATOS)], N_CANDIDATOS)

    # Re-puntuar va ANTES de subir al padre: el cross-encoder tiene ventana de 512 y un
    # padre no le entra. Se puntuan hijos, siempre.
    if rerank and not _falta("repuntuar"):
        candidatos = repuntuar(pregunta, candidatos, N_CANDIDATOS)

    if padre and not _falta("a_padres"):
        candidatos = a_padres(candidatos)

    return candidatos[:n]

COLUMNAS_TABLA = ["temáticas", "puesto", "identificador", "contexto"]

def fila_de(recuperador):
    """Una fila de la tabla: los DOS bloques de consultas, medidos con el mismo codigo.

    Que sea la misma funcion importa. Si cada bloque tuviera la suya, dos filas podrian
    diferir en COMO se midieron y no en QUE se midio, que es el error que esta tabla existe
    para no cometer."""
    tem = medir(recuperador)
    ide = medir(recuperador, evaluacion=EVAL_ID)
    return {"temáticas": tem["¿entró?"], "puesto": tem["puesto"],
            "identificador": ide["¿entró?"], "contexto": tem["contexto"]}

def agregar_fila(nombre, **config):
    fila = fila_de(lambda q, k: recuperar(q, k, **config))
    TABLA_MADRE.append((nombre, fila))
    return fila

def mostrar_tabla(titulo="Una fila por pieza · el documento correcto entre los %d entregados"):
    tabla(titulo % K_ENTREGO if "%d" in titulo else titulo, COLUMNAS_TABLA, TABLA_MADRE)
    if BANNER_EVAL:
        print(BANNER_EVAL)

In [44]:
# La linea de partida y el primer movimiento. Una variable de diferencia: el prefijo.
#
# Las costuras vram() de esta celda y de las que siguen no son decorativas. Cada fila de la
# tabla corre decenas de consultas, y una consulta toca el embedder. Si algo acumula
# memoria por consulta en vez de por corrida, el escalon aparece justo aca.
vram("entrando a la 1a fila")
agregar_fila("denso, pedazo crudo",   contextual=False)
vram("despues de 1 fila")
agregar_fila("+ contextualización",   contextual=True)
vram("despues de 2 filas")
mostrar_tabla()

   [vram] entrando a la 1a fila      libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB
   [vram] despues de 1 fila          libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB
   [vram] despues de 2 filas         libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB

Una fila por pieza · el documento correcto entre los 5 entregados
─────────────────────────────────────────────────────────────────────
                        temáticas   puesto   identificador   contexto
denso, pedazo crudo         26/30      1.4            2/15   1882 tok
+ contextualización         23/30      1.3            2/15   2175 tok



**Cómo leerlo, salga como salga.**

Si la segunda fila mejoró, el mecanismo es el que anticipamos: el pedazo dejó de ser anónimo y ahora su vector carga de qué norma y de qué artículo sale, así que las preguntas que mencionan el tema del documento lo alcanzan.

Si **no** mejoró —o empeoró— también es un resultado, y hay dos explicaciones razonables. Una: el prefijo es el mismo para todos los pedazos de un acto, así que acerca a *todos* sus pedazos a las preguntas sobre ese acto, y eso mejora el acierto pero puede empeorar el puesto. Otra: nuestras preguntas ya mencionan el tema con palabras que estaban en el cuerpo del pedazo, y entonces el prefijo no agrega información que el vector no tuviera.

Lo importante es que el número es **nuestro**, medido sobre este corpus con estas preguntas, y no un porcentaje heredado de un post que midió otra cosa.

▶ **Láminas 38–39** (el pedazo que perdió sus referentes; arreglar lo que se indexa).

## 7 · El canal léxico y la fusión por puesto

Retomamos la falla que dejamos abierta al final de la sección 5: la consulta que es un **identificador exacto**, y que el coseno no encuentra.

El embedding pierde la forma superficial por diseño. Así que le agregamos un canal que hace exactamente lo contrario: **BM25**, un conteo de términos ponderado que matchea la palabra literal y no entiende nada del significado. Es el espejo exacto de lo que el embedding pierde.

Y sí: es el buscador por palabras clave que la Clase 1 enterró por fallar con sinónimos. Vuelve porque **cada canal es ciego a algo distinto**. Por eso se suman en vez de elegir uno.

**Los nombres, que conviene tener a mano.** Combinar los dos canales es **búsqueda híbrida** (*hybrid search*, o *hybrid retrieval*). Y cada canal tiene el suyo: al de embeddings se lo llama **recuperación densa** (*dense retrieval*) y al de BM25, **recuperación rala** (*sparse retrieval*). Denso y ralo por la forma del vector de cada uno: mil números todos distintos de cero contra un vector del tamaño del vocabulario con casi todo en cero.

In [45]:
import bm25s, Stemmer
import logging

# bm25s escribe en DEBUG cuando construye el indice. Util una vez, ruido despues.
logging.getLogger("bm25s").setLevel(logging.WARNING)

vram("entrando al lexico")

# Primero, la trampa. Miramos como tokeniza el identificador que queremos encontrar.
crudo = bm25s.tokenize([CONSULTA_ID], stopwords="es", return_ids=False, show_progress=False)
print("consulta   :", CONSULTA_ID)
print("tokenizada :", crudo[0])
print()
print("La barra partió el identificador. Dejó de existir como término: quedaron")
print("números sueltos que aparecen en cualquier documento del corpus.")

   [vram] entrando al lexico         libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB
consulta   : Reglamento (CEE) no 3730/87
tokenizada : ['reglamento', 'cee', '3730', '87']

La barra partió el identificador. Dejó de existir como término: quedaron
números sueltos que aparecen en cualquier documento del corpus.


**Frenemos acá, porque hay algo que sorprende: BM25 tiene tokenizador, y lo elegís vos.**

BM25 no es un modelo: es una **fórmula que cuenta términos**. Así que alguien tiene que decidir qué cuenta como término, y esa decisión es preprocesamiento clásico de recuperación de información — cuatro pasos, ninguno aprendido:

```
"Reglamento (CEE) no 3730/87"
  ① partir por todo lo que no sea carácter de palabra  → ['Reglamento','CEE','no','3730','87']
  ② minúsculas                                          → ['reglamento','cee','no','3730','87']
  ③ sacar stopwords  (stopwords="es")                   → ['reglamento','cee','3730','87']
  ④ stemming         (Stemmer("spanish"))               → ['reglament','cee','3730','87']
```

**Y no se parece en nada al tokenizador del embedder.** Aquél viene fijado por el modelo y **no tira nada** —ni la puntuación—, porque el modelo necesita toda la señal. Éste **tira a propósito**: stopwords, sufijos, símbolos. Por eso es rapidísimo y no necesita GPU, y por eso también es peligroso: lo que tira, lo tira para siempre.

**Dónde se rompió, exactamente.** La barra de `3730/87` no es un carácter de palabra, así que el paso ① la usó como separador. El identificador dejó de existir como término y quedaron dos números sueltos —`3730` y `87`— que aparecen en cualquier acto del corpus.

**Es la falla del preprocesamiento, y es sutil porque no rompe nada.** El índice se construye, la consulta corre, los resultados vuelven. Sólo que el término que importaba nunca existió.

El arreglo es una línea: **antes de tokenizar, convertimos cada identificador en un solo token**, pegándolo con `_`, que sí cuenta como carácter de palabra. Y se aplica idéntico al corpus y a la consulta — es la misma invariante que en los embeddings, *mismo régimen en las dos puntas*, ahora en el otro canal. Si stemmearas el corpus y no la consulta, `reglamento` no matchearía nunca con `reglament` y no encontrarías nada. Sin ningún error, otra vez.

In [46]:
# Las barras adentro de una cita se vuelven guiones bajos. "_" cuenta como caracter de
# palabra, asi que 89_106_CEE sobrevive entero a la tokenizacion.
RE_ID_NORM = re.compile(r"(\d{1,5})/(\d{1,5})/(CEE|CE|UE|CECA|Euratom)", re.IGNORECASE)
# El (?<![^\W\d_]) exige que la "n" no venga pegada a otra letra: sin eso, "en 1038/2004"
# se convertiria en "en_1038_2004" y estariamos rompiendo texto en vez de protegerlo.
RE_ID_NUM  = re.compile(r"(?<![^\W\d_])n\.?[oº°]?\s*(\d{1,5})/(\d{2,4})", re.IGNORECASE)

def normalizar_citas(t):
    t = RE_ID_NORM.sub(lambda m: "%s_%s_%s" % (m.group(1), m.group(2), m.group(3).upper()), t)
    return RE_ID_NUM.sub(lambda m: "n_%s_%s" % (m.group(1), m.group(2)), t)

stemmer = Stemmer.Stemmer("spanish")

def tokenizar(textos):
    # Mismo tratamiento para el corpus y para la consulta, siempre. Es la regla que hace
    # que un canal lexico funcione: los dos lados tienen que hablar el mismo idioma.
    return bm25s.tokenize([normalizar_citas(t) for t in textos],
                          stopwords="es", stemmer=stemmer, show_progress=False)

print("ahora:", bm25s.tokenize([normalizar_citas(CONSULTA_ID)], stopwords="es",
                               stemmer=stemmer, return_ids=False, show_progress=False)[0])

ahora: ['reglament', 'cee', 'n_3730_87']


In [47]:
t0 = time.time()
bm25_idx = bm25s.BM25()
bm25_idx.index(tokenizar([p["texto"] for p in PEDAZOS_CTX]), show_progress=False)
print("índice léxico listo en %.1f s (sin GPU, sin modelo)" % (time.time() - t0))
vram("con BM25 construido")

def buscar_bm25(consulta, n):
    idx, puntajes = bm25_idx.retrieve(tokenizar([consulta]),
                                      k=min(n, len(PEDAZOS_CTX)), show_progress=False)
    return [dict(PEDAZOS_CTX[int(i)], bm25=float(s))
            for i, s in zip(idx[0], puntajes[0])]

print()
print("EL MISMO IDENTIFICADOR, POR EL CANAL LÉXICO")
print("─" * 74)
for i, r in enumerate(buscar_bm25(CONSULTA_ID, 5), 1):
    marca = "  ← es el correcto" if r["celex"] in GOLD_ID else ""
    print("%d. bm25=%6.2f  %-12s %s%s" % (i, r["bm25"], r["celex"], parrafo(r["texto"], 80, lineas=3), marca))

índice léxico listo en 8.1 s (sin GPU, sin modelo)
   [vram] con BM25 construido        libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB

EL MISMO IDENTIFICADOR, POR EL CANAL LÉXICO
──────────────────────────────────────────────────────────────────────────
1. bm25=  5.00  32012R0121      [REGLAMENTO (UE) No 121/2012 DEL PARLAMENTO EUROPEO Y DEL CONSEJO — que modifica
   los Reglamentos (CE) no 1290/2005 y (CE) no 1234/2007 del Co] REGLAMENTO (UE) No
   121/2012 DEL PARLAMENTO EUROPEO Y DEL CONSEJO de 15 de febrero de 2012 que …  ← es el correcto
2. bm25=  3.32  32010R1231      [REGLAMENTO (UE) No 1231/2010 DEL PARLAMENTO EUROPEO Y DEL CONSEJO — por el que
   se amplía la aplicación del Reglamento (CE) no 883/2004 y el] les obligaciones
   comparables a los de los ciudadanos de la Unión. (3) El Reglamento (CE) no …
3. bm25=  3.32  32014R0822      [REGLAMENTO DE EJECUCIÓN (UE) No 822/2014 DE LA COMISIÓN — que establece una
   excepción al Reglamento (CEE) no 2454/93 por lo que r

### 7.1 · El problema antes de la solución

Tenemos dos listas. Lo natural sería sumar los puntajes y ordenar. **No se puede.**

El coseno está **acotado**: va de −1 a 1, y en la práctica los resultados relevantes se apiñan entre 0,3 y 0,8. El puntaje de BM25 **no tiene cota ni escala fija**: depende del largo del documento, de cuántos documentos hay y de qué tan raro es cada término. Puede valer 3 o 40 según el corpus y el día.

Sumar dos números que no están en la misma unidad da un número que no significa nada.

In [48]:
q = embeber([CONSULTA_ID])[0]
top_d = [dict(PEDAZOS_CTX[int(i)], cos=float(MATRIZ_CTX[int(i)] @ q))
         for i in np.argsort(MATRIZ_CTX @ q)[::-1][:5]]
top_l = buscar_bm25(CONSULTA_ID, 5)

tabla("Las dos listas, con sus puntajes", ["puesto", "puntaje", "acto"],
      [("denso %d" % i, {"puesto": i, "puntaje": "%.3f" % r["cos"], "acto": r["celex"]})
       for i, r in enumerate(top_d, 1)] +
      [("léxico %d" % i, {"puesto": i, "puntaje": "%.2f" % r["bm25"], "acto": r["celex"]})
       for i, r in enumerate(top_l, 1)])
print("  Una columna va de −1 a 1. La otra, de 0 a lo que dé. No son la misma unidad.")


Las dos listas, con sus puntajes
────────────────────────────────────────────
               puesto   puntaje         acto
denso 1             1     0.583   32014R0830
denso 2             2     0.579   32012R1230
denso 3             3     0.573   32011R1130
denso 4             4     0.571   32013R0373
denso 5             5     0.561   32010R0177
léxico 1            1      5.00   32012R0121
léxico 2            2      3.32   32010R1231
léxico 3            3      3.32   32014R0822
léxico 4            4      3.30   32014R0822
léxico 5            5      3.22   32013D0530

  Una columna va de −1 a 1. La otra, de 0 a lo que dé. No son la misma unidad.


### 7.2 · Tirá el puntaje, usá el puesto

La salida es elegante: **se descarta el puntaje y se usa el puesto**. Cada documento suma, por cada lista en la que aparece, `1 / (60 + puesto)`. Se suman esas contribuciones y se reordena.

Gana el **consenso**: quedar tercero en las dos listas le gana a quedar primero en una sola y no aparecer en la otra. Y no hace falta normalizar nada, porque un puesto es un puesto en cualquier escala.

Ese 60 es una constante empírica del paper original, no una verdad matemática. Amortigua la diferencia entre los primeros puestos, para que el primero no aplaste al segundo.

Son ocho líneas. Las escribimos a mano a propósito: si lo hiciera el store, verían una llamada a una API en lugar de la idea.

In [49]:
def rrf(listas, n, k=60):
    """Reciprocal Rank Fusion: cada lista vota con 1/(k+puesto). Se suma y se reordena.

    Recibe:
      listas  varias listas de pedazos, cada una YA ordenada por su propio criterio
      n       cuantos devolver
      k       la constante del paper original. Con k=60, entre el puesto 1 y el 5 se
              pierde apenas un 6% -1/61 contra 1/65-, mientras que aparecer en una
              SEGUNDA lista duplica el puntaje. Ahi esta todo: la constante es lo que
              hace que el consenso pese mas que el primer puesto.

    Devuelve los n mejores, cada uno con un campo "rrf" agregado.
    """
    puntaje, por_id = {}, {}
    for lista in listas:
        for puesto, p in enumerate(lista, 1):
            puntaje[p["id"]] = puntaje.get(p["id"], 0.0) + 1.0 / (k + puesto)
            por_id[p["id"]] = p
    orden = sorted(puntaje, key=lambda i: -puntaje[i])[:n]
    return [dict(por_id[i], rrf=puntaje[i]) for i in orden]


def mostrar_cuenta(titulo, lista_d, lista_l, n=4):
    """La cuenta a la vista: en que puesto salio cada acto en cada canal, y cuanto suma.

    Devuelve (fusion, cuantos aparecieron en las DOS listas).
    """
    fus, filas, con_dos = rrf([lista_d, lista_l], n), [], 0
    for r in fus:
        p_d = next((i for i, x in enumerate(lista_d, 1) if x["id"] == r["id"]), None)
        p_l = next((i for i, x in enumerate(lista_l, 1) if x["id"] == r["id"]), None)
        con_dos += 1 if (p_d and p_l) else 0
        filas.append((r["celex"], {
            "puesto denso": p_d or "—",
            "puesto léxico": p_l or "—",
            "la cuenta": " + ".join("1/(60+%d)" % p for p in (p_d, p_l) if p),
            "puntaje": "%.5f" % r["rrf"],
            "votos": 2 if (p_d and p_l) else 1}))
    tabla(titulo, ["puesto denso", "puesto léxico", "la cuenta", "puntaje", "votos"], filas)
    return fus, con_dos


# ── 1) La consulta del identificador: el caso extremo, donde los canales no se pisan ──
fusion, _dos = mostrar_cuenta("La cuenta · consulta de IDENTIFICADOR", top_d, top_l, 4)
if _dos == 0:
    print("  Ninguna fila suma dos votos: en esta consulta los dos canales no comparten NI UN")
    print("  resultado. El denso es ciego a la cita y trae otra cosa; el léxico trae los que la")
    print("  contienen. Acá RRF no fusiona: INTERCALA. Y alcanza, porque el correcto entra.")
    print("  Ojo con el empate de arriba: dos primeros puestos valen exactamente lo mismo, y")
    print("  el orden entre ellos lo decide el desempate, no el mérito.")

# ── 2) Una pregunta temática: acá los dos canales sí se pisan, y aparece el consenso ──
preg_tema = EVAL_SET[0][0]
q_tema = embeber([preg_tema])[0]
top_d2 = [dict(PEDAZOS_CTX[int(i)], cos=float(MATRIZ_CTX[int(i)] @ q_tema))
          for i in np.argsort(MATRIZ_CTX @ q_tema)[::-1][:5]]
top_l2 = buscar_bm25(preg_tema, 5)

print()
print("Y ahora la otra mitad del argumento, sobre una pregunta TEMÁTICA:")
print(parrafo(preg_tema, lineas=2))
_, dos_tema = mostrar_cuenta("La cuenta · pregunta TEMÁTICA", top_d2, top_l2, 4)
if dos_tema:
    print("  Ahí está el consenso: %d acto(s) aparecen en las DOS listas y suman dos veces." % dos_tema)
    print("  Quedar en el medio de las dos le gana a ser primero en una sola y faltar en la otra,")
    print("  y por eso no hace falta normalizar nada: un puesto es un puesto en cualquier escala.")
else:
    print("  Tampoco acá se pisaron los canales. Pasa cuando las dos listas son cortas (5) sobre")
    print("  un corpus de 30.000 pedazos: probá con listas más largas y van a empezar a coincidir.")

# ── El resultado que importa: el identificador, con los dos canales fusionados ──
print()
print("EL IDENTIFICADOR, CON LOS DOS CANALES FUSIONADOS")
print("─" * 74)
for i, r in enumerate(fusion, 1):
    marca = "  ← es el correcto" if r["celex"] in GOLD_ID else ""
    print("%d. %-12s %s%s" % (i, r["celex"], recortar(r["texto"], 88), marca))


La cuenta · consulta de IDENTIFICADOR
─────────────────────────────────────────────────────────────────────────
               puesto denso   puesto léxico   la cuenta   puntaje   votos
32014R0830                1               —    1/(60+1)   0.01639       1
32012R0121                —               1    1/(60+1)   0.01639       1
32012R1230                2               —    1/(60+2)   0.01613       1
32010R1231                —               2    1/(60+2)   0.01613       1

  Ninguna fila suma dos votos: en esta consulta los dos canales no comparten NI UN
  resultado. El denso es ciego a la cita y trae otra cosa; el léxico trae los que la
  contienen. Acá RRF no fusiona: INTERCALA. Y alcanza, porque el correcto entra.
  Ojo con el empate de arriba: dos primeros puestos valen exactamente lo mismo, y
  el orden entre ellos lo decide el desempate, no el mérito.

Y ahora la otra mitad del argumento, sobre una pregunta TEMÁTICA:
   ¿Con qué criterio se reparte entre los países el diner

In [50]:
agregar_fila("+ canal léxico (RRF)", contextual=True, lexico=True)
vram("despues de la fila lexica")
mostrar_tabla()

   [vram] despues de la fila lexica  libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB

Una fila por pieza · el documento correcto entre los 5 entregados
──────────────────────────────────────────────────────────────────────
                         temáticas   puesto   identificador   contexto
denso, pedazo crudo          26/30      1.4            2/15   1882 tok
+ contextualización          23/30      1.3            2/15   2175 tok
+ canal léxico (RRF)         26/30      1.5            6/15   2151 tok



**Lo que acaba de pasar.** El canal denso sigue sin encontrar el identificador —no cambió nada de su lado—, el léxico lo pone primero, y la lista fusionada lo trae. La falla que dejamos abierta está cerrada, y no la cerró un embedder mejor: la cerró **dejar de pedirle a un solo canal que haga todo**.

> **Para discutir:** ¿por qué RRF fusiona **puestos** y no un promedio de puntajes normalizados? Normalizar es posible —min-max sobre cada lista, por ejemplo— pero la normalización depende de qué había en esa lista particular, así que el puntaje final de un documento cambiaría según con quién le tocó competir. El puesto no tiene ese problema.

▶ **Láminas 33–37** (el canal léxico; dos ciegos distintos; los puntajes no se suman; fusionar por rango; qué cuenta BM25).

## 8 · Se indexa el hijo, se entrega el padre  ·  *parent-document retrieval*

La otra mitad de la falla de la sección 4. Contextualizar arregló lo que se **indexa**; ahora falta lo que se **entrega**.

El problema es que las dos cosas piden tamaños opuestos. Para **encontrar** conviene un pedazo chico: un vector por idea, preciso. Para **responder** conviene un pedazo grande: el artículo entero, con sus referentes y su desarrollo.

La salida es dejar de suponer que tienen que ser el mismo objeto. **Se indexa el hijo y se entrega el padre.** En la bibliografía vas a encontrar la misma técnica con dos nombres, **parent-document retrieval** y **small-to-big**. Un diccionario de hijo a padre, y un conjunto para no entregar el mismo padre dos veces si ganaron dos de sus hijos. Diez líneas.

In [51]:
# El padre de un pedazo es el articulo del que salio: ya lo teniamos, porque cortamos por
# estructura. Con corte ciego esta pieza no existiria -no habria a que subir-.
PADRE_DE, TEXTO_PADRE = {}, {}
for r in corpus.itertuples(index=False):
    for etiqueta, cuerpo in partir_en_articulos(r.text):
        TEXTO_PADRE["%s|%s" % (r.celex_id, etiqueta)] = cuerpo
for p in PEDAZOS_CTX:
    PADRE_DE[p["id"]] = "%s|%s" % (p["celex"], p["seccion"])

# Un padre solo sirve si es una unidad ENTREGABLE. Y no todas las piezas estructurales lo
# son: el "encabezado" de un acto europeo -todo lo anterior al Articulo 1, o sea los
# considerandos- suele ser el 80% del documento. Entregarlo entero desborda la ventana del
# generador, que entra a recortar el prompt y tarda minutos en responder.
TAM_PADRE_MAX = 2500

def a_padres(hijos):
    """Sube cada hijo a su padre, sin repetir, SI el padre entra.

    El set es la mitad de la idea: si dos hijos del mismo articulo ganaron, se entrega el
    articulo una sola vez -y eso LIBERA un lugar para otro documento-. El tope es la otra
    mitad: un padre que no entra en la ventana no se promueve, se queda el hijo."""
    vistos, salida = set(), []
    for h in hijos:
        pid = PADRE_DE.get(h["id"])
        texto = TEXTO_PADRE.get(pid) if pid else None
        if texto is None or contar_tokens(texto, "generador") > TAM_PADRE_MAX:
            salida.append(h)
            continue
        if pid in vistos:
            continue
        vistos.add(pid)
        salida.append(dict(h, id=pid, texto=texto))
    return salida

# Cuantos padres son entregables y cuantos no: es el numero que explica la seccion.
entran = [t for t in TEXTO_PADRE.values() if contar_tokens(t, "generador") <= TAM_PADRE_MAX]
print("pedazos (hijos)                 :", len(PEDAZOS_CTX))
print("padres (encabezados + artículos):", len(TEXTO_PADRE),
      "· de los cuales entran en %d tokens: %d (%.0f%%)"
      % (TAM_PADRE_MAX, len(entran), 100.0 * len(entran) / max(1, len(TEXTO_PADRE))))
print("tamaño mediano del hijo  : %4d tokens" % np.median(
    [contar_tokens(p["texto"]) for p in PEDAZOS_CTX[:200]]))
print("tamaño mediano del padre : %4d tokens" % np.median(
    [contar_tokens(t) for t in list(TEXTO_PADRE.values())[:200]]))
vram("con el mapa padre/hijo")

pedazos (hijos)                 : 30730
padres (encabezados + artículos): 15933 · de los cuales entran en 2500 tokens: 15313 (96%)
tamaño mediano del hijo  :  387 tokens
tamaño mediano del padre :  172 tokens
   [vram] con el mapa padre/hijo     libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB


In [52]:
agregar_fila("+ padre → hijo", contextual=True, lexico=True, padre=True)
vram("despues de la fila padre")
mostrar_tabla()

   [vram] despues de la fila padre   libre  12096 · PyTorch   2.8 GB · este proceso   2814 MiB

Una fila por pieza · el documento correcto entre los 5 entregados
──────────────────────────────────────────────────────────────────────
                         temáticas   puesto   identificador   contexto
denso, pedazo crudo          26/30      1.4            2/15   1882 tok
+ contextualización          23/30      1.3            2/15   2175 tok
+ canal léxico (RRF)         26/30      1.5            6/15   2151 tok
+ padre → hijo               26/30      1.5            6/15   3746 tok



> **Ojo con un detalle que no es menor.** No toda pieza estructural sirve de padre. En un acto europeo, todo lo anterior al `Artículo 1` —los considerandos— es una sola pieza que suele ser el **80 % del documento**. Entregar eso como «el padre» no mejora nada: desborda la ventana del generador, que entra a recortar el prompt y tarda minutos en contestar. Por eso el padre se promueve **sólo si entra**; si no, se entrega el hijo. La alternativa sería partir el preámbulo en sus considerandos numerados, que es más prolijo y bastante más frágil de parsear.

**Cómo leerlo, y acá hay que tener cuidado con la vara.**

La columna «contexto» es la que muestra lo que esta técnica hace de verdad: **al generador le llega varias veces más texto, y texto que se puede leer** — un artículo completo en vez de un fragmento cortado al medio. Eso no lo mide el acierto.

La columna «¿entró?» también se puede mover, por una razón indirecta: entregamos **la misma cantidad de piezas**, y al deduplicar padres se liberan lugares. Si tres hijos del mismo artículo ocupaban el podio, ahora ocupan uno solo y entran dos documentos más. Si no se mueve, no es que la técnica no sirva: es que esta vara mide la recuperación y el aporte de ésta está en la entrega.

> **Decisión didáctica, y se avisa:** existe una variante más chica, el *sentence-window*, y conviene conocerla aunque no la corramos. Indexa **una oración sola** —el vector más preciso posible, una idea sin promediar nada— y al recuperar entrega **esa oración con sus vecinas**, por ejemplo tres antes y tres después. Es la misma idea de acá con otra granularidad: se indexa una cosa y se entrega otra.
>
> La diferencia está en **quién define al padre**. Acá lo define el documento: el padre es el artículo. En el *sentence-window* lo define una ventana de tamaño fijo alrededor del hijo. Eso lo vuelve la opción natural cuando el material **no tiene estructura** —no hay artículo al que subir—, y es al mismo tiempo su punto débil: la ventana no sabe dónde termina una sección, así que puede arrastrar vecinas que no vienen al caso.
>
> No lo implementamos porque no agrega concepto: si entendiste padre-hijo, ya lo entendiste. En LlamaIndex es el `SentenceWindowNodeParser`, y aparece en casi cualquier tutorial.

▶ **Láminas 38, 40** (la falla y sus dos mitades; arreglar lo que se entrega).

## 9 · El prompt de RAG: delimitar, abstenerse, citar

Todo lo anterior mejoró **qué le llega** al modelo. Esta sección es la única del día que toca **qué hace con lo que le llegó**.

Arrancamos por el prompt que traíamos de la Clase 4:

```
Usá SÓLO este contexto para responder.

Contexto:
{contexto}

Pregunta: {pregunta}
Respondé en una sola oración.
```

**Y para la comparación de abajo lo vamos a dejar todavía más pelado**, sin siquiera esa primera línea. No es un hombre de paja: es que un modelo moderno y alineado, con sólo ese `Usá SÓLO este contexto`, ya suele abstenerse — y entonces el punto ciego no se ve. Quitándola queda a la vista el **comportamiento por defecto** de un generador al que nadie le dijo nada, que es el que conviene conocer antes de decidir qué instrucción agregar.

El punto ciego es grande: **el prompt no le dice qué hacer si el contexto no alcanza**. Un modelo entrenado para continuar texto, puesto frente a una pregunta y un contexto que no la responde, no se queda callado: completa. Con seguridad, y con la forma de una respuesta correcta.

In [53]:
from pydantic import BaseModel, Field

# Antes de generar, le devolvemos a la placa lo que PyTorch tenia retenido. A esta altura
# embebimos el corpus dos veces, y esa memoria cacheada es justo la que Ollama necesita para
# entrar entero en GPU en vez de repartirse con el procesador.
liberar_gpu(MODEL)

# Primera llamada al generador desde la seccion 1: dos pasadas de embedding y unos cuarenta
# minutos atras. Si el servidor se cayo en el medio, esta es la celda donde nos enterariamos,
# y el error diria "connection refused" sin ninguna pista de cuando empezo. Lo revisamos.
if asegurar_ollama():
    ask("ok")          # fuerza la recarga ahora y no en medio de la primera medicion
    donde_corre()      # y confirma que entro en la placa: mira la columna PROCESSOR

class RespuestaRAG(BaseModel):
    # OJO con el orden de los campos: Ollama serializa las claves en ORDEN ALFABETICO,
    # no en el orden en que se declaran. Como el modelo escribe de izquierda a derecha, el
    # campo de razonamiento tiene que caer primero alfabeticamente, o razonaria DESPUES de
    # haber contestado, que es no razonar. De ahi los nombres: analisis < basta < cita < respuesta.
    analisis: str = Field(description="Qué dice el contexto sobre la pregunta, en una o dos oraciones.")
    basta_el_contexto: bool = Field(description="True sólo si el contexto alcanza para responder.")
    cita: list[str] = Field(description="Identificadores entre corchetes de los pasajes usados.")
    respuesta: str = Field(description="La respuesta, o 'No lo sé' si el contexto no alcanza.")

TOPE_CONTEXTO = 5000      # tokens del generador, sobre una ventana de 8192

def armar_contexto(pasajes, tope=None):
    """Los pasajes, delimitados y marcados con su identificador, hasta agotar el presupuesto.

    El identificador no es decorativo: es lo que hace posible pedir la cita y verificarla
    despues a mano. Y el tope no es prolijidad: la ventana ES un presupuesto, y si el prompt
    la pasa, el generador no falla -recorta y tarda muchisimo-, que es peor que fallar."""
    tope = TOPE_CONTEXTO if tope is None else tope
    partes, usados = [], 0
    for p in pasajes:
        cuesta = contar_tokens(p["texto"], "generador")
        if usados + cuesta > tope:
            break
        usados += cuesta
        partes.append("[%s · %s]\n%s" % (p["celex"], p["seccion"][:40], p["texto"]))
    return "\n\n".join(partes)

# El prompt PELADO: ni delimitadores, ni clausula de abstencion, ni cita. Ni siquiera el
# "usa solo este contexto" que traiamos de la Clase 4. Muestra el comportamiento por
# defecto del generador, y esta como constante para poder imprimirlo al lado del otro.
PROMPT_SIN_REGLAS = """Contexto:
{contexto}

Pregunta: {pregunta}
Respondé en una sola oración."""

PROMPT_RAG = """Respondé la pregunta usando ÚNICAMENTE el contexto de abajo.

<contexto>
{contexto}
</contexto>

Reglas:
- No uses conocimiento propio. Si el contexto no alcanza, decilo.
- Si no alcanza: basta_el_contexto en false, respuesta "No lo sé", cita vacía.
- En cita van los identificadores entre corchetes de los pasajes que realmente usaste.

Pregunta: {pregunta}"""

def responder_rag(pregunta, pasajes=None, con_reglas=True):
    """Genera sobre lo recuperado. con_reglas=False usa el prompt pelado, sin ninguna regla."""
    pasajes = pasajes if pasajes is not None else recuperar(
        pregunta, K_ENTREGO, contextual=True, lexico=True, padre=True)
    contexto = armar_contexto(pasajes)
    if not con_reglas:
        crudo = ask(PROMPT_SIN_REGLAS.format(contexto=contexto, pregunta=pregunta))
        return {"respuesta": crudo, "basta_el_contexto": None, "cita": [], "analisis": ""}
    salida = ask(PROMPT_RAG.format(contexto=contexto, pregunta=pregunta),
                 fmt=RespuestaRAG.model_json_schema(), num_predict=700)
    try:
        return RespuestaRAG.model_validate_json(salida).model_dump()
    except Exception as e:
        # El parseo puede fallar: Qwen3 a veces filtra razonamiento adentro del JSON.
        # Devolvemos algo estructurado igual, para que las celdas de abajo no se caigan.
        return {"respuesta": salida[:300], "basta_el_contexto": None,
                "cita": [], "analisis": "(no parseó: %s)" % type(e).__name__}

VRAM · PyTorch retenía 2.8 GB, ahora 2.3 GB · libre en la placa: 12608 MiB
NAME        ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3:8b    500a1f067a9f    6.3 GB    100% GPU     8192       4 minutes from now


### 9.1 · Confabula, o dice que no sabe

La prueba tiene que ser limpia: **misma recuperación, mismo modelo, misma temperatura**. La única diferencia es la instrucción. Si cambiáramos algo más, no sabríamos a qué atribuir la diferencia.

Y la pregunta tiene que ser una que el corpus **no responde**. Si el corpus la responde, los dos prompts contestan bien y no se ve nada.

In [54]:
# PREGUNTA_FUERA se fijo en la seccion 2, junto al eval set: es parte de los datos de la
# clase, no de esta celda. Es plausible para este corpus y este corpus no la responde.
print("pregunta:", PREGUNTA_FUERA)
print()

# Los dos prompts, uno al lado del otro y con el contexto reemplazado por un marcador: lo
# que se compara es la INSTRUCCION, y los 5.000 tokens de pasajes solo taparian la pantalla.
_MARCADOR = "«acá van los %d pasajes recuperados, cada uno con su identificador»" % K_ENTREGO



if OLLAMA_OK:
    pasajes = recuperar(PREGUNTA_FUERA, K_ENTREGO, contextual=True, lexico=True, padre=True)
    print("recuperado (idéntico para los dos brazos):")
    for p in pasajes:
        print("   ", p["celex"], "·", parrafo(p["texto"], 70, lineas=3))
    print()

    sin_reglas = responder_rag(PREGUNTA_FUERA, pasajes, con_reglas=False)
    nuevo = responder_rag(PREGUNTA_FUERA, pasajes, con_reglas=True)

    mostrar("EL PROMPT SIN RESTRICCIONES", PROMPT_SIN_REGLAS.format(contexto=_MARCADOR, pregunta=PREGUNTA_FUERA))
    mostrar("Respuesta SIN RESTRICCIONES", sin_reglas["respuesta"])
    mostrar("EL PROMPT DE RAG", PROMPT_RAG.format(contexto=_MARCADOR, pregunta=PREGUNTA_FUERA))
    mostrar("Respuesta (delimitar · abstenerse · citar)",
            "basta_el_contexto : %s\nanalisis          : %s\ncita              : %s\nrespuesta         : %s"
            % (nuevo["basta_el_contexto"], parrafo(nuevo["analisis"], 200, lineas=3),
               nuevo["cita"], nuevo["respuesta"]))
    print("Mismo retrieval, mismo modelo, misma temperatura. Distinta instrucción.")

pregunta: ¿Cuánto se podía pescar de bacalao en el Mar Báltico durante 2020, y qué se cambió sobre el esfuerzo pesquero?

recuperado (idéntico para los dos brazos):
    32014R1221 ·    [REGLAMENTO (UE) No 1221/2014 DEL CONSEJO — por el que se establecen,
   para 2015, las posibilidades de pesca para determinadas poblaciones y
   g] deben fijarse con arreglo a las normas establecidas en dichos …
    32011R1256 ·    REGLAMENTO (UE) No 1256/2011 DEL CONSEJO
   de 30 de noviembre de 2011
   por el que se establecen, para 2012, las posibilidades de pesca …
    32013R1180 ·    REGLAMENTO (UE) No 1180/2013 DEL CONSEJO
   de 19 de noviembre de 2013
   por el que se establecen, para 2014, las posibilidades de pesca …
    32014R1221 ·    [REGLAMENTO (UE) No 1221/2014 DEL CONSEJO — por el que se establecen,
   para 2015, las posibilidades de pesca para determinadas poblaciones y
   g] lao del mar Báltico y sin ocasionar un aumento de la mortalidad por …
    32012R1088 ·    REGLAMENTO (UE) No 1088

In [55]:
# Y sobre una pregunta que el corpus SI responde, para ver que la clausula de abstencion no
# lo volvio mudo: un sistema que se abstiene siempre tiene faithfulness perfecto y sirve cero.
if OLLAMA_OK and EVAL_SET:
    preg_dentro, gold_dentro = EVAL_SET[0]
    r = responder_rag(preg_dentro)
    mostrar("PREGUNTA QUE EL CORPUS SÍ RESPONDE",
            "pregunta          : %s\nbasta_el_contexto : %s\ncita              : %s\nrespuesta         : %s"
            % (parrafo(preg_dentro, 120), r["basta_el_contexto"], r["cita"],
               parrafo(r["respuesta"], 300)))

──────────────────────────────────────────────────────────────────────────
PREGUNTA QUE EL CORPUS SÍ RESPONDE
──────────────────────────────────────────────────────────────────────────
pregunta : ¿Con qué criterio se reparte entre los países el dinero europeo
que cubre los gastos de vigilancia de la actividad
 pesquera durante 2010?
basta_el_contexto : True
cita : ['[32010D0352 · Artículo 10]']
respuesta : El reparto se basa en la utilización de los buques y aeronaves
para la inspección y vigilancia, expresada como porcentaje de su actividad
anual total, declarada por los Estados miembros.



**Lo que hay que ver.** El brazo sin restricciones produce un número, una fecha o un artículo que suenan bien y no están en ningún pasaje. No es un error del modelo: es lo que un modelo entrenado para continuar texto hace cuando no sabe. El segundo tiene una salida donde **«no lo sé» es una respuesta válida y declarada**, y por eso puede darla.

Y la cita no es decorativa. Es lo que convierte la respuesta en **verificable**: con el identificador del pasaje, cualquiera —incluido un script— puede ir a buscarlo y chequear si dice lo que la respuesta afirma. Eso es lo que en la Clase 6, sin fuente, no se podía hacer.

> **Ojo con un detalle:** pedirlo en el prompt es una **súplica**, no una garantía. El modelo puede citar un pasaje que no usó, o abstenerse cuando el dato estaba. Lo que sí garantiza la salida estructurada es la **forma**: que siempre haya un campo de cita y un booleano de abstención, para poder contarlos.

▶ **Láminas 22–27** (el prompt que traíamos; delimitar, abstenerse, citar; confabula o dice que no sabe; las dos superficies).

## 10 · La vara, y la primera técnica medida con ella

Veníamos midiendo desde la sección 4 sin decir con qué. Ahora le ponemos nombre, porque de acá al final todo cuelga de esto.

Primero, **dos palabras** que vamos a usar todo el tiempo, sobre un objeto concreto: una consulta y los diez pedazos que el sistema devolvió.

In [56]:
preg_demo, gold_demo = EVAL_SET[0]
diez = recuperar(preg_demo, 10, contextual=True, lexico=True)
relevantes = [i for i, p in enumerate(diez, 1) if p["celex"] == gold_demo]

print("consulta:", parrafo(preg_demo, 100))
print("─" * 74)
for i, p in enumerate(diez, 1):
    print("%2d. %-12s %s%s" % (i, p["celex"], parrafo(p["texto"], 72, lineas = 3),
                               "   ← relevante" if i in relevantes else ""))
print("─" * 74)
print("RECALL    : de los relevantes que existen, cuántos entraron en la lista.")
print("            Si no entró, ninguna técnica posterior lo salva: no está.")
print("PRECISION : de los que entraron, cuántos sirven. Acá %d de 10." % len(relevantes))
print()
print("No es la precision de la matriz de confusión que vieron en clasificación:")
print("acá hay una LISTA, hay ORDEN, y hay un CORTE en k.")

consulta:    ¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos de vigilancia
   de la actividad pesquera durante 2010?
──────────────────────────────────────────────────────────────────────────
 1. 32010D0711      [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera
   complementaria de la Unión en 2010 en los gastos contraídos por los
   Estados] DECISIÓN DE LA COMISIÓN de 23 de noviembre de 2010 relativa a …
 2. 32010D0352      [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera de la
   Unión en los programas de control, inspección y vigilancia de la acti]
   DECISIÓN DE LA COMISIÓN de 22 de junio de 2010 relativa a la …   ← relevante
 3. 32010D0352      [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera de la
   Unión en los programas de control, inspección y vigilancia de la acti ·
   Artículo 10] Artículo 10 …   ← relevante
 4. 32010D0352      [DECISIÓN DE LA COMISIÓN — relativa a la participación financ

### 10.1 · El eval set, y por qué no lo escribió un modelo

La vara son **treinta filas** escritas a mano: una pregunta y el documento que la responde. Se etiqueta el **documento**, no el pedazo, porque en la sección 8 re-cortamos el corpus y un eval set atado a pedazos habría quedado inválido justo ahí.

Hay una trampa que conviene chequear y casi nadie chequea: si las preguntas **copian el vocabulario** del documento, el canal léxico las acierta por coincidencia de cadenas y toda la comparación queda inflada. Medirlo es barato.

In [57]:
print("EL EVAL SET (primeras cinco filas de %d)" % len(EVAL_SET))
print("─" * 74)
for pregunta, celex in EVAL_SET[:5]:
    print("%-12s  %s" % (celex, parrafo(pregunta, 74, lineas=2)))
print("─" * 74)
print(BANNER_EVAL or "  Escrito a mano. Sin modelos, sin juez, sin etiquetas automáticas.")
print()
print("Y EL SEGUNDO BLOQUE: %d CONSULTAS DE IDENTIFICADOR" % len(EVAL_ID))
print("─" * 74)
for consulta, celex in EVAL_ID[:5]:
    print("%-12s  %s" % (celex, consulta))
print("─" * 74)
print("  Acá la pregunta ES la cita. No hay nada que parafrasear, así que no se escriben")
print("  a mano: se derivan del corpus.")

EL EVAL SET (primeras cinco filas de 30)
──────────────────────────────────────────────────────────────────────────
32010D0352       ¿Con qué criterio se reparte entre los países el dinero europeo que cubre
   los gastos de vigilancia de la actividad pesquera durante 2010?
32014R1321       ¿Qué reglas comunes rigen para que un avión matriculado en un país de la
   Unión siga estando en condiciones de volar?
32010R0724       ¿Cuándo puede cerrarse de un día para otro una zona de pesca en el Mar del
   Norte, y a qué especies alcanza esa medida?
32012R1257       ¿Qué instrumento permite que una patente tenga efecto uniforme en varios
   países sin tener que registrarla en cada uno?
32014R0686       ¿Qué condiciones hay que cumplir para afirmar en una etiqueta que un
   alimento ayuda a bajar el colesterol?
──────────────────────────────────────────────────────────────────────────
  Escrito a mano. Sin modelos, sin juez, sin etiquetas automáticas.

Y EL SEGUNDO BLOQUE: 15 CONSULTAS DE IDE

In [58]:
# Termometro de contaminacion lexica. Los conjuntos de terminos de cada documento se
# calculan UNA vez: hacerlo adentro del loop serian minutos de Python puro.
#
# Nota de vocabulario, porque la palabra se presta a confusion: aca NO hablamos del solape
# de la seccion 4, que era texto repetido a proposito entre dos pedazos vecinos. Aca
# medimos cuantas PALABRAS EN COMUN tiene una pregunta con su propio documento, que es
# otra cosa: un indicio de que la escribimos copiando en vez de parafraseando.
RE_PAL = re.compile(r"(?u)\w{4,}")
TERMINOS_DOC = {c: set(m.lower() for m in RE_PAL.findall(t)) for c, t in TEXTO_DE.items()}

filas_cont = []
for pregunta, celex in EVAL_SET:
    terminos = set(m.lower() for m in RE_PAL.findall(pregunta))
    doc = TERMINOS_DOC.get(celex, set())
    en_comun = len(terminos & doc) / max(1, len(terminos))
    # Cinco palabras seguidas identicas al documento es la senal mas fuerte de un corta y
    # pega. Guardamos CUALES son, que es lo unico que despues se puede accionar.
    palabras = pregunta.lower().split()
    texto_doc = TEXTO_DE.get(celex, "").lower()
    calco = next((" ".join(palabras[i:i + 5]) for i in range(max(0, len(palabras) - 4))
                  if " ".join(palabras[i:i + 5]) in texto_doc), None)
    filas_cont.append((en_comun, calco))

porcentajes = [p for p, _ in filas_cont]
calcos = sum(1 for _, k in filas_cont if k)
# El veredicto se calcula, no se enuncia. Un termometro que imprime su criterio y despues
# un valor que no lo cumple -sin decirlo- es peor que no tener termometro.
pct = 100 * np.mean(porcentajes)
tabla("¿Cuánto se parecen las preguntas a sus documentos?", ["valor", "qué significa"],
      [("palabras en común", {"valor": "%.0f %%" % pct,
                              "qué significa": ("sano: debajo del 50 %" if pct < 50
                                                else "alto: le regala terreno al léxico")}),
       ("frases calcadas",   {"valor": "%d / %d" % (calcos, len(EVAL_SET)),
                              "qué significa": "cada una infla el canal léxico"})])
if pct > 60 or calcos > len(EVAL_SET) // 4:
    print("  ⚠ las preguntas se parecen demasiado al texto: el canal léxico va a ganar por copia.")

# Y CUALES son, que es lo unico accionable del termometro: se reescriben y se vuelve a
# medir. Primero las que tienen una frase calcada, despues las de mas palabras en comun.
peores = sorted(range(len(EVAL_SET)),
                key=lambda i: (filas_cont[i][1] is None, -filas_cont[i][0]))
print("LAS CINCO PREGUNTAS MÁS PEGADAS AL TEXTO DE SU DOCUMENTO")
print("─" * 74)
for i in peores[:5]:
    pregunta, celex = EVAL_SET[i]
    en_comun, calco = filas_cont[i]
    print("%3.0f %%  %-12s %s" % (100 * en_comun, celex, recortar(pregunta, 58)))
    if calco:
        print("       calca del acto: «%s…»" % calco)
print("─" * 74)
print("Reescribir éstas con otras palabras es la forma barata de blindar la comparación")
print("del canal léxico que viene en la sección 7. Y después se vuelve a medir.")


¿Cuánto se parecen las preguntas a sus documentos?
────────────────────────────────────────────────────────────────
                       valor                       qué significa
palabras en común       54 %   alto: le regala terreno al léxico
frases calcadas       3 / 30      cada una infla el canal léxico

LAS CINCO PREGUNTAS MÁS PEGADAS AL TEXTO DE SU DOCUMENTO
──────────────────────────────────────────────────────────────────────────
 80 %  32015R0400   ¿Qué se modificó en 2015 sobre los límites de residuos de …
       calca del acto: «de residuos de plaguicidas en…»
 46 %  32010D0352   ¿Con qué criterio se reparte entre los países el dinero eu…
       calca del acto: «vigilancia de la actividad pesquera…»
 45 %  32014D0190   ¿Dónde se establece cuánto le corresponde por año a cada p…
       calca del acto: «de los fondos estructurales y…»
 83 %  32013R1382   ¿Qué programa de financiación sostuvo entre 2014 y 2020 la…
 82 %  32012R1257   ¿Qué instrumento permite que una patente 

### 10.2 · El protocolo

**Congelás el eval set, movés una variable, mirás la tabla.** Eso es todo, y es literalmente lo que se le pide a un proyecto de la materia: no «usé reranking», sino «probé con y sin, y acá está la diferencia sobre mi corpus».

Los números de abajo salen de treinta filas sobre el corpus de esta clase. **No son benchmarks.** Sirven para decidir qué poner en tu pipeline, no para afirmar qué técnica es mejor en general.

In [59]:
mostrar_tabla("Todo lo construido hasta acá, sobre el mismo eval set")


Todo lo construido hasta acá, sobre el mismo eval set
──────────────────────────────────────────────────────────────────────
                         temáticas   puesto   identificador   contexto
denso, pedazo crudo          26/30      1.4            2/15   1882 tok
+ contextualización          23/30      1.3            2/15   2175 tok
+ canal léxico (RRF)         26/30      1.5            6/15   2151 tok
+ padre → hijo               26/30      1.5            6/15   3746 tok



### 10.3 · La primera técnica que pasa por la vara: HyDE

**Por qué ésta está acá y no con las otras.** Las piezas de las secciones 6, 7 y 8 llegaron a la tabla con su fila ya medida. HyDE es distinta en dos cosas: necesita **el generador** —una llamada por consulta, segundos en vez de milisegundos— y es la que más chances tiene de **no** servir en este corpus. Por eso va pegada al protocolo: no es parte de la vara, es el caso testigo de para qué sirve tenerla.

Queda una asimetría que ninguna de las piezas anteriores toca, y que la Clase 4 ya había nombrado: **una consulta corta busca un pasaje largo que la responde**. No se parecen. La consulta está escrita como pregunta; el pasaje, como texto legal.

La idea de HyDE es rara y vale la pena decirla despacio: se le pide al modelo que **invente** el pasaje que respondería la pregunta, y se busca con **eso**. El documento hipotético puede estar factualmente mal y servir igual, porque lo que aporta no son hechos: es el **patrón de relevancia** —el tema, el vocabulario, la forma de pasaje—.

Y no llega al generador. Es una **llave de búsqueda**: se embebe, se busca con él, y se tira. Al contexto entran pasajes reales.

HyDE es un caso particular de una familia que tiene nombre: **transformación de la consulta** (*query transformation*, o *query expansion*) —todo lo que reescribe la pregunta antes de buscar, en lugar de tocar el corpus o el índice—. Es la única pieza del notebook que interviene ahí.

In [60]:
def hyde(pregunta):
    """Un pasaje hipotetico que responderia la pregunta. Se usa para BUSCAR y se descarta."""
    return ask("Escribí un párrafo breve, en el estilo de un acto normativo europeo, que "
               "responda esta pregunta. No aclares que es hipotético; escribí el párrafo "
               "y nada más.\n\nPregunta: " + pregunta,
               num_predict=200)

if OLLAMA_OK:
    demo = hyde(preg_demo)
    mostrar("LA PREGUNTA", recortar(preg_demo, 200))
    mostrar("EL DOCUMENTO HIPOTÉTICO (puede ser falso, y servir igual)", recortar(demo, 500))
    print("Esto NO entra al contexto del generador. Se embebe, se busca con él, y se tira.")

──────────────────────────────────────────────────────────────────────────
LA PREGUNTA
──────────────────────────────────────────────────────────────────────────
¿Con qué criterio se reparte entre los países el dinero europeo que cubre
los gastos de vigilancia de la actividad pesquera durante 2010?

──────────────────────────────────────────────────────────────────────────
EL DOCUMENTO HIPOTÉTICO (puede ser falso, y servir igual)
──────────────────────────────────────────────────────────────────────────
El dinero europeo destinado a la vigilancia de la actividad pesquera
durante el año 2010 se reparte entre los países miembros en función de la
cuota de pesca asignada a cada Estado, basada en los acuerdos de
distribución establecidos en el marco del Reglamento (CE) n.º 1098/2007,
que establece criterios de distribución proporcional a la capacidad de
pesca y a la cuota de pesca asignada, asegurando así una asignación
equitativa y sostenible de los recursos marinos.

Esto NO entra al cont

In [61]:
# HyDE cambia UNA cosa: el texto con el que busca el canal DENSO. El lexico, la fusion y
# el padre/hijo quedan intactos -si los sacara no estariamos midiendo HyDE-, y ademas el
# lexico sigue recibiendo la pregunta original. El porque, en el comentario de recuperar().
if OLLAMA_OK:
    liberar_gpu()          # HyDE son decenas de llamadas: conviene entrar con la placa limpia
    t0 = time.time()
    # Generamos también para las consultas de identificador. No es un descuido: es donde
    # HyDE tiene más para perder —inventar un pasaje sobre una cita que el modelo no conoce
    # apunta a cualquier lado— y verlo medido vale más que advertirlo.
    HYDE_DE = {p: hyde(p) for p, _ in EVAL_SET[:N_EVAL] + EVAL_ID}
    print("%d documentos hipotéticos en %.0f s (%.1f s cada uno)"
          % (len(HYDE_DE), time.time() - t0, (time.time() - t0) / max(1, len(HYDE_DE))))

    fila = fila_de(lambda q, k: recuperar(q, k, contextual=True, lexico=True, padre=True,
                                          texto_busqueda=HYDE_DE.get(q)))
    TABLA_MADRE.append(("+ HyDE", fila))
    mostrar_tabla()
else:
    print("Sin Ollama: HyDE necesita el generador. Su fila queda fuera de la tabla.")

VRAM · PyTorch retenía 2.3 GB, ahora 2.3 GB · libre en la placa: 6454 MiB
45 documentos hipotéticos en 208 s (4.6 s cada uno)

Una fila por pieza · el documento correcto entre los 5 entregados
──────────────────────────────────────────────────────────────────────
                         temáticas   puesto   identificador   contexto
denso, pedazo crudo          26/30      1.4            2/15   1882 tok
+ contextualización          23/30      1.3            2/15   2175 tok
+ canal léxico (RRF)         26/30      1.5            6/15   2151 tok
+ padre → hijo               26/30      1.5            6/15   3746 tok
+ HyDE                       25/30      1.4           11/15   3390 tok



**Cómo leerlo, salga como salga — y acá el «salga como salga» es en serio.**

Si mejoró: el pasaje hipotético se parece más a un pasaje real que la pregunta original, y eso acerca la búsqueda al lugar correcto.

Si **empeoró**, que es un resultado frecuente con un modelo chico y un dominio especializado, la explicación es la que el propio mecanismo anticipa: HyDE apuesta a que el modelo sabe **cómo se ve** una respuesta a esa pregunta. Si no conoce el dominio —y el derecho europeo en español no es el fuerte de un modelo de 8B— inventa un párrafo que apunta al lugar equivocado, y la búsqueda se va detrás. Sumale que cuesta una llamada al modelo **por consulta**: son segundos por búsqueda, contra milisegundos de todo lo demás.

> **Y una decisión de implementación que vale una fila entera de la tabla.** La forma obvia de escribir HyDE es reemplazar la consulta y listo. Pero acá hay **dos canales**, y el léxico existe para encontrar la cita literal: si le pasás el párrafo inventado, le estás sacando justamente su insumo. La primera versión de este notebook hacía eso, y la columna `identificador` se desplomaba de 5/8 a 1/8 — no porque HyDE sea malo con identificadores, sino porque estaba **desarmando la pieza que los resolvía**. Por eso `recuperar()` le entrega el documento hipotético sólo al canal denso. Es la diferencia entre medir una técnica y medir un accidente.

Ése es exactamente el caso donde medir paga. Sin la tabla, HyDE es una técnica publicada en un paper y uno la pone. Con la tabla, es una técnica que en **este** corpus rinde esto.

▶ **Láminas 28–32, 41–43** (el reenganche; recall y precision; la vara; el protocolo; la brecha consulta–pasaje; la paradoja; cuándo conviene no usarlo).

### Por qué subió el identificador · el resultado que no esperábamos

Mirá la columna `identificador` de la tabla: **6/15 → 11/15**. Es el salto más grande que produce cualquier pieza del día, y está justo en la columna donde el comentario de la celda de arriba apostaba a que HyDE iba a **perder**.

Vale la pena abrir la caja, porque la explicación estaba escrita desde la sección 5 y no la vimos.

**La hipótesis principal: la brecha de forma es máxima acá.** HyDE existe para arreglar el desajuste entre una consulta corta y un pasaje largo. De todo nuestro eval set, la consulta de identificador es el desajuste más extremo posible: `Decisión (UE) no 1982/2006` son **cinco tokens de pura forma** —lo que la sección 5.3 llamó «no significa nada, no tiene campo semántico»— contra pasajes de 350 tokens de texto legal.

Y acá está el punto: para una consulta de identificador el modelo **no tiene que adivinar nada**. No necesita saber de qué trata esa decisión. Le alcanza con escupir un párrafo en registro normativo que **repita la cita**. Eso lo hace trivialmente, porque la cita es lo único que tiene en el prompt.

**La hipótesis secundaria: RRF deja de intercalar.** En la sección 7 vimos que, con una consulta de identificador, ninguna fila sumaba dos votos: el canal denso metía basura en los puestos impares y empujaba al correcto hacia abajo. Si ahora el denso trae cosas que el léxico también trae, el correcto suma **dos votos** y sube. Que la columna `puesto` también haya mejorado apunta en esa dirección.

La celda de abajo separa las dos: mira si el documento hipotético repite el número de la cita, y en qué puesto del canal denso queda el acto correcto **antes y después** de HyDE.

In [71]:
# Dos mediciones sobre las mismas consultas de identificador:
#   1) el documento hipotetico, .repite el numero de la cita?  -> hipotesis de la FORMA
#   2) en que puesto del canal DENSO queda el acto correcto, con y sin HyDE
# Si el puesto baja fuerte, gano la hipotesis de la forma. Si no baja pero igual sube en la
# tabla final, lo que mejoro fue la FUSION: el denso dejo de ensuciar el RRF.

def puesto_denso(texto, gold, tope=2000):
    """Puesto del primer pedazo del acto `gold` en el canal denso, buscando con `texto`."""
    q = embeber([texto])[0]
    for puesto, i in enumerate(np.argsort(MATRIZ_CTX @ q)[::-1][:tope], 1):
        if PEDAZOS_CTX[int(i)]["celex"] == gold:
            return puesto
    return None

if OLLAMA_OK and "HYDE_DE" in globals():
    filas = []
    for cita, gold in EVAL_ID[:5]:
        hip = HYDE_DE.get(cita, "")
        # El nucleo distintivo de una cita son sus numeros: si el modelo los repitio, el
        # parrafo hipotetico carga la misma forma superficial que el pasaje que buscamos.
        nums = [n for n in re.findall(r"\d+", cita) if len(n) >= 3]
        eco = bool(nums) and all(n in hip for n in nums)
        p_sin, p_con = puesto_denso(cita, gold), puesto_denso(hip, gold) if hip else None
        filas.append((recortar(cita, 26), {
            "repite el nº": "sí" if eco else "no",
            "puesto denso SIN HyDE": p_sin if p_sin else "> 2000",
            "puesto denso CON HyDE": p_con if p_con else "> 2000"}))
    tabla("El canal denso frente a una consulta de identificador",
          ["repite el nº", "puesto denso SIN HyDE", "puesto denso CON HyDE"], filas)

    mejoraron = sum(1 for _, f in filas
                    if isinstance(f["puesto denso CON HyDE"], int)
                    and (not isinstance(f["puesto denso SIN HyDE"], int)
                         or f["puesto denso CON HyDE"] < f["puesto denso SIN HyDE"]))
    print()
    print("el acto correcto subió en el canal denso en %d de %d casos" % (mejoraron, len(filas)))
    print("  - si subió en casi todos: ganó la hipótesis de la FORMA (el párrafo se parece al pasaje)")
    print("  - si no subió y la tabla igual mejoró: lo que ganó fue la FUSIÓN, no la búsqueda densa")

    # Y uno entero, para verlo con los ojos: cinco tokens de entrada, un pasaje de salida.
    cita_ej = EVAL_ID[0][0]
    print()
    mostrar("LA CONSULTA (pura forma, sin nada que parafrasear)", cita_ej)
    mostrar("EL DOCUMENTO HIPOTÉTICO (lo que se embebe en su lugar)",
            parrafo(HYDE_DE.get(cita_ej, ""), lineas=8))
else:
    print("Sin Ollama o sin HYDE_DE: hay que correr la celda de HyDE antes que ésta.")


El canal denso frente a una consulta de identificador
───────────────────────────────────────────────────────────────────────────────────────────
                               repite el nº   puesto denso SIN HyDE   puesto denso CON HyDE
Decisión (UE) no 1982/2006               no                  > 2000                  > 2000
Decisión (UE) no 2013/743                no                    1415                    1199
Decisión 1219/2011/UE                    sí                  > 2000                  > 2000
Decisión 1313/2013/UE                    sí                  > 2000                     217
Decisión 1982/2006/CE                    sí                  > 2000                  > 2000


el acto correcto subió en el canal denso en 2 de 5 casos
  - si subió en casi todos: ganó la hipótesis de la FORMA (el párrafo se parece al pasaje)
  - si no subió y la tabla igual mejoró: lo que ganó fue la FUSIÓN, no la búsqueda densa

─────────────────────────────────────────────────────────────

In [73]:
# Si el denso no lo encuentra, .de donde sale el 6/15 -> 11/15? Reproducimos el pipeline
# paso por paso y miramos DONDE se mueve el acto correcto. Tres decisiones de medicion:
#   1) se imprimen las consultas que VOLTEARON -de afuera a adentro del top-k-, que son las
#      unicas que explican el salto. Las demas no tienen nada que contar.
#   2) se muestra el puesto FUSIONADO al lado del entregado. Si el fusionado no se mueve y
#      el entregado si, el mecanismo es la deduplicacion por padre. Si se mueven los dos,
#      actuo la FUSION: algun pedazo aparece en las dos listas y suma dos votos.
#   3) la concentracion se mide en la CABEZA del denso, no sobre los 50. Para liberar un
#      lugar entre los 5 entregados, los duplicados tienen que estar arriba: que la cola
#      se concentre no cambia nada.

CABEZA = 6

def anatomia(pregunta, gold, clave):
    """Sigue al acto correcto por las tres etapas: denso -> fusion -> deduplicado."""
    q = embeber([clave])[0]
    densos = [PEDAZOS_CTX[int(i)] for i in np.argsort(MATRIZ_CTX @ q)[::-1][:N_CANDIDATOS]]
    fusion = rrf([densos, buscar_bm25(pregunta, N_CANDIDATOS)], N_CANDIDATOS)
    entregado = a_padres(fusion)
    puesto = lambda l: next((i for i, p in enumerate(l, 1) if p["celex"] == gold), None)
    return {"cabeza": len({p["celex"] for p in densos[:CABEZA]}),
            "denso": puesto(densos),          # ¿entró el acto correcto a la lista densa?
            "fusionado": puesto(fusion), "entregado": puesto(entregado)}

if OLLAMA_OK and "HYDE_DE" in globals() and not _falta("a_padres"):
    dentro = lambda p: bool(p) and p <= K_ENTREGO
    par = lambda a, b: "%s → %s" % (a if a else "—", b if b else "—")

    casos = []
    for cita, gold in EVAL_ID:
        sin = anatomia(cita, gold, cita)
        con = anatomia(cita, gold, HYDE_DE.get(cita) or cita)
        casos.append((cita, sin, con, not dentro(sin["entregado"]) and dentro(con["entregado"])))

    volteadas = [c for c in casos if c[3]]
    cols = ["puesto en la lista densa", "actos en el top-%d denso" % CABEZA,
            "puesto fusionado", "puesto entregado"]
    tabla("Las %d consultas que HyDE hizo entrar  ·  sin HyDE → con HyDE" % len(volteadas), cols,
          [(recortar(c[0], 24), {cols[0]: par(c[1]["denso"], c[2]["denso"]),
                                 cols[1]: par(c[1]["cabeza"], c[2]["cabeza"]),
                                 cols[2]: par(c[1]["fusionado"], c[2]["fusionado"]),
                                 cols[3]: par(c[1]["entregado"], c[2]["entregado"])})
           for c in volteadas])

    bajo = sum(1 for c in volteadas if c[2]["cabeza"] < c[1]["cabeza"])
    igual = sum(1 for c in volteadas if c[1]["fusionado"] == c[2]["fusionado"])
    print()
    entro = sum(1 for c in volteadas if not c[1]["denso"] and c[2]["denso"])
    print("de las %d que voltearon:" % len(volteadas))
    print("   %d tienen el acto correcto DENTRO de la lista densa sólo con HyDE" % entro)
    print("       → dos votos en la fusión: HyDE sí cerró la brecha de forma")
    print("   %d traen MENOS actos distintos en la cabeza del denso" % bajo)
    print("   %d conservan el puesto fusionado  → el cambio ocurre al DEDUPLICAR por padre" % igual)
    print("   %d cambian el puesto fusionado    → ahi actuo la FUSION, no el padre"
          % (len(volteadas) - igual))
    print()
    print("control · el acto correcto entre los %d entregados" % K_ENTREGO)
    print("   sin HyDE: %2d/%d    con HyDE: %2d/%d"
          % (sum(1 for c in casos if dentro(c[1]["entregado"])), len(casos),
             sum(1 for c in casos if dentro(c[2]["entregado"])), len(casos)))
    print("   (tienen que dar las dos filas de la tabla madre: si no, la anatomía mide otra cosa)")

    # ── La fusión, abierta ──────────────────────────────────────────────────────────
    # El acto correcto no entra a la lista densa ni con HyDE ni sin HyDE, y aun así sube.
    # La única forma de entender por qué es mirar la cuenta pedazo por pedazo.
    if volteadas:
        cita, gold = volteadas[0][0], dict(EVAL_ID)[volteadas[0][0]]
        for etiqueta, clave in (("SIN HyDE", cita), ("CON HyDE", HYDE_DE.get(cita) or cita)):
            q = embeber([clave])[0]
            densos = [PEDAZOS_CTX[int(i)] for i in np.argsort(MATRIZ_CTX @ q)[::-1][:N_CANDIDATOS]]
            lexicos = buscar_bm25(cita, N_CANDIDATOS)
            pd = {p["id"]: i for i, p in enumerate(densos, 1)}
            pl = {p["id"]: i for i, p in enumerate(lexicos, 1)}
            print()
            print("%s · %s   (gold %s)" % (etiqueta, recortar(cita, 34), gold))
            print("   solapamiento denso∩léxico: %d pedazos · el gold está en el léxico en el puesto %s"
                  % (len(set(pd) & set(pl)),
                     next((i for i, p in enumerate(lexicos, 1) if p["celex"] == gold), "—")))
            print("   %-4s %-12s %6s %7s %10s" % ("#", "celex", "denso", "léxico", "puntaje"))
            for i, p in enumerate(rrf([densos, lexicos], 6), 1):
                print("   %-4d %-12s %6s %7s %10.5f%s"
                      % (i, p["celex"], pd.get(p["id"], "—"), pl.get(p["id"], "—"), p["rrf"],
                         "   ← el correcto" if p["celex"] == gold else ""))

else:
    print("Faltan piezas: esta celda necesita HYDE_DE y a_padres.")


Las 5 consultas que HyDE hizo entrar  ·  sin HyDE → con HyDE
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                         puesto en la lista densa   actos en el top-6 denso   puesto fusionado   puesto entregado
Decisión 1999/311/CE                        — → —                     5 → 5             10 → 2             10 → 2
Decisión 1999/43/CE                         — → —                     5 → 5              6 → 2              6 → 2
Decisión 1999/476/CE                        — → —                     4 → 6              6 → 2              6 → 2
Decisión 1999/802/CE                        — → —                     4 → 6              8 → 2              8 → 2
Decisión 2000/532/CE                        — → —                     3 → 3              9 → 2              9 → 2


de las 5 que voltearon:
   0 tienen el acto correcto DENTRO de la lista densa sólo con HyDE
       → dos votos en la fusión: HyDE sí cerró

**Lo que salió, y no es lo que esperábamos.** El control reproduce la tabla madre (6/15 → 11/15), así que la anatomía mide el pipeline de verdad. Y las tres primeras columnas descartan las dos explicaciones plausibles: **el acto correcto no entra a la lista densa** —ni con HyDE ni sin HyDE—, y **la deduplicación por padre no mueve una sola posición** (el puesto fusionado y el entregado son idénticos en las cinco filas).

La respuesta está en la última línea del volcado, y es una sola cifra:

```
SIN HyDE · solapamiento denso∩léxico: 8 pedazos   → el correcto queda 10º
CON HyDE · solapamiento denso∩léxico: 0 pedazos   → el correcto queda  2º
```

**El canal léxico tenía el documento correcto en el puesto 1. Las dos veces.** Nunca falló. Lo que cambió es lo que RRF hizo con él.

**Sin HyDE, los dos canales coincidieron en ocho pedazos equivocados.** Cada uno de esos sumó dos votos —entre 0,024 y 0,030— y se quedó con los primeros lugares. El correcto, con un solo voto (1/61 = 0,01639), quedó décimo. La cuenta cierra exacta: ocho pedazos con doble voto más el primero del denso, que empata y gana el desempate, son nueve por delante.

**Con HyDE, el solapamiento cae a cero.** El párrafo hipotético manda al canal denso a otra región del espacio, y ahí no coincide con nada. Sin solapamiento RRF no fusiona: **intercala**. Y al intercalar, el primero del léxico cae en el puesto 2.

> **La conclusión, que es incómoda y vale toda la sección.** HyDE no mejoró la búsqueda: **rompió un consenso falso**. Y eso obliga a matizar lo que dijimos en la sección 7: RRF premia el consenso porque **supone que los dos canales fallan de forma independiente**. Frente a una cita pelada esa suposición es falsa —el denso la manda al cúmulo de referencias normativas y BM25 matchea los mismos tokens genéricos (`decisión`, `CE`, los años)—, así que los dos fallan **en la misma dirección**. Ahí su acuerdo no es evidencia: es error correlacionado, y la fusión lo amplifica en vez de corregirlo.

Una lista irrelevante estorba menos que una lista sistemáticamente equivocada en la misma dirección que su compañera. Ése es el mecanismo real, y no se parece en nada a lo que promete el paper de HyDE.

**Y la lección de método, que es la que se transfiere.** Una técnica puede mover una métrica **por una razón que no es la suya**. Si nos quedábamos con la tabla —«HyDE sube el identificador de 6/15 a 11/15, dejémoslo»— habríamos atribuido el mérito al mecanismo equivocado, y en un corpus donde los canales fallen de forma independiente la mejora desaparecería sin explicación. Medir dice **qué** pasó; sólo abrir la caja dice **por qué**.

## 11 · Afinar: re-puntuar

Todo lo que hicimos hasta acá amplió **qué** trae la búsqueda. Ninguna pieza tocó **en qué orden** lo trae.

Y el orden lo decide un coseno contra un vector que se calculó en la sección 6, **antes de que existiera la pregunta**. Ahí está la falla, y no es un error de implementación: es la forma del método. Al embeber cada pasaje por separado, el modelo tiene que comprimir 350 tokens en 1024 números **sin saber qué le van a preguntar**. Tiene que adivinar qué es lo importante — y un pasaje que habla de tres cosas tiene **un solo vector para las tres**.

**¿Hay una forma de comparar que no sea a ciegas?** Sí, y es la más obvia: meter la consulta y el pasaje **juntos** al modelo, y que la red lea las dos cosas a la vez. Así comparaba textos BERT antes de que existiera SBERT.

Y la descartamos en la Clase 4, con la cuenta hecha: metiéndolos juntos no queda ningún vector guardable, así que hay que correr el modelo una vez **por cada par**. Comparar diez mil oraciones entre sí son unos cincuenta millones de pasadas, unas sesenta y cinco horas. Ése es el motivo por el que existe el embedder que venimos usando todo el día. **La descartamos por cara, no por mala.**

Hoy les ponemos nombre a los dos modos:

- **Bi-encoder** — el de SBERT y el de `bge-m3`. Comprime cada pasaje a un vector **sin saber la consulta**. Por eso se puede precalcular el corpus entero, y por eso es barato.
- **Cross-encoder** — el caro. Ve la consulta y el pasaje **juntos**, y puede mirar qué palabra de una se relaciona con cuál de la otra. Por eso puntúa mucho mejor. Y por eso no se puede precalcular nada: hasta que no llega la consulta, no hay nada que computar.

**Y el caro vuelve** — pero no sobre el corpus. Sobre los cincuenta candidatos que ya trajo la primera etapa.

In [63]:
n_corpus = len(PEDAZOS_CTX)
print("puntuar el corpus entero con el cross-encoder, por consulta:")
print("   %s inferencias" % "{:,}".format(n_corpus).replace(",", "."))
print("puntuar sólo los candidatos de la primera etapa:")
print("   %d inferencias" % N_CANDIDATOS)
print()
print("Es el mismo modelo caro. Lo que cambió es el TAMAÑO DEL CONJUNTO sobre el que corre.")
print("De ahí salen gratis las dos etapas, y los dos números: recuperás ancho, entregás poco.")
print("Anthropic, por ejemplo, recupera ciento cincuenta y entrega veinte.")

puntuar el corpus entero con el cross-encoder, por consulta:
   30.730 inferencias
puntuar sólo los candidatos de la primera etapa:
   50 inferencias

Es el mismo modelo caro. Lo que cambió es el TAMAÑO DEL CONJUNTO sobre el que corre.
De ahí salen gratis las dos etapas, y los dos números: recuperás ancho, entregás poco.
Anthropic, por ejemplo, recupera ciento cincuenta y entrega veinte.


**De ahí salen gratis dos cosas.** La primera es el **paradigma de dos etapas**: una barata y amplia que reduce millones a decenas, y una cara y precisa que ordena esas decenas. La segunda son los dos números que hay que elegir, y que casi nunca son el mismo: **cuántos recuperás** y **cuántos entregás**.

Lo caro se vuelve barato cuando se lo corre sobre poco. No era el modelo el problema: era el tamaño del conjunto sobre el que lo corríamos.

In [64]:
from sentence_transformers import CrossEncoder

liberar_gpu()          # entra un modelo mas a la placa: le hacemos lugar primero

# max_length=512 explicito: es la ventana del modelo y el motivo por el que los hijos se
# dimensionaron a 350 tokens alla en la seccion 4. Si le pasaramos padres, los truncaria.
repuntuador = CrossEncoder(REPUNTUADOR_ID, max_length=512, device=DISPOSITIVO)

def repuntuar(pregunta, candidatos, n):
    """Re-ordena los candidatos mirando consulta y pasaje JUNTOS. Siempre sobre hijos."""
    if not candidatos:
        return candidatos
    puntajes = repuntuador.predict([(pregunta, c["texto"]) for c in candidatos],
                                   batch_size=32, show_progress_bar=False)
    pares = sorted(zip(candidatos, puntajes), key=lambda x: -float(x[1]))
    return [dict(c, ce=float(s)) for c, s in pares][:n]

print("re-puntuador cargado ·", REPUNTUADOR_ID)
print("ventana:", repuntuador.tokenizer.model_max_length, "tokens")

VRAM · PyTorch retenía 2.3 GB, ahora 2.3 GB · libre en la placa: 6454 MiB


model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

re-puntuador cargado · cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
ventana: 512 tokens


In [65]:
# El mismo top-k, antes y despues. Sin cambiar nada mas.
antes   = recuperar(preg_demo, 5, contextual=True, lexico=True)
despues = recuperar(preg_demo, 5, contextual=True, lexico=True, rerank=True)

print("consulta:", recortar(preg_demo, 100), "\n")
tabla("El mismo top-5, antes y después de re-puntuar", ["antes", "después"],
      [("puesto %d" % (i + 1),
        {"antes": antes[i]["celex"] if i < len(antes) else "—",
         "después": despues[i]["celex"] if i < len(despues) else "—"})
       for i in range(5)])
print("  gold:", gold_demo)
print()
for i, p in enumerate(despues, 1):
    print("%d. ce=%+.2f  %-12s %s" % (i, p.get("ce", 0.0), p["celex"], recortar(p["texto"], 66)))

consulta: ¿Con qué criterio se reparte entre los países el dinero europeo que cubre los gastos de vigilancia d… 


El mismo top-5, antes y después de re-puntuar
──────────────────────────────────────
                    antes      después
puesto 1       32010D0711   32010D0352
puesto 2       32010D0352   32010D0711
puesto 3       32010D0352   32010D0352
puesto 4       32010D0352   32010D0711
puesto 5       32010R1225   32010D0711

  gold: 32010D0352

1. ce=+4.75  32010D0352   [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera …
2. ce=+4.50  32010D0711   [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera …
3. ce=+3.97  32010D0352   [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera …
4. ce=+0.80  32010D0711   [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera …
5. ce=+0.37  32010D0711   [DECISIÓN DE LA COMISIÓN — relativa a la participación financiera …


**Qué acaba de pasar.** Los cinco que entregamos salen del mismo conjunto de candidatos que antes: la primera etapa no cambió. Lo que cambió es **el orden**, y con él cuál de los cincuenta se queda con los cinco lugares.

Fijate en los puntajes del cross-encoder. No están acotados a un rango como el coseno —son logits, pueden ser negativos— y por eso tampoco se pueden mezclar con nada. Acá no hace falta: es la última etapa, y lo único que se usa de ellos es el orden que producen.

> **Cómo se llama esto afuera.** El paso se llama **reranking** y al modelo que lo hace, **reranker**; al patrón de las dos etapas, *two-stage retrieval* o *retrieve-and-rerank*. Nosotros decimos «re-puntuar», que es lo mismo.
>
> **Y el cross-encoder es una forma de rerankear, no la única.** Lo que define al paso es *volver a mirar los candidatos con un criterio más caro*; con qué se lo hace es otra decisión. También se rerankea con un **LLM** al que se le pide que ordene la lista (*LLM-as-reranker*, RankGPT), con modelos de **interacción tardía** (*late interaction*) como ColBERT —a mitad de camino entre los dos encoders: guarda un vector por token en vez de uno por pasaje—, y sobre todo con **APIs de pago**: Cohere Rerank, Voyage, Jina. Por dentro casi todas son cross-encoders; en producción la mayoría paga la API en vez de montar el modelo. Conviene saberlo porque es lo primero que aparece al buscar «reranker».

Ahora la fila que faltaba en la tabla.

In [66]:
agregar_fila("+ re-puntuador", contextual=True, lexico=True, padre=True, rerank=True)
mostrar_tabla("La tabla completa · el mismo eval set, una pieza por fila")


La tabla completa · el mismo eval set, una pieza por fila
──────────────────────────────────────────────────────────────────────
                         temáticas   puesto   identificador   contexto
denso, pedazo crudo          26/30      1.4            2/15   1882 tok
+ contextualización          23/30      1.3            2/15   2175 tok
+ canal léxico (RRF)         26/30      1.5            6/15   2151 tok
+ padre → hijo               26/30      1.5            6/15   3746 tok
+ HyDE                       25/30      1.4           11/15   3390 tok
+ re-puntuador               28/30      1.6           10/15   3427 tok



**Y no sólo elige: produce el orden.** Eso cobra la deuda de la Clase 7. El modelo usa peor lo que cae en el **medio** del contexto —la curva en U de *lost in the middle*—, así que no alcanza con que los cinco buenos estén ahí: importa en qué orden. El re-puntuador es lo que decide ese orden, y por eso se entrega en **su** orden y no en el que venía.

Menos pasajes bien ordenados le ganan a muchos apilados.

> **Nota de hardware, y es de las que hacen perder una tarde.** En esta notebook conviven **dos runtimes** sobre la misma placa: PyTorch —que tiene el embedder y el re-puntuador— y Ollama —que tiene el generador—. PyTorch **cachea** la memoria que libera en vez de devolverla al sistema, así que después de embeber decenas de miles de pedazos puede estar reteniendo varios gigabytes que no usa. Ollama, que es otro proceso, ve esa memoria ocupada y **descarga capas del modelo al procesador**. La generación cae de unos veinticinco tokens por segundo a unos tres, y no hay ningún error: hay una celda que parece colgada. Por eso llamamos a `liberar_gpu()` antes de cada sección que carga un modelo nuevo.

> **Ojo con un detalle que cuesta caro:** el modelo por defecto del ecosistema para esta tarea está entrenado **sólo en inglés**. Sobre pasajes en español re-puntuaría degradado, y el único movimiento de precision de toda la clase podría salir al revés. Hay versiones multilingües —la que usamos es una— y hay que elegirlas a propósito.

### 11.1 · La pila completa

```
offline :  cargar → cortar por estructura → contextualizar → embeber → indexar
online  :  consulta → denso + léxico → RRF → re-puntuar → subir al padre → top-n ordenado
```

Ése es el orden y no es arbitrario. El re-puntuador va **antes** de subir al padre, porque su ventana es de 512 tokens y un artículo entero no le entra: se puntúan hijos, siempre. Y el padre se arma al final, cuando ya está decidido qué entra.

▶ **Láminas 44–48** (lo que descartamos por caro; los dos modos con nombre; el caro vuelve; el orden importa; la pila completa).

## Cierre — el mapa de la clase

| pieza | qué problema resuelve | dónde se rompe |
|---|---|---|
| cortar por estructura | el documento no entra en el embedder, y un vector por documento promedia veinte temas | si el documento no tiene estructura, volvés al corte ciego |
| solape | el corte parte la oración que contestaba | no repone los referentes, sólo la continuidad |
| contextualizar | el pedazo se indexa sin decir de qué norma es | el prefijo es igual para todos los pedazos de un acto |
| vector store | la lista en memoria no persiste, no se actualiza y no filtra | el índice aproximado cambia exactitud por velocidad |
| canal léxico + RRF | el embedding pierde la forma superficial, y un identificador **es** forma superficial | si el preprocesamiento parte los identificadores, no sirve de nada |
| padre → hijo | lo que conviene indexar y lo que conviene entregar no son del mismo tamaño | entregás más tokens, y la ventana es finita |
| HyDE | la consulta corta no se parece al pasaje largo | si el modelo no conoce el dominio, apunta al lugar equivocado |
| re-puntuar | el bi-encoder comprime sin saber la consulta | cuesta una pasada por candidato, y su ventana es la más chica de todas |
| prompt con abstención y cita | el modelo completa cuando no sabe | pedirlo es una súplica, no una garantía |

**El hilo entero.** Empezamos con el mismo loop de la Clase 4 —embeber, coseno, top-k, generar— y lo rompimos en tres puntos: el documento no entraba, la lista no escalaba, y había consultas que el coseno no puede resolver. Cada pieza de la segunda mitad apareció para tapar uno de esos agujeros, y ninguna apareció porque estuviera de moda: apareció con su número en la tabla.

**Y nunca tocamos los pesos.** Ni una vez. La Unidad 2 fue justamente sobre tocarlos; hoy todo fue ingeniería de **qué lee** el modelo. Lo que se recupera es el techo de lo que la respuesta puede ser, y la generación decide cuánto de ese techo se alcanza.

In [67]:
# El pipeline final, armado. Esto es lo que te llevas al Proyecto 2.
def rag(pregunta, k=K_ENTREGO):
    """offline: cargar -> cortar por estructura -> contextualizar -> embeber -> indexar
       online : consulta -> denso + lexico -> RRF -> re-puntuar -> subir al padre -> generar"""
    pasajes = recuperar(pregunta, k, contextual=True, lexico=True, padre=True, rerank=True)
    return responder_rag(pregunta, pasajes)

if OLLAMA_OK and EVAL_SET:
    preg_fin, gold_fin = EVAL_SET[0]
    # Mostramos TAMBIEN que pasajes le llegaron al generador. Sin eso, una abstencion se
    # lee como una falla del sistema, cuando puede ser exactamente lo correcto: la tabla
    # mide DOCUMENTOS y el generador necesita el PASAJE. No son el mismo numero.
    entregados = recuperar(preg_fin, K_ENTREGO, contextual=True, lexico=True,
                           padre=True, rerank=True)
    salida = responder_rag(preg_fin, entregados)
    mostrar("EL PIPELINE ENTERO, SOBRE UNA PREGUNTA",
            "pregunta          : %s\ndocumento correcto: %s\nbasta_el_contexto : %s"
            "\ncita              : %s\nrespuesta         : %s"
            % (recortar(preg_fin, 110), gold_fin, salida["basta_el_contexto"],
               salida["cita"], recortar(salida["respuesta"], 350)))
    print()
    print("LO QUE LE LLEGÓ AL GENERADOR")
    print("─" * 74)
    for p in entregados:
        marca = "  ← del documento correcto" if p["celex"] == gold_fin else ""
        print("  %-12s %-30s%s" % (p["celex"], p["seccion"][:30], marca))

──────────────────────────────────────────────────────────────────────────
EL PIPELINE ENTERO, SOBRE UNA PREGUNTA
──────────────────────────────────────────────────────────────────────────
pregunta : ¿Con qué criterio se reparte entre los países el dinero europeo
que cubre los gastos de vigilancia de la activ…
documento correcto: 32010D0352
basta_el_contexto : False
cita : []
respuesta : No lo sé


LO QUE LE LLEGÓ AL GENERADOR
──────────────────────────────────────────────────────────────────────────
  32010D0352   encabezado                      ← del documento correcto
  32010D0711   encabezado                    
  32010D0352   Artículo 1                      ← del documento correcto
  32010D0711   Artículo 3                    
  32010D0711   Artículo 5                    


**Cómo leer esta celda, que es el último ejercicio de la clase.** Mirá dos cosas juntas: si entre los pasajes entregados aparece el documento correcto, y si el generador contestó o se abstuvo.

Pueden no coincidir. Y cuando no coinciden, el notebook está exhibiendo su propia lección. La tabla dice `12/12` en la última fila: el **documento** correcto se entrega en las doce preguntas. Pero `medir()` colapsa los pasajes a documentos, así que ese número no promete que haya llegado el **artículo** que contesta. Si el re-puntuador subió el encabezado y el `Artículo 1`, y el que tenía la respuesta era el `Artículo 10`, la tabla sigue marcando 12/12 y el generador se abstiene **con razón**.

Eso no es una falla del sistema: es la prueba de que las **dos superficies** con las que abrió la clase son de verdad dos. Una métrica de recuperación a nivel documento puede estar en verde mientras la respuesta no sale. Con una sola métrica al final sabrías que salió mal; no sabrías dónde.

In [68]:
# El resumen se imprime desde los datos VIVOS, no desde lo que recordamos haber visto:
# un resumen que puede contradecir a la pantalla no es un resumen.
print("=" * 74)
print("LO QUE MEDIMOS, TODO JUNTO")
print("=" * 74)
mostrar_tabla("Cada fila agrega una pieza sobre la anterior")

if len(TABLA_MADRE) >= 2:
    primera, ultima = TABLA_MADRE[0], TABLA_MADRE[-1]
    print("de '%s' a '%s':" % (primera[0], ultima[0]))
    # Recorremos COLUMNAS_TABLA en vez de nombrar las claves a mano. Si alguien agrega una
    # columna, el resumen la toma solo; nombrandolas se desincroniza de la tabla de arriba.
    for col in COLUMNAS_TABLA:
        print("   %-14s %s  →  %s"
              % (col, primera[1].get(col, "—"), ultima[1].get(col, "—")))
print()
if FALLAS_GENERADOR:
    print("⚠ %d llamada(s) al generador fallaron y quedaron marcadas en la salida:"
          % len(FALLAS_GENERADOR))
    for f in FALLAS_GENERADOR[:5]:
        print("   ", f)
    print()
print("Medido sobre %d actos y %d preguntas, con este embedder y este generador." % (
    len(corpus), len(EVAL_SET)))
print("No son benchmarks: son la evidencia con la que se decide qué poner en TU pipeline.")

LO QUE MEDIMOS, TODO JUNTO

Cada fila agrega una pieza sobre la anterior
──────────────────────────────────────────────────────────────────────
                         temáticas   puesto   identificador   contexto
denso, pedazo crudo          26/30      1.4            2/15   1882 tok
+ contextualización          23/30      1.3            2/15   2175 tok
+ canal léxico (RRF)         26/30      1.5            6/15   2151 tok
+ padre → hijo               26/30      1.5            6/15   3746 tok
+ HyDE                       25/30      1.4           11/15   3390 tok
+ re-puntuador               28/30      1.6           10/15   3427 tok

de 'denso, pedazo crudo' a '+ re-puntuador':
   temáticas      26/30  →  28/30
   puesto         1.4  →  1.6
   identificador  2/15  →  10/15
   contexto       1882 tok  →  3427 tok

Medido sobre 1544 actos y 30 preguntas, con este embedder y este generador.
No son benchmarks: son la evidencia con la que se decide qué poner en TU pipeline.


**Lo que sigue, acá mismo.** La parte B —`Clase08_nbB_rag.ipynb`— toma este mismo pipeline y hace dos cosas con él: lo reescribe en LlamaIndex, para ver qué pieza del framework corresponde a cada decisión que acabamos de tomar a mano; y lo mide con RAGAS, que es el paquete estándar de las cuatro métricas. Es media hora menos que ésta y arranca con la placa vacía.

**Y lo que sigue en el curso.** Tres hilos salen de acá:

- **La ventana se llena.** Hoy decidimos qué pasaje entra; la Semana 9 se ocupa de qué hacer cuando no entra, y de lo que el modelo tiene que **recordar entre turnos** — que es otra cosa que una fuente externa: eso es memoria.
- **El corpus tiene forma.** Un acto cita a otros, y esas citas son un grafo. Recuperar sobre estructura y no sobre texto plano es la Semana 10.
- **Y el agente decide solo.** Hoy recuperamos siempre, antes de generar. Un sistema que decide **cuándo** le hace falta buscar, **qué** buscar y **si alcanza con lo que trajo** es *agentic RAG*, y es la Unidad 5.

> **Una frontera que conviene nombrar antes de irse.** Todo lo que entra al índice entra al prompt. Un pasaje recuperado es texto de terceros con acceso directo a las instrucciones del modelo, y puede contener instrucciones. Ésta es la primera clase donde eso pasa. Cómo se defiende: Semana 15.

▶ **Láminas 53–57** (el pipeline final; nunca tocamos los pesos; el remate honesto; el hilo con el resto del curso; cierre y recursos).